## **Setup**

### Setup Index
1. Imports & Environment
2. Global Config & Constants
3. Paths & Persistence
4. Core Utilities (Data/Math)
5. Plotting Utilities
6. Model/Experiment Helpers
7. UPO Helpers



### Imports & Environment

In [ ]:
# %matplotlib widget

# Modules Imports
import os

# Select device: "cpu" or "gpu"
# If using GPU, make sure CUDA is properly installed.
# Transformers and TFTs are very slow on CPU.

device = "gpu"  # change to "cpu" if needed

if device == "cpu":
    os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
else:
    os.environ["CUDA_VISIBLE_DEVICES"] = "0"

import time
import warnings
from datetime import datetime

# Ignores warnings: Creful, they are there for a reason so do not ignore them!
warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=DeprecationWarning)
warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)
warnings.filterwarnings("ignore", category=Warning)

import gc
import math
import numpy as np
import keras
import pickle
import pandas as pd


import tensorflow as tf
from tensorflow import keras
from tabulate import tabulate
import matplotlib.ticker
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

import multiprocessing as mp
from concurrent.futures import ProcessPoolExecutor,ThreadPoolExecutor, as_completed

# tf.compat.v1.enable_eager_execution()
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# I found a lot of bugs in Vs Code when using diferent matplotlib backends
# Best on remains widged. However, there is a known bug when running vs code split window, so if you want to use vs code, make sure to run it in a single window and not split.
# Plots disapear or freeze when zooming or drawing in split window, but they work fine in single window. 
# Also, some backends do not work at all, so I recommend using widget backend for best experience.

# NOT VECTORIAL
# %matplotlib inline 

# BUGS ON ZOOM AND DRAW when in split window!!!!
%matplotlib widget 

# NOT WORKING
# %matplotlib notebook  

# MIGHT AS WELL TRY
# %matplotlib ipympl

# Nothing
# %matplotlib nbagg

# New window always
# %matplotlib qt

# Custom modules
import sys
# caution: path[0] is reserved for script path (or '' in REPL)
sys.path.insert(1, 'modules/')

from model import SequenceModel
from data import DataHandler
from metrics import Metrics

# From https://github.com/DurstewitzLab/DynaMix-python/tree/main (DynaMix)

from metrics_dyna import geometrical_misalignment, temporal_misalignment, MASE

# Used for UPOs
from scipy.spatial import KDTree, cKDTree
from scipy.integrate import solve_ivp

# Transformers
# DARST is a very powerful library for time series forecasting, it has a lot of models implemented, including transformers and TFTs.

from darts.models import TransformerModel
from darts.models import TFTModel
from darts import TimeSeries

# Torch is used for some custom models and for some metrics, it is also used for some data handling 
import torch
torch.set_float32_matmul_precision('high')

# For various task such as parallelization, file handling, etc.
import glob
from collections import defaultdict

### Global Config & Constants

In [ ]:
def reset_matplotlib_defaults():
    # Reset ALL settings to matplotlib factory defaults first
    plt.rcParams.update(plt.rcParamsDefault)

    # Then apply your custom settings on top
    font_main = 26
    font_title = 26
    font_legend = 21
    linewidth = 4

    plt.rcParams.update({
        'font.size': font_main,
        'axes.labelsize': font_main,
        'xtick.labelsize': font_main,
        'ytick.labelsize': font_main,
        'legend.fontsize': font_legend,
        'axes.titlesize': font_title,
        'figure.titlesize': font_main,
        'axes.grid': True,
        'axes.grid.which': 'major',
        'axes.linewidth': 0.8,
        'lines.linewidth': linewidth
    })

    plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.rcParamsDefault['axes.prop_cycle'].by_key()['color'])

# Prameters Program
program_parameters = {
    'data_path': 'datasets/',
    'plot_path': 'plots/',
    'results_path': 'results/',
    'models_path': 'models/',
    'dataset' : ['Lorenz'],
    'multivariate': True
}

# # Plotting Parameters, you can reuse it before the experiment block

# # Font sizes
# font_main = 16
# font_title = 18
# font_legend = 14
# linewidth = 3

# Font sizes
font_main = 24
font_title = 24
font_legend = 24
linewidth = 4

plt.rcParams.update({
    # Font sizes
    'font.size': font_main,
    'axes.labelsize': font_main,
    'xtick.labelsize': font_main,
    'ytick.labelsize': font_main,
    'legend.fontsize': font_legend,
    'axes.titlesize': font_title,
    'figure.titlesize': font_main,
    # Axis & grid styling
    'axes.grid': True,
    'axes.grid.which': 'major',
    'axes.linewidth': 0.8, 

    # Line widths
    'lines.linewidth': linewidth
})



METRIC_RANGES_DELTA = {
    'r50': (0, 0.35),
    'r100': (0, 0.5),
    'r150': (0, 0.15)
}

METRIC_RANGES_DELTA_UPO = {
    'r50': (0, 0.35),
    'r100': (0, 0.5),
    'r150': (0, 0.5)
}

PLOT_LOG_SCALE_DEFAULT_PREDICTIONS = True

METRIC_RANGES_PLOT = {
    'mse': (0, 2.5), # 3
    'sbd': (0, 1),
    'dstsp': (0, 12)
}

FORECAST_TYPES_RANGES = {
    'short-term': (1, 30),
    'long-term': (40, 500)
}

In [ ]:
# Parameters - ML Models - Use it and change values in cells if needed, use as a template, just keep track of changes

# LSTMS and recurrent models
models_parameters = {
        'model_type': 'LSTM',
        'seq_len': 256,
        'hidden_size': [32, 16],
        'LR': 0.01,
        'decay_rate': 0.99,
        'decay_step': 100,
        'batch_size': 128,
        'epochs': 300,
        'train_ratio': 1,
        'optimizer': 'Adam',
        'loss': 'MSE',
        'metrics': ['MeanSquaredError'],
        'seed': 42,
        'TF': False,
        'stateful': True,
        'MISO': False,
        'forecast_size': 1,
        'rolling': False
}

# Transformers and TFTs
transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}


In [ ]:
# Parameters - Datasets
datasets_parameters = {
    'LORENZ': {
        'inputs': [3, 4, 5 ],
        'outputs': [3, 4, 5],
        'dropped_columns': [0, 1, 2],
        'sep' : ',',
        'decimal' :'.'
    },
}

In [ ]:
NAME = "LORENZ"


### Paths & Persistence

In [ ]:
def save_models_to_file(models, filename):
    with open(filename, 'wb') as f:
        pickle.dump(models, f)      

def load_models_from_file(filename):
    with open(filename, 'rb') as f:
        models = pickle.load(f)
    return models

# Create data and plot folders (directorys)
if not os.path.exists(program_parameters['data_path']):
    os.makedirs(program_parameters['data_path'])

if not os.path.exists(program_parameters['plot_path']):
    os.makedirs(program_parameters['plot_path'])

if not os.path.exists(program_parameters['results_path']):
    os.makedirs(program_parameters['results_path'])

if not os.path.exists(program_parameters['models_path']):
    os.makedirs(program_parameters['models_path'])

In [ ]:
# Initialize the model (see model.py)
# For now, supports LSTM, GRU and RNN
def reset_seeds(seed = 42):
    
    models_parameters['seed'] = seed
    keras.utils.set_random_seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

    return seed

seed = reset_seeds(42)

### Core Utilities (Data/Math)

In [ ]:
# Convert to real values from sliding window
def create_real(input, forecast_size, seq_len):
    real = np.array([]).reshape(0, np.shape(input)[1])

    for i in range(seq_len, 
                   np.shape(input)[0]- seq_len, 
                   seq_len):
        
        real = np.vstack((real, input[i- forecast_size: i, :]))
    return real

In [ ]:
def compute_basic_metrics(all_ground_truth, all_predictions, scale=False):
    
    obs = all_ground_truth
    pred = all_predictions
    
    if scale:
        minmaxscaler = MinMaxScaler(feature_range=(0 , 1))
        all_obs = np.vstack((all_predictions, all_ground_truth))
        minmaxscaler.fit(all_obs)

        obs = minmaxscaler.transform(all_ground_truth)
        pred = minmaxscaler.transform(all_predictions)
    
    m1 = np.mean((obs - pred)**2)
    _, m2 = Metrics().compute_residuals(obs, pred, residual_type='SBD')
    _, m3 = Metrics().compute_residuals(obs, pred, residual_type='energy')

    results = {
        "mse": m1,
        "sbd": m2,
        "energy": m3
    }

    return results

def compute_all_metrics(predicted, observed, bins=5, smoothing=10, mase_steps=10, scale=False):
    ground_truth_tensor = torch.tensor(observed, dtype=torch.float32)
    reconstruction_tensor = torch.tensor(predicted, dtype=torch.float32)
    
    basic_metrics = compute_basic_metrics(observed, predicted, scale=scale)
    dyna_metrics = compute_all_dyna_metrics(reconstruction_tensor, ground_truth_tensor, bins=bins, smoothing=smoothing, mase_steps=mase_steps)

    results = {**basic_metrics, **dyna_metrics}
    return results

def compute_all_dyna_metrics(predicted, observed, bins=5, smoothing=10, mase_steps=10):
    ground_truth_tensor = torch.tensor(observed, dtype=torch.float32)
    reconstruction_tensor = torch.tensor(predicted, dtype=torch.float32)

    dstsp = geometrical_misalignment(reconstruction_tensor, ground_truth_tensor, n_bins=bins)
    dh = temporal_misalignment(reconstruction_tensor, ground_truth_tensor, smoothing=smoothing)
    pe = MASE(reconstruction_tensor, ground_truth_tensor, steps=mase_steps)

    results = {
        "dstsp": dstsp,
        "dh": dh,
        "pe": pe
    }
    return results


In [ ]:
def format_metric(value, name=""):
    # if value == 0:
    #     return "0"
    # abs_val = abs(value)
    # if abs_val < 1e-3 or abs_val >= 1e4:
    #     return f"{value:.2e}"         
    # elif abs_val < 0.1:
    #     return f"{value:.4f}"          
    # else:
    #     return f"{value:.3f}"          
    if value == 0:
        return r"$\mathbf{0}$"
    abs_val = abs(value)
    if abs_val < 1e-4 or abs_val >= 1e6:
        exp = int(np.floor(np.log10(abs_val)))
        mantissa = value / 10**exp
        return rf"$\mathbf{{{mantissa:.2f} \times 10^{{{exp}}}}}$"
    elif abs_val < 0.1:
        return rf"$\mathbf{{{value:.4f}}}$"
    else:
        return rf"$\mathbf{{{value:.3f}}}$"

### Model/Experiment Helpers

In [ ]:
def extend_with_metrics(df, model, sequence_length, bins, smoothing, mase_steps):
    df_copy = df.copy()

    for index, row in df_copy.iterrows():
        if model in ["LSTM-A", "LSTM-B"]:
            observed = row["dfs"]["obs"]
            predicted = row["dfs"]["pred"]
        elif model in ["UPO"]:
            observed = row["observed_scaled"]
            predicted = row["predicted_scaled"]
        else:
            observed = row["observed"]
            predicted = row["predicted"]

        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN - sequence_length]
        first_sim_predicted = predicted[0:SIMULATION_LEN - sequence_length]

        metrics_temp.append(compute_all_metrics(
            first_sim_predicted,
            first_sim_observed,
            bins=bins,
            smoothing=smoothing,
            mase_steps=mase_steps,
            scale=False,
        ))

        for i in range(SIMULATION_LEN, len(predicted) - SIMULATION_LEN, SIMULATION_LEN):
            observed_temp = observed[i:i + SIMULATION_LEN]
            predicted_temp = predicted[i:i + SIMULATION_LEN]

            observed_temp = observed_temp[sequence_length:]
            predicted_temp = predicted_temp[sequence_length:]

            metrics_temp.append(compute_all_metrics(
                predicted_temp,
                observed_temp,
                bins=bins,
                smoothing=smoothing,
                mase_steps=mase_steps,
                scale=False,
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, "item") else v for k, v in metrics.items()}

        for col in ["mse", "sbd", "dstsp"]:
            if col in metrics:
                df_copy.loc[index, col] = metrics[col]

    return df_copy

In [ ]:
def train_test_single_model_thread_delta(
                             datasets_parameters = None, 
                             train_rhos = None, 
                             test_rhos = None, 
                             train_data = None,
                             test_data = None, 
                             all_data = None,
                             forecast_horizons = None, 
                             scale_per_rho = True,
                             models_parameters = None,
                             model = None
                             ):
    
    if datasets_parameters is None or train_rhos is None \
            or test_rhos is None  or forecast_horizons is None \
            or test_data is None or train_data is None or all_data is None:
        
        raise ValueError("Missing arguments")
    
    local_params = models_parameters.copy()
    local_params['forecast_size'] = forecast_horizons[0]

    if model is None:
        model = SequenceModel(local_params, logging=False)    
        model.create_model(dataset_parameters=datasets_parameters[NAME])    

        model.create_scaler(train_data, train_data, scale_per_rho = scale_per_rho)

        print(f"Running Training: Train rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}")
        model.train_model(
            data=train_data,
            dataset_parameters=datasets_parameters[NAME],
            all_data=train_data,
            fit_scaler=False,
            scale_per_rho = scale_per_rho,
            verbose = 0
        )
    else:
        print(f"Using existing model for Testing: Train rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}")
        
        model.create_scaler(test_data, test_data, scale_per_rho = scale_per_rho)

    model.create_model(dataset_parameters=datasets_parameters[NAME], test=True)

    print(f"Running Testing: Train rho={train_rhos}, test rho={test_rhos}")
    results = model.test_model(
        data=test_data,
        dataset_parameters=datasets_parameters[NAME],
        metric="mse",
        scale_per_rho = scale_per_rho
    )

    obs = create_real(results['y'], forecast_size=forecast_horizons[0], seq_len=local_params['seq_len'])
    pred = create_real(results['y_hat'], forecast_size=forecast_horizons[0], seq_len=local_params['seq_len'])

    # obs_inversed = model.inverse_transform(obs)
    dfs = {'obs': obs, 'pred': pred}

    print(f"Done Training and Testing rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}")

    del model
    del train_data, test_data, all_data, local_params
    gc.collect()
    
    return dfs

def train_test_single_model_thread_hyperparameters(
                             datasets_parameters = None, 
                             train_rhos = None, 
                             test_rhos = None, 
                             train_data = None,
                             test_data = None, 
                             all_data = None,
                             forecast_horizons = None, 
                             scale_per_rho = True, 
                             scale_metrics = False,
                             models_parameters = None,
                             parameter_names = [],
                             parameter_values = [],
                             TEACHER_FORCING = False,
                             ATTENTION = False,
                             model = None,
                             measure_time = False
                             ):


    if datasets_parameters is None or train_rhos is None \
            or test_rhos is None  or forecast_horizons is None \
            or test_data is None or train_data is None or all_data is None:
        
        raise ValueError("Missing arguments")
    
    local_params = models_parameters.copy()
    temp_seq_len = local_params['seq_len']

    if TEACHER_FORCING:
        local_params['forecast_size'] = 1
        # local_params['seq_len'] = local_params['seq_len'] + forecast_horizons[0]
    else:
        local_params['forecast_size'] = forecast_horizons[0]

    if not len(parameter_names) == len(parameter_values):
        raise ValueError("Missing arguments")
    
    if len(parameter_names) > 0 and len(parameter_values) > 0:
        for parameter_name, parameter_value in zip(parameter_names, parameter_values):
            local_params[parameter_name] = parameter_value

    if model is None:
        model = SequenceModel(local_params, logging=False)

        if ATTENTION:
            model.create_model_attention(dataset_parameters=datasets_parameters[NAME], num_heads = 4, key_dim = 8)
        else:
            model.create_model(dataset_parameters=datasets_parameters[NAME])
            
        model.create_scaler(train_data, train_data, scale_per_rho = scale_per_rho, mode = 2)

        if len(parameter_names) == 0 and len(parameter_values) == 0:
            print(f"Running Training: Train rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, TEACHER_FORCING={TEACHER_FORCING}")
        else:
            print(f"Running Training: Train rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, {parameter_names}={parameter_values}")

        # Measure Training Time
        if measure_time:
            start = time.perf_counter()
        model.train_model(
            data=train_data,
            dataset_parameters=datasets_parameters[NAME],
            all_data=train_data,
            fit_scaler=False,
            scale_per_rho = scale_per_rho,
            verbose = 0,
            early_stop = False,
            validation = False
        )
        if measure_time:
            end = time.perf_counter()
            print(f"Finished LSTM training horizon={forecast_horizons} and architecture{device} in {end - start:0.4f} seconds\n")
        # END Measure Training Time

    else:
        print(f"Using existing model for Testing: Train rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, TEACHER_FORCING={TEACHER_FORCING}")
        model.create_scaler(test_data, test_data, scale_per_rho = scale_per_rho, mode = 2)


    if ATTENTION:
        model.create_model_attention(dataset_parameters=datasets_parameters[NAME], test=True, num_heads = 4, key_dim = 8)
    else:
        model.create_model(dataset_parameters=datasets_parameters[NAME], test=True)

    if len(parameter_names) == 0 and len(parameter_values) == 0:
        print(f"Running Testing: rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, TEACHER_FORCING={TEACHER_FORCING}")
    else:
        print(f"Running Testing: rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, {parameter_names}={parameter_values}")

    

    # Measure Testing Time
    if measure_time:
        start = time.perf_counter()
        
    if TEACHER_FORCING:
        model.params['forecast_size'] = forecast_horizons[0] 
        model.params['seq_len'] = temp_seq_len

        results = model.test_model_tf3(
            data=test_data,
            dataset_parameters=datasets_parameters[NAME],
            metric="mse",
            scale_per_rho = scale_per_rho
        )
    else:
        results = model.test_model(
            data=test_data,
            dataset_parameters=datasets_parameters[NAME],
            metric="mse",
            scale_per_rho = scale_per_rho
        )
    if measure_time:
        end = time.perf_counter()
        print(f"Finished LSTM testing horizon={forecast_horizons} and architecture_{device} in {end - start:0.4f} seconds\n")
    # END Measure Testing Time
    
    if TEACHER_FORCING or forecast_horizons[0] == 1 or local_params['rolling'] == False:
        obs = results['y']
        pred = results['y_hat']

    elif forecast_horizons[0] > 1 or local_params['rolling'] == True:
        obs = create_real(results['y'], forecast_size=forecast_horizons[0], seq_len=local_params['seq_len'])
        pred = create_real(results['y_hat'], forecast_size=forecast_horizons[0], seq_len=local_params['seq_len'])

    # obs_inversed = model.inverse_transform(obs)
    dfs = {'obs': obs, 'pred': pred}

    print(f"Done Training and Testing rho={train_rhos}, test rho={test_rhos}, forecast={forecast_horizons}, TEACHER_FORCING={TEACHER_FORCING}, {parameter_names}={parameter_values}")

    del model
    del train_data, test_data, all_data, local_params
    gc.collect()
    
    return dfs

def worker_delta(job):
    datasets_parameters, training_rho, testing_rho, train_data, test_data, all_data, horizon, SCALE_PER_RHO, models_parameters, model = job

    dfs = train_test_single_model_thread_delta(
        datasets_parameters = datasets_parameters,
        train_rhos = training_rho,
        test_rhos = testing_rho,
        train_data = train_data,
        test_data = test_data,
        all_data = all_data,
        forecast_horizons = horizon,
        scale_per_rho = SCALE_PER_RHO,
        models_parameters = models_parameters,
        model = model
    )

    return {
        "train_rhos": training_rho[0],
        "test_rhos": testing_rho,
        "forecast_horizons": horizon[0],
        "dfs": dfs
    }

def worker_param_hyper(job):
    datasets_parameters, \
    train_rhos, \
    test_rho, \
    train_data, \
    test_data, \
    all_data, \
    forecast_horizons, \
    SCALE_PER_RHO,\
    models_parameters, \
    parameter_names, \
    parameter_values, \
    TFGEN, \
    model = job
    

    
    dfs = train_test_single_model_thread_hyperparameters(
        datasets_parameters = datasets_parameters,
        train_rhos = train_rhos,
        test_rhos = test_rho,
        train_data = train_data,
        test_data = test_data,
        all_data = all_data,
        forecast_horizons = forecast_horizons,
        scale_per_rho = SCALE_PER_RHO,
        models_parameters = models_parameters,
        parameter_names = parameter_names,
        parameter_values = parameter_values,
        TEACHER_FORCING = TFGEN,
        model = model
    )

    
    hidden_sizes = models_parameters['hidden_size']

    return_dict = {
        "TEACHER_FORCING": TFGEN,
        "train_rhos": train_rhos,
        "test_rhos": test_rho,
        "forecast_horizons": forecast_horizons[0],
        "SL": models_parameters['seq_len'],
        "BS": models_parameters['batch_size'],
        "Hidden Size 1": hidden_sizes[0] if len(hidden_sizes) > 0 else None,
        "Hidden Size 2": hidden_sizes[1] if len(hidden_sizes) > 1 else None,
        "epochs": models_parameters['epochs'],
        "dfs": dfs
    }

    for name, value in zip(parameter_names, parameter_values):
        if name == "seq_len":
            return_dict["SL"] = int(value)
            continue
        if name == "batch_size":
            return_dict["BS"] = int(value)
            continue
        if name == "hidden_size":
            return_dict["Hidden Size 1"] = int(value[0])
            return_dict["Hidden Size 2"] = int(value[1])
            continue
        if name == "epochs":
            return_dict["epochs"] = int(value)
            continue
    return return_dict

In [ ]:
# Dynamix metrics and stuff (one of the pretrained models is from DynaMix-python, see)

# From DynaMix-python, see

### https://github.com/DurstewitzLab/DynaMix-python/tree/main

from scipy.ndimage import gaussian_filter1d

def compute_and_smooth_power_spectrum(x, smoothing):
    x_ =  (x - x.mean()) / x.std()
    fft_real = np.fft.rfft(x_)
    ps = np.abs(fft_real)**2 * 2 / len(x_)
    ps_smoothed = gaussian_filter1d(ps, smoothing)
    return ps_smoothed / ps_smoothed.sum()

def plot_3D_attractor(
    context,
    prediction,
    ground_truth=None,
    lim_gen=2000,
    lim_pse=500,
    smoothing_sigma=2.0,
    horizon=10,
    title_3D="LSTM",
    title_big=None,
    components=None,  # default: [0, 2] → X and Z
):
    """
    Plot 3D attractor with time series and power spectrum.

    Args:
        context:          Context data, shape (context_len, 3+).
        prediction:       Model prediction, shape (pred_len, 3+).
        ground_truth:     Optional ground truth, shape (seq_len, 3+).
        lim_gen:          Max points shown in time-series panels.
        lim_pse:          Max frequency bins shown in power-spectrum panels.
        smoothing_sigma:  Gaussian sigma used when smoothing power spectra.
        horizon:          Forecast horizon (shown in title).
        title_3D:         Title for the 3D plot.
        title_big:        Big title for the entire figure.
        components:       List of component indices to plot in the time-series
                          and power-spectrum panels. Defaults to [0, 2] (X, Z).
                          Pass [0, 1, 2] to restore all three.

    Returns:
        matplotlib.figure.Figure
    """
    import matplotlib
    import matplotlib.pyplot as plt

    # ------------------------------------------------------------------ #
    # Defaults & coercions                                                 #
    # ------------------------------------------------------------------ #
    if components is None:
        components = [0, 2]          # X and Z

    all_labels = ["x", "y", "z"]
    component_labels = [all_labels[c] for c in components]
    n_panels = len(components)

    def _to_numpy(x):
        if x is None:
            return None
        if not isinstance(x, np.ndarray):
            return x.detach().cpu().numpy()
        return x

    context      = _to_numpy(context)
    prediction   = _to_numpy(prediction)
    ground_truth = _to_numpy(ground_truth)

    CL_length = context.shape[0]

    # ------------------------------------------------------------------ #
    # Style                                                                #
    # ------------------------------------------------------------------ #
    plt.style.use("seaborn-v0_8-whitegrid")
    matplotlib.rcParams.update({
        "pdf.fonttype":  42,
        "ps.fonttype":   42,
        "font.family":   "sans-serif",
        "font.sans-serif": ["Arial", "DejaVu Sans", "Helvetica"],
        "font.size":     14,
    })

    # ------------------------------------------------------------------ #
    # Layout geometry                                                      #
    # ------------------------------------------------------------------ #
    panel_h   = 0.30          # height of each row panel (normalised figure)
    row_gap   = 0.1          # vertical gap between row panels
    bottom    = 0.10          # bottom margin

    # Total height covered by all panels
    total_h = n_panels * panel_h + (n_panels - 1) * row_gap   # e.g. 2×0.25 + 0.10 = 0.60

    # Column widths (proportional)
    width_ratios  = [1.25, 2.8, 1.1]
    col_gap       = 0.02
    total_ratio   = sum(width_ratios)
    col_widths    = [r / total_ratio for r in width_ratios]

    col_lefts = [0.0]
    for w in col_widths[:-1]:
        col_lefts.append(col_lefts[-1] + w)

    fig = plt.figure(figsize=(15, 4))
    fig.suptitle(title_big, fontsize=16, y=0.90, fontweight="bold")

    # ------------------------------------------------------------------ #
    # Column 0 – 3D attractor                                             #
    # (same bounding box height as the stacked time-series panels)        #
    # ------------------------------------------------------------------ #
    ax3d_offset = 0.05   # nudge the 3D box upward relative to the other panels
    ax3d = fig.add_axes(
        [col_lefts[0], bottom + ax3d_offset, col_widths[0] - col_gap, total_h - ax3d_offset],
        projection="3d",
    )

    if ground_truth is not None:
        ax3d.plot(
            ground_truth[:, 0], ground_truth[:, 1], ground_truth[:, 2],
            label="Ground truth", linewidth=2, color="#2C3E50", alpha=0.5,
        )

    ax3d.plot(
        prediction[:, 0], prediction[:, 1], prediction[:, 2],
        label=title_3D, linewidth=2, color="#FF4242", alpha=0.9,
    )

    # ax3d.set_xlabel("X", fontsize=16)
    # ax3d.set_ylabel("Y", fontsize=16)
    # ax3d.set_zlabel("Z", fontsize=16)
    ax3d.legend(fontsize=14, loc="upper center")
    # ax3d.tick_params(axis="both", which="major", labelsize=12)
    ax3d.tick_params(labelsize=12)

    # ------------------------------------------------------------------ #
    # Columns 1 & 2 – time series and power spectrum                      #
    # ------------------------------------------------------------------ #
    for row_idx, comp in enumerate(components):
        # Bottom-most component sits at `bottom`; top-most is highest
        y_pos = bottom + (n_panels - 1 - row_idx) * (panel_h + row_gap)
        label = component_labels[row_idx]
        is_bottom_row = (row_idx == n_panels - 1)

        # -- Time series (column 1) --
        ax_ts = fig.add_axes(
            [col_lefts[1] + col_gap, y_pos, col_widths[1] - 2 * col_gap, panel_h]
        )

        if ground_truth is not None:
            ax_ts.plot(
                np.arange(CL_length, CL_length + lim_gen),
                ground_truth[:lim_gen, comp],
                label="Ground truth", linewidth=2.5, color="#2C3E50", alpha=0.5,
            )

        ax_ts.plot(
            np.arange(CL_length, CL_length + lim_gen),
            prediction[:lim_gen, comp],
            label=title_3D, linewidth=2.5, color="#FF4242", alpha=0.9,
        )

        ax_ts.set_ylabel(label, fontsize=16)
        ax_ts.set_yticks([-2, 0, 2])
        ax_ts.grid(True, alpha=0.3)
        ax_ts.tick_params(axis="both", which="major", labelsize=12)
        if is_bottom_row:
            ax_ts.set_xlabel("Step", fontsize=12)

        # -- Power spectrum (column 2) --
        ax_ps = fig.add_axes(
            [col_lefts[2] + col_gap, y_pos, col_widths[2] - 2 * col_gap, panel_h]
        )

        if ground_truth is not None:
            gt_data = ground_truth[:, comp].flatten()
            ps_gt   = compute_and_smooth_power_spectrum(gt_data, smoothing_sigma)
            ax_ps.plot(
                ps_gt[:lim_pse],
                label="Ground truth", linewidth=2.5, color="#2C3E50", alpha=0.5,
            )

        pred_data = prediction[:, comp].flatten()
        ps_pred   = compute_and_smooth_power_spectrum(pred_data, smoothing_sigma)
        ax_ps.plot(
            ps_pred[:lim_pse],
            label=title_3D, linewidth=2.5, color="#FF4242", alpha=0.9,
        )

        ax_ps.set_yscale("log")
        ax_ps.grid(True, alpha=0.3)
        ax_ps.tick_params(axis="both", which="major", labelsize=12)
        if is_bottom_row:
            ax_ps.set_xlabel("Frequency", fontsize=12)

    # ------------------------------------------------------------------ #
    # To restore all three components, pass components=[0, 1, 2].         #
    # The Y panel (index 1) is excluded by default.                       #
    # ------------------------------------------------------------------ #

    plt.savefig(rf"{title_3D}"+".pdf", dpi=300, bbox_inches="tight", format="pdf")
    return fig



### UPO Helpers and Pipelinies

In [ ]:
# UPO Analysis and Visualization
def lorenz_rhs(t, X, sigma=10.0, rho=28.0, beta=8/3):
    x, y, z = X
    return [
        sigma * (y - x),
        rho * x - y - x * z,
        x * y - beta * z
    ]

def load_lorenz_orbit(orbit_file, T_file):
    data = np.loadtxt(orbit_file)
    if data.shape[0] == 3 and data.shape[1] > 3:
        traj = data.T
    else:
        traj = data
    X0 = traj[0]
    T = float(np.loadtxt(T_file))
    return X0, traj, T

def generate_lorenz_trajectory(X0, T, dt=0.01, sigma=10.0, rho=28.0, beta=8/3):
    n_points = int(np.ceil(T / dt)) + 1
    t_eval = np.linspace(0, T, n_points)
    sol = solve_ivp(
        lambda t, X: lorenz_rhs(t, X, sigma=sigma, rho=rho, beta=beta),
        t_span=(0, T),
        y0=X0,
        t_eval=t_eval,
        rtol=1e-14,
        atol=1e-16,
        method="RK45"
    )
    return sol.t, sol.y.T

def compute_symbol_sequence_AB(traj, z_threshold=27.0):
    symbols = []
    z = traj[:, 2]
    y = traj[:, 1]
    n = len(z)
    for i in range(n - 1):
        if z[i] <= z_threshold and z[i + 1] > z_threshold:
            symbols.append('A' if y[i] < 0 else 'B')
    if z[-1] <= z_threshold and z[0] > z_threshold:
        symbols.append('A' if y[-1] < 0 else 'B')
    return ''.join(symbols)

def process_single_orbit(args):
    orbit_num, orbit_file, data_dir, dt, sigma, rho, beta = args
    base_name = os.path.basename(orbit_file)
    orbit_id_str = base_name[5:] 
    T_file = os.path.join(data_dir, "T" + orbit_id_str)
    
    try:
        X0, traj_orig, T = load_lorenz_orbit(orbit_file, T_file)
        _, traj_new = generate_lorenz_trajectory(X0, T, dt=dt, sigma=sigma, rho=rho, beta=beta)
        word = compute_symbol_sequence_AB(traj_new)
        
        return {
            "Orbit_ID": orbit_num,
            "Word": word,
            "K": len(word),
            "X0_x": X0[0],
            "X0_y": X0[1],
            "X0_z": X0[2],
            "Trajectory": traj_new,
            "T": T,
            "dt": dt
        }
    except Exception as e:
        return None

def load_all_orbits_parallel(data_dir, dt=0.01, sigma=10.0, rho=28.0, beta=8/3):
    raw_files = glob.glob(os.path.join(data_dir, "orbit*.dat"))
    decorated = sorted([(int(os.path.basename(f)[5:-4]), f) for f in raw_files])
    
    tasks = [(num, f, data_dir, dt, sigma, rho, beta) for num, f in decorated]
    
    print(f"Starting parallel processing on {len(tasks)} orbits...")
    
    results = []
    with ProcessPoolExecutor() as executor:
        results = list(executor.map(process_single_orbit, tasks))

    results = [r for r in results if r is not None]
    
    df = pd.DataFrame(results)
    df.sort_values(['Orbit_ID', 'K'], inplace=True)  
    df.set_index("Orbit_ID", inplace=True)
    # df = pd.DataFrame(results)
    return df

def plot_multiple_trajectory_comparisons(df_orbits, n_orbits, data_dir="data/DATA1", ncols=4):
    n_to_plot = min(n_orbits, len(df_orbits))
    nrows = math.ceil(n_to_plot / ncols)
    
    fig, axes = plt.subplots(nrows, ncols, figsize=(4 * ncols, 3.5 * nrows))
    axes = np.array(axes).reshape(-1)

    orbit_indices = df_orbits.index[:n_to_plot]

    for i, orbit_idx in enumerate(orbit_indices):
        ax = axes[i]
        orbit_file = os.path.join(data_dir, f"orbit{orbit_idx}.dat")
        T_file = os.path.join(data_dir, f"T{orbit_idx}.dat")
        
        X0, traj_orig, T = load_lorenz_orbit(orbit_file, T_file)
        row = df_orbits.loc[orbit_idx]
        
        ax.plot(traj_orig[:, 0], traj_orig[:, 2], 'r-', linewidth=1, alpha=0.7)
        ax.plot(row["Trajectory"][:, 0], row["Trajectory"][:, 2], 'b--', linewidth=1)

        ax.set_title(f"Orbit {orbit_idx} | K={row['K']}\n{row['Word']}", fontsize=9)
        ax.tick_params(labelsize=8)

    for j in range(len(orbit_indices), len(axes)):
        axes[j].axis("off")

    fig.tight_layout(rect=[0, 0, 1, 0.95])
    fig.suptitle("Lorenz UPOs: Original (Red) vs Integrated (Blue)", fontsize=16, fontweight='bold', y=0.98)
    plt.show()

def upo_shadow_traj(traj_xyz, tree, eps, labels=None):
    
    n_orbits = len(np.unique(labels))
    orbit_counts = np.zeros(n_orbits, dtype=int)
    orbit_longest = np.zeros(n_orbits, dtype=int)
    current_streak = np.zeros(n_orbits, dtype=int)

    # Find ALL UPO points near each trajectory point
    neighbors = tree.query_ball_point(traj_xyz, r=eps)

    # print("Processing trajectory points for shadow analysis...")
    for idx, neigh in enumerate(neighbors):
        # print(f"Processing n = {idx} neighbor set with {len(neigh)} points...")
        # Orbits close at THIS timestep
        if len(neigh) > 0:
            orbit_ids = np.unique(labels[neigh])
        else:
            orbit_ids = []

        # Update all orbits
        for oid in range(n_orbits):
            if oid in orbit_ids:
                orbit_counts[oid] += 1
                current_streak[oid] += 1
                orbit_longest[oid] = max(orbit_longest[oid], current_streak[oid])
            else:
                current_streak[oid] = 0

    total_close = np.sum(orbit_counts > 0)

    return orbit_counts, orbit_longest

def build_labeled_tree(upo_df):
    all_points = []
    labels = []

    start_indices = []
    end_indices = []

    current_index = 0  

    for orbit_id, traj in enumerate(upo_df["Trajectory"], start=1):
        traj = np.asarray(traj)[:, :3]  

        n_pts = traj.shape[0]

        # Store start/end indices for this orbit in the global array
        start_indices.append(current_index)
        end_indices.append(current_index + n_pts - 1)

        # Add points and labels
        all_points.append(traj)
        labels.append(np.full(n_pts, orbit_id))

        current_index += n_pts

    all_points = np.vstack(all_points)
    labels = np.concatenate(labels)

    upo_df = upo_df.copy()
    upo_df["start_idx"] = start_indices
    upo_df["end_idx"] = end_indices

    tree = cKDTree(all_points)

    return tree, labels, upo_df

def _longest_streak(mask):
    longest = 0
    current = 0
    for val in mask:
        if val:
            current += 1
            longest = max(longest, current)
        else:
            current = 0
    return longest

def upo_closeness_stats(traj_xyz, tree, eps, mode="nearest", labels=None):
    """
    Parameters
    ----------
    traj_xyz : (N,3) array
        Trajectory points
    tree : KDTree
        Built from all UPO points
    eps : float
        Distance threshold
    mode : str
        "nearest" → nearest-point distance
        "ball"    → any point inside epsilon ball
        "shadow"  → longest consecutive segment near SAME UPO
    labels : array (required for mode="shadow")
        Orbit ID label for each KD-tree point

    Returns
    -------
    total_close : int
        Total number of trajectory points close to any UPO point
    longest_streak : int
        Longest consecutive segment of trajectory points close to UPO points (with orbit identity if mode="shadow")
    close_mask : (N,) bool array
        True for trajectory points close to any UPO point, False otherwise
    """

    if mode == "nearest":
        dists, idxs = tree.query(traj_xyz, k=1)
        close_mask = dists < eps

        # streak ignoring orbit identity
        longest_streak = _longest_streak(close_mask)

    elif mode == "ball":
        neighbors = tree.query_ball_point(traj_xyz, r=eps)
        close_mask = np.array([len(n) > 0 for n in neighbors])

        longest_streak = _longest_streak(close_mask)

    elif mode == "shadow":
        if labels is None:
            raise ValueError("labels array required for shadow mode")

        dists, idxs = tree.query(traj_xyz, k=1)
        close_mask = dists < eps
        orbit_ids = labels[idxs]

        longest_streak = 0
        current_len = 0
        current_orbit = -1

        for is_close, oid in zip(close_mask, orbit_ids):
            if is_close and oid == current_orbit:
                current_len += 1
            elif is_close:
                current_orbit = oid
                current_len = 1
            else:
                current_len = 0
                current_orbit = -1

            longest_streak = max(longest_streak, current_len)

    total_close = np.sum(close_mask)
    return total_close, longest_streak, close_mask

In [ ]:
# TRANSFORMER PREDICTION PIPELINE
def run_transformer_pipeline(train, test, local_parameters, forecast_horizons, rho_value, model = None, measure_time = False):

    # Scale Data
    
    if model is None:
        scaler = StandardScaler()
        all_data = np.vstack((train, test))

        scaler.fit(train[:, 3:6])
        train[:, 3:6] = scaler.transform(train[:, 3:6])

        scaler.fit(test[:, 3:6])
        test[:, 3:6] = scaler.transform(test[:, 3:6])

        # Create time series
        train_series = pd.DataFrame(train, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
        test_series = pd.DataFrame(test, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])

        train_series = TimeSeries.from_dataframe(train_series[local_parameters["features"]].copy()).astype(np.float32)
        test_series  = TimeSeries.from_dataframe(test_series[local_parameters["features"]].copy()).astype(np.float32)

        input_length = local_parameters["input_length"]

        print(f"Running Transformer for horizons {forecast_horizons} and rho {rho_value}:")

        model = TransformerModel(
            input_chunk_length=input_length,     # number of past time steps used as input
            output_chunk_length=forecast_horizons,   # how many future steps to predict
            d_model=local_parameters["model_dim"],                   # internal embedding dimension
            nhead=local_parameters["attention_heads"],               # number of attention heads
            num_encoder_layers=local_parameters["encoder_layers"],   # how many encoder layers
            num_decoder_layers=local_parameters["decoder_layers"],   # how many decoder layers
            batch_size=local_parameters["batch_size"],
            n_epochs=local_parameters["num_epochs"],
            likelihood=None,
            pl_trainer_kwargs=local_parameters["pl_trainer_kwargs"],
            optimizer_kwargs={'lr': local_parameters["learning_rate"]},
        )

        # Measure Training Time
        if measure_time:
            start = time.perf_counter()
        model.fit(train_series.copy() , verbose=True, stride = forecast_horizons)
        if measure_time:
            end = time.perf_counter()
            print(f"Finished Transformer training with rho={rho_value} and horizon={forecast_horizons} and architecture {device} in {end - start:0.4f} seconds \n")
        # END Measure Training Time
    
    else:
        scaler = StandardScaler()
        scaler.fit(test[:, 3:6])
        test[:, 3:6] = scaler.transform(test[:, 3:6])

        test_series = pd.DataFrame(test, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
        test_series  = TimeSeries.from_dataframe(test_series[local_parameters["features"]].copy()).astype(np.float32)
        
        print(f"Running Transformer (pre-trained) for horizons {forecast_horizons} and rho {rho_value}:")

    # Measure Testing Time
    if measure_time:
        start = time.perf_counter()
    predictions_all = model.historical_forecasts(
        series=test_series,
        forecast_horizon=forecast_horizons,
        stride=forecast_horizons,
        retrain=False,
        verbose=True,
        start=local_parameters["input_length"],
        last_points_only=False,
        overlap_end=False
    )
    if measure_time:
        end = time.perf_counter()
        print(f"Finished Transformer forecasting with rho={rho_value} and horizon={forecast_horizons} and architecture {device} in {end - start:0.4f} seconds \n")
    # END Measure Testing Time

    res = [ts.all_values()[..., 0] for ts in predictions_all]
    np_pred = np.vstack(res)

    offset = local_parameters["input_length"]
    n_pred_steps = np_pred.shape[0]

    np_test = test_series.values()
    aligned_test = np_test[offset : offset + n_pred_steps, :]

    # Sanity check
    assert aligned_test.shape == np_pred.shape, \
        f"Shape mismatch: test {aligned_test.shape} vs pred {np_pred.shape}"
    
    del model, predictions_all, res, test_series
    print(f"Transformer Pipeline completed for horizons {forecast_horizons} and rho {rho_value}.")
    return aligned_test, np_pred


def run_tft_pipeline(train, test, local_parameters, forecast_horizons, rho_value, model = None, measure_time = False):


    if model is None:
        scaler = StandardScaler()
        all_data = np.vstack((train, test))
        # scaler.fit(all_data[:, 3:6])

        scaler.fit(train[:, 3:6])
        train[:, 3:6] = scaler.transform(train[:, 3:6])

        scaler.fit(test[:, 3:6])
        test[:, 3:6] = scaler.transform(test[:, 3:6])

        # Create time series
        train_series = pd.DataFrame(train, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
        test_series = pd.DataFrame(test, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])

        train_series = TimeSeries.from_dataframe(train_series[local_parameters["features"]].copy()).astype(np.float32)
        test_series  = TimeSeries.from_dataframe(test_series[local_parameters["features"]].copy()).astype(np.float32)

        input_length = local_parameters["input_length"]

        print(f"Running TFT for horizons {forecast_horizons} and rho {rho_value}:")

        model = TFTModel(
            input_chunk_length=input_length,
            output_chunk_length=forecast_horizons,
            hidden_size=local_parameters["model_dim"],          # replaces d_model
            lstm_layers=2,                  # number of LSTM layers inside TFT
            num_attention_heads=local_parameters["attention_heads"],  # replaces nhead
            batch_size=local_parameters["batch_size"],
            n_epochs=local_parameters["num_epochs"],
            likelihood=None,
            pl_trainer_kwargs=local_parameters["pl_trainer_kwargs"],
            add_relative_index=True, 
            optimizer_kwargs={'lr': local_parameters["learning_rate"]},

        )
        
        # Measure Training Time
        if measure_time:
            start = time.perf_counter()

        model.fit(train_series.copy() , verbose=True, stride = forecast_horizons)

        if measure_time:
            end = time.perf_counter()
            print(f"Finished TFT training with rho={rho_value} and horizon={forecast_horizons} and architecture {device} in {end - start:0.4f} seconds \n")
        # END Measure Training Time

    else:
        scaler = StandardScaler()
        scaler.fit(test[:, 3:6])
        test[:, 3:6] = scaler.transform(test[:, 3:6])

        test_series = pd.DataFrame(test, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
        test_series  = TimeSeries.from_dataframe(test_series[local_parameters["features"]].copy()).astype(np.float32)
        
        print(f"Running TFT (pre-trained) for horizons {forecast_horizons} and rho {rho_value}:")

    # Measure Testing Time
    if measure_time:
        start = time.perf_counter()
    predictions_all = model.historical_forecasts(
        series=test_series,
        forecast_horizon=forecast_horizons,
        stride=forecast_horizons,
        retrain=False,
        verbose=True,
        start=local_parameters["input_length"],
        last_points_only=False,
        overlap_end=False
    )
    if measure_time:
        end = time.perf_counter()
        print(f"Finished TFT forecasting with rho={rho_value} and horizon={forecast_horizons} and architecture {device} in {end - start:0.4f} seconds \n")
    # END Measure Testing Time

    res = [ts.all_values()[..., 0] for ts in predictions_all]
    np_pred = np.vstack(res)

    offset = local_parameters["input_length"]
    n_pred_steps = np_pred.shape[0]

    np_test = test_series.values()
    aligned_test = np_test[offset : offset + n_pred_steps, :]

    # Sanity check
    assert aligned_test.shape == np_pred.shape, \
        f"Shape mismatch: test {aligned_test.shape} vs pred {np_pred.shape}"

    print(f"TFT Pipeline completed for horizons {forecast_horizons} and rho {rho_value}.")
    return aligned_test, np_pred

In [ ]:
def create_delta_combinations(rho_centers, delta):
    # Start from the center and extend outwards, in both directions, until the maximum delta is reached, with steps of 1
    combinations = []
    results = {}
    for rho_value in rho_centers:
        for d in range(-delta, delta + 1):
            combinations.append(rho_value + d)
        results[rho_value] = combinations.copy()  # Store the combinations for this center value
        combinations.clear()  # Clear the combinations list for the next center value

    return results

### UPO Prediction Main Function

In [ ]:
def UPO_prediction_window(
    traj_xyz,
    window_size=10,
    tree=None,
    labels=None,
    norm=2,
    k=3,
    upo_df=None,
    time_diff = 5
):
    
    """
        UPO based prediction for a trajectory using a sliding window approach.
        For each window of the trajectory, find the k nearest UPOs to the first point of the window, then extract the corresponding segments of those UPOs and compute their mean as the prediction for that window.

        Parameters:
        - traj_xyz: (N, 3) array of trajectory points
        - window_size: number of points in each window (forecast horizon)
        - tree: KDTree built on UPO points for fast nearest neighbor search
        - labels: array mapping tree indices to UPO identifiers
        - norm: distance metric for nearest neighbor search (e.g., 2 for Euclidean)
        - k: number of nearest UPOs to consider for each window
        - upo_df: DataFrame containing UPO trajectories and their starting indices

        Returns:
        - traj_xyz: original trajectory points
        - upo_all_stacked: all extracted UPO segments stacked together
        - upo_mean_xyz (prediction): mean trajectory of the extracted UPO segments for each window. The actual prediction.
        - upo_all_xyz: list of all extracted UPO segments for each window
    """
    
    n_points = traj_xyz.shape[0]
    n_windows = n_points // window_size
    
    if k > upo_df.shape[0]:
        k = upo_df.shape[0]
        
    # 1. Batch query the tree for all window starts at once
    # Only grab the first point of each window: traj_xyz[0, 10, 20...]
    start_points = traj_xyz[:n_windows * window_size:window_size]
    _, all_idxs = tree.query(start_points, k=k, p=norm)
        
    # Ensure idxs is 2D even if k=1
    if k == 1:
        all_idxs = all_idxs[:, np.newaxis]


    # 2. Pre-extract data from DataFrame to avoid slow indexing in loop
    # Mapping orbit_id -> (trajectory_array, start_idx)
    upo_lookup = {
        idx: (np.asarray(row["Trajectory"])[:, :3], row["start_idx"]) 
        for idx, row in upo_df.iterrows()
    }

    upo_mean_xyz = []
    upo_all_xyz = []
    
    # Pre-compute a range for window indexing
    window_range = np.arange(window_size)
        
    # 3. Process windows
    for w in range(n_windows):
        window_idxs = all_idxs[w]
        upo_segments = []

        for idx in window_idxs:
            orbit_id = labels[idx]
            upo_traj, start_idx = upo_lookup[orbit_id]

            local_idx = idx - start_idx
            upo_len = len(upo_traj)

            # Circular indexing via modulo
            indices = (local_idx + window_range*time_diff) % upo_len
            upo_segments.append(upo_traj[indices])

        upo_segments = np.array(upo_segments)
        upo_all_xyz.append(upo_segments)
        upo_mean_xyz.append(np.mean(upo_segments, axis=0))

    upo_all_stacked = np.concatenate(upo_all_xyz).reshape(-1, 3)
    predicted = np.concatenate(upo_mean_xyz).reshape(-1, 3)

    return traj_xyz, upo_all_stacked, predicted, upo_all_xyz



def UPO_Predict_Counter(
    traj_xyz,
    window_size=10,
    tree=None,
    labels=None,
    norm=2,
    k=3,
    upo_df=None,
    time_diff = 5
):
    
    n_points = traj_xyz.shape[0]
    n_windows = n_points // window_size
    
    if k > upo_df.shape[0]:
        k = upo_df.shape[0]
        
    start_points = traj_xyz[:n_windows * window_size:window_size]
    _, all_idxs = tree.query(start_points, k=k, p=norm)
        
    if k == 1:
        all_idxs = all_idxs[:, np.newaxis]

    upo_lookup = {
        idx: (np.asarray(row["Trajectory"])[:, :3], row["start_idx"]) 
        for idx, row in upo_df.iterrows()
    }

    upo_mean_xyz = []
    upo_all_xyz = []
    
    window_range = np.arange(window_size)

    # Count what UPOS and how many times they are used in the prediction
    upo_usage = [0] * (upo_df.shape[0] + 1)  # Initialize usage count for each UPO
    upo_frequency = [0] * (upo_df.shape[0] + 1)  # Initialize frequency for each UPO
    total_forecasts = 0  # Total number of forecasts made
    
    for w in range(n_windows):
        window_idxs = all_idxs[w]
        upo_segments = []

        for idx in window_idxs:
            orbit_id = labels[idx]
            upo_traj, start_idx = upo_lookup[orbit_id]
            upo_usage[orbit_id] += 1
            local_idx = idx - start_idx
            upo_len = len(upo_traj)

            indices = (local_idx + window_range*time_diff) % upo_len
            upo_segments.append(upo_traj[indices])

        total_forecasts += 1
        upo_segments = np.array(upo_segments)
        upo_all_xyz.append(upo_segments)
        upo_mean_xyz.append(np.mean(upo_segments, axis=0))

    upo_all_stacked = np.concatenate(upo_all_xyz).reshape(-1, 3)
    predicted = np.concatenate(upo_mean_xyz).reshape(-1, 3)

    # Convert values from usage to frequency based on total usage in a new column in the upo_df
    total_usage = sum(upo_usage)
    upo_frequency = [count / total_usage for count in upo_usage]

    return traj_xyz, upo_all_stacked, predicted, upo_all_xyz, upo_usage, upo_frequency, total_forecasts

def check_increase(arr, type="any_increase"):
    """
    Check whether an array is increasing
    -----
    monotonic              every consecutive step strictly increases
    non_decreasing         every step >= previous (flat sections allowed)
    any_increase           at least one consecutive step is positive
    majority               more than half of consecutive steps are positive
    regression_slope       OLS slope over the full sequence > 0
    """

    arr = np.asarray(arr, dtype=float)
    n = len(arr)
    diffs = np.diff(arr)

    if type == "monotonic":
        return bool(np.all(diffs > 0))

    elif type == "non_decreasing":
        return bool(np.all(diffs >= 0))

    elif type == "any_increase":
        return bool(np.any(diffs > 0))

    elif type == "majority":
        return bool(np.sum(diffs > 0) > np.sum(diffs <= 0))

    elif type in ("regression_slope", "slope_increase_regression"):
        slope = np.polyfit(np.arange(n), arr, 1)[0]
        return bool(slope > 0)

    else:
        raise ValueError(f"Unknown type: '{type}'")


def UPO_prediction_sliding(
    traj_xyz,
    window_size=10,
    tree=None,
    labels=None,
    norm=2,
    k=3,
    upo_df=None,
    time_diff = 5
):
    
    predicted_xyz = []   
    jump_to_new_upo = True
    counter = 0
    
    upo_lookup = {
        idx: (np.asarray(row["Trajectory"])[:, :3], row["start_idx"]) 
        for idx, row in upo_df.iterrows()
    }

    # Predict first window to initialize
    index = 0
    first_window = traj_xyz[index : index + window_size]

    _, idx = tree.query(first_window[0], k=1, p=norm)

    orbit_id = labels[idx]
    upo_traj, start_idx = upo_lookup[orbit_id]

    local_idx = idx - start_idx
    upo_len = len(upo_traj)
    predicted_window = np.zeros((window_size, 3)) 
    
    # Circular indexing via modulo
    for w in range(window_size):
        indices = (local_idx) % upo_len
        predicted_window[w] = upo_traj[indices]
        local_idx += time_diff

    err = np.linalg.norm(predicted_window - first_window, axis=1)
    jump_to_new_upo = check_increase(err)

    predicted_xyz.append(predicted_window)

    for w in range(window_size, traj_xyz.shape[0] - window_size + 1):

        window_points = traj_xyz[w - window_size+1 : w+1]
        
        start_point = traj_xyz[w]

        if jump_to_new_upo:
            _, idx = tree.query(start_point, k=1, p=norm)

            index = idx
            orbit_id = labels[index]
            upo_traj, start_idx = upo_lookup[orbit_id]

            upo_len = len(upo_traj)
            predicted_window = np.zeros((window_size, 3)) 
            counter = 0

        local_idx = index - start_idx
        indices = (local_idx) % upo_len
        index += time_diff

        pred = upo_traj[indices]

        # Keep only last window_size points for prediction and add latest one, np array
        counter += 1

        predicted_window[:-1] = predicted_window[1:]
        predicted_window[-1] = pred

        # error between predicted and actual point by point 
        err = np.linalg.norm(predicted_window - window_points, axis=1)
        # squared error
        err = np.sum((predicted_window - window_points)**2, axis=1)

        # if error increases monotonically, jump to another UPO else follow same UPO for next window
        if counter >= window_size:  # Only check after we have a full window of predictions
            jump_to_new_upo = check_increase(err)

        predicted_xyz.append(np.array(pred))

    predicted = np.vstack(predicted_xyz)
    return traj_xyz, predicted


### Hyperparam Helpers

In [ ]:
def hyperparameter_analysis(df, simulation_len=0, bins=0, smoothing=0, mase_steps=0, UPO = False):

    for model, df_model in df.groupby("model"):

        print(f"\n {'='*60}\n Model: {model} \n {'='*60}" )

        for hz, df_hz in df_model.groupby("horizon"):
            for index, row in df_hz.iterrows():
                predicted =  row["predicted"]
                observed = row["observed"]
                SL = row["SL"]

                # DONE
                metrics_temp = []

                first_sim_observed = observed[0:simulation_len - SL]
                first_sim_predicted = predicted[0:simulation_len - SL ]

                metrics_temp.append(compute_all_metrics(
                        first_sim_predicted, first_sim_observed,
                        bins=bins, smoothing=smoothing,
                        mase_steps=mase_steps, scale=False
                    ))
                
                for i in range(simulation_len, len(predicted)-simulation_len, simulation_len):
                
                    observed_temp = observed[i:i+simulation_len]
                    predicted_temp = predicted[i:i+simulation_len]

                    observed_temp = observed_temp[SL :]
                    predicted_temp = predicted_temp[SL :]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=bins, smoothing=smoothing,
                        mase_steps=mase_steps, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

                for col in ["mse", "sbd", "dstsp", "pe"]:
                    if col in metrics:
                        df.loc[index, col] = metrics[col]

    return df


In [ ]:

def hyperparameter_analysis_upo(df, simulation_len=0, bins=0, smoothing=0, mase_steps=0):

    for model, df_model in df.groupby("model"):

        print(f"\n {'='*60}\n Model: {model} \n {'='*60}" )

        for hz, df_hz in df_model.groupby("horizon"):
            for index, row in df_hz.iterrows():
                predicted =  row["predicted_scaled"]
                observed = row["observed_scaled"]

                # DONE
                metrics_temp = []
                for i in range(0, len(predicted)-simulation_len, simulation_len):
                
                    observed_temp = observed[i:i+simulation_len]
                    predicted_temp = predicted[i:i+simulation_len]

                    observed_temp = observed_temp[1:]
                    predicted_temp = predicted_temp[1:]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=bins, smoothing=smoothing,
                        mase_steps=mase_steps, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

                for col in ["mse", "sbd", "dstsp", "pe"]:
                    if col in metrics:
                        df.loc[index, col] = metrics[col]

    return df

## **Loaders, datasets, and model params**

In [ ]:
# |0: Simulation | 1: Sub-Sim (not used) | 2: Time | 3: X | 4: Y | 5: Z | 6: RHO |
NAME = "LORENZ"

MAX_SIMULATIONS_TRAIN = 3
MAX_SIMULATIONS_TEST = 3

DT = 0.05

EXPERIMENT_NAME = "dt005_sim10_all_rhos_500"


ATTENTION = False
TEACHER_FORCING = False

# 28, 50, 100, 150, 175, 210, 
RHO = 28

# For LSTMS, UPOs, and data loaders. 

if DT == 0.01:
    forecast_horizons = [100]
    models_parameters['seq_len'] = 1280 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 256 #128 # 64
    TRANSIENT_LEN = 500
elif DT == 0.03:
    forecast_horizons = [40]
    models_parameters['seq_len'] = 512 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 128 #128 # 64
    TRANSIENT_LEN = 300
else:
    forecast_horizons = [20]
    models_parameters['seq_len'] = 512 #512 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 32 #128 # 64
    TRANSIENT_LEN = 200


# 005 - 300 epochs, 128 batch size, 256 seq len, horizon 10
# 003 - 300 epochs, 128 batch size, 512 seq len, horizon 30
# 001 - 300 epochs, 256 batch size, 1280 seq len, horizon 50

models_parameters['model_type'] = 'LSTM'  # 'LSTM' 'GRU' 'RNN' 'MLP'
models_parameters['epochs'] = 300
models_parameters['hidden_size'] = [32, 16] #[64, 32] # [32], [64, 32], [64, 16]
models_parameters['stateful'] = True
models_parameters['LR'] = 0.01
models_parameters['rolling'] = True
models_parameters['loss'] = 'mse'
models_parameters['decay_rate'] = 0.95 #0.95
models_parameters['decay_step'] = 100
models_parameters['optimizer'] = 'Adam'
models_parameters['seed'] = 42
models_parameters['forecast_size'] = forecast_horizons[0]
models_parameters['metrics'] = ['mae']
models_parameters['TF'] = False
models_parameters['MISO'] = False

if TEACHER_FORCING:
    models_parameters['dropout'] = 0.2
    models_parameters['recurrent_dropout'] = 0.2
    models_parameters['l2_reg'] = 1e-4
    models_parameters['clipnorm'] = 1.5
else:
    models_parameters['dropout'] = 0.0
    models_parameters['recurrent_dropout'] = 0.0
    models_parameters['l2_reg'] = 0.0
    models_parameters['clipnorm'] = None
    
# DATA COLUMNS:
# |0: Simulation | 1: Sub-Sim (not used) | 2: Time | 3: X | 4: Y | 5: Z | 6: RHO |
datasets_parameters['LORENZ']['outputs'] = [3, 4, 5]
datasets_parameters['LORENZ']['inputs'] = [3,4, 5]

# BASED ON WHAT YOU SELECT, MAKE SURE THE CORRECT EXPERIMENT IS LOADED!
DT_STR = format(DT, 'g').replace('.', '')

res = DataHandler.load_exp_data_(EXPERIMENT_NAME, MAX_SIMULATIONS_TRAIN, MAX_SIMULATIONS_TEST, TRANSIENT_LEN, datasets_parameters, NAME, DT = DT)
SIMULATION_LEN, sim_step, SIMULATIONS, train_o, test_o, all_data_o = res

TOTAL_POINTS = test_o.shape[0]

copy_train = train_o.copy()
copy_test = test_o.copy()

In [ ]:
# Isolate data based on rho (If needed)

training_rho = [RHO]
testin_rhos = [RHO]

SCALE_PER_RHO = True

train_mask = np.isin(copy_train[:, 6], training_rho)
test_mask  = np.isin(copy_test[:, 6], testin_rhos)

train = copy_train[train_mask]
test  = copy_test[test_mask]

all_data = np.vstack((train, test))

forecast_horizons = [10]

In [ ]:
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
rho_vector = [28, 50, 100, 150, 175, 210]

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10

In [ ]:


# diferent colors for each line
color_list = ['#FF4242', '#42FF42', '#4242FF']
color_list_dark = ['#8B0000', '#006400', '#00008B']

fig, axes = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
fig.subplots_adjust(hspace=0)

labels = ['X', 'Y', 'Z']
for i, ax in enumerate(axes):
    ax.plot(train[2000:2500, 3 + i], color=color_list[i])
    ax.set_ylabel(labels[i])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

axes[-1].set_xlabel('Time')
plt.show()

fig, axes = plt.subplots(3, 1, figsize=(10, 4), sharex=True)
fig.subplots_adjust(hspace=0)

labels = ['X', 'Y', 'Z']
for i, ax in enumerate(axes):
    ax.plot(train[5000:5500, 3 + i], color=color_list_dark[i])
    ax.set_ylabel(labels[i])
    ax.set_xticks([])
    ax.set_yticks([])
    for spine in ax.spines.values():
        spine.set_visible(False)

axes[-1].set_xlabel('Time')
plt.show()


## **UPO Forecasting Evaluation**

### **Data Visualizations**

In [ ]:
def plot3D_a_given_rho(data, rho):
    x = data[data[:, 6] == rho, 3]
    y = data[data[:, 6] == rho, 4]
    z = data[data[:, 6] == rho, 5]
    print("Mean Z: ", np.mean(z))

    fig = plt.figure(figsize=(18, 10))

    # 3D plot
    ax3d = fig.add_subplot(141, projection='3d')
    ax3d.plot(x[1000:2000], y[1000:2000], z[1000:2000])
    ax3d.set_title(f"3D  Rho = {rho}")

    # XY projection
    ax_xy = fig.add_subplot(142)
    ax_xy.plot(x[1000:2000], y[1000:2000], lw=0.7)
    ax_xy.set_title(f"XY  Rho = {rho}")
    ax_xy.set_xlabel("X")
    ax_xy.set_ylabel("Y")

    # XZ projection
    ax_xz = fig.add_subplot(143)
    ax_xz.plot(x[1000:2000], z[1000:2000], lw=0.7)
    ax_xz.set_title(f"XZ  Rho = {rho}")
    ax_xz.set_xlabel("X")
    ax_xz.set_ylabel("Z")

    # YZ projection
    ax_yz = fig.add_subplot(144)
    ax_yz.plot(y[1000:2000], z[1000:2000], lw=0.7)
    ax_yz.set_title(f"YZ  Rho = {rho}")
    ax_yz.set_xlabel("Y")
    ax_yz.set_ylabel("Z")

    plt.suptitle(f"Lorenz Attractor Projections  (Rho = {rho})", fontsize=13)
    plt.tight_layout()
    plt.show()

rho_z_values_safe = {}
rho_values_all = np.unique(train_o[:, 6])
plot3D_a_given_rho(train_o, 28)

### **Extract UPOS**

#### **Extract and recreate UPOS - DONT RUN IF NOT NEEDED!**

In [ ]:
# CREATE DATA (DONT RUN IF NOT NEEDED)

DT_UPOS = 0.001

RHO_RANGE = range(25, 225, 1)

# RHO_RANGE = range(210, 211, 1)

RHO_RANGE = [28, 50, 100, 150, 175, 210]
# RHO_RANGE = [100]
# RHO_RANGE = [100, 150, 175, 210]
# RHO_RANGE = [175]
# RHO_RANGE = [150]

DATA_DIR = rf"data_upos_original/"
# DATA_DIR = rf"data/DATA1_{RHO}"
OUTPUT_DIR = "UPOS"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for RHO in RHO_RANGE:
    DATA_DIR_RHO = os.path.join(DATA_DIR, f"DATA1_{RHO}")
    os.makedirs(DATA_DIR_RHO, exist_ok=True)

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS}.pkl")
    df_orbits_individual = load_all_orbits_parallel(DATA_DIR_RHO, rho=RHO, dt=DT_UPOS)
    df_orbits_individual = df_orbits_individual.sort_values("K")
    df_orbits_individual.to_pickle(final_filename)

    print(f"Saved orbits for rho {RHO} to {final_filename}")

# Optional plot
# plot_multiple_trajectory_comparisons(df_orbits, n_orbits=8, data_dir=DATA_DIR)

#### **LOAD UPOS FROM PKL - IF YOU CREATED THEM ONCE!**

In [ ]:
# [0.001,0.003, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1]
DT_UPOS = 0.001

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]

# RHO_TO_LOAD = [175]
# RHO_TO_LOAD = [150]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# DATA_DIR = rf"data_upos_original/DATA1_{RHO}"

upo_data_multi_rho = {}
for RHO in RHO_TO_LOAD:

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho[RHO] = df_orbits_individual

In [ ]:
reset_matplotlib_defaults()
PLOT_SOME_UPOS_FOR_RHOs = [28, 50, 175, 210]
PLOT_SOME_UPOS_FOR_RHOs = [28]
plot_dir = "plots/UPOS_general_visualization/"
os.makedirs(plot_dir, exist_ok=True)


NR_UPOS = 10
df_orbits = []
for R_R in PLOT_SOME_UPOS_FOR_RHOs:
    for idx in range(NR_UPOS):
        df_orbits.append(upo_data_multi_rho[R_R].iloc[idx]["Trajectory"])

PLOT_3D = False 

for i, traj in enumerate(df_orbits):
    if PLOT_3D:
        fig = plt.figure(figsize=(5, 4))
        ax = fig.add_subplot(111, projection='3d')
        ax.plot(traj[:, 0], traj[:, 1], traj[:, 2])
        ax.set_title(rf"$\rho = {PLOT_SOME_UPOS_FOR_RHOs[i]}$")
        ax.set_xlabel("X")
        ax.set_ylabel("Y")
        ax.set_zlabel("Z")
    else:
        fig, ax = plt.subplots(figsize=(8, 8))
        ax.plot(traj[:, 0], traj[:, 2])
        if NR_UPOS == 1:
            ax.set_title(rf"$\rho = {PLOT_SOME_UPOS_FOR_RHOs[i]}$")
        if i == 0:
            ax.set_ylabel("Z")
        ax.set_xlabel("X")

    
        plt.tight_layout()
        if NR_UPOS == 1:
            plt.savefig(os.path.join(plot_dir, f"upo_rho_{PLOT_SOME_UPOS_FOR_RHOs[i]}_dt{DT_UPOS}.pdf"), format="pdf", bbox_inches="tight")

        plt.show()

In [ ]:
# Plot some trajectories for a given rho (UPOs)   

RHO = 28

DT_UPOS = 0.001

PLOT_SOME_UPOS = True
PLOT_SOME_UPOS_FOR_RHOs = [28, 50, 175]
DATA_DIR = rf"data_upos_original/DATA1_{RHO}"

OUTPUT_DIR = "UPOS"

if PLOT_SOME_UPOS:
    df_orbits_individual = load_all_orbits_parallel(DATA_DIR, rho=RHO, dt=DT_UPOS)
    plot_multiple_trajectory_comparisons(df_orbits_individual, n_orbits=10, ncols=5, data_dir=DATA_DIR)



### **Extract and View Some UPOS and DATA**

In [ ]:
# Cols data = |0: Simulation | 1: Sub-Sim (not used) | 2: Time | 3: X | 4: Y | 5: Z | 6: RHO |
# Cols UPO = 'Word', 'K', 'X0_x', 'X0_y', 'X0_z', 'Trajectory', 'T', 'dt'
all_data_df = pd.DataFrame(all_data_o, columns=["Simulation", "Sub-Sim", "Time", "X", "Y", "Z", "RHO"])

print("All Data Shape:", all_data_df.shape)

In [ ]:
from matplotlib.collections import LineCollection
from mpl_toolkits.mplot3d.art3d import Line3DCollection

reset_matplotlib_defaults()

RHO = 100
START_POINT = 7000
END_POINT = 12000
LIMIT_POINTS = 5000
plot_type = "3D"  # "3D" or "2D"
maximum_orbits_to_plot = 70
ticks_on_axes = False

STYLE = "twilight"  # "plasma", "viridis", "cool", "twilight"

data = all_data_df[all_data_df["RHO"] == RHO].to_numpy()
x, y, z = data[START_POINT:END_POINT, 3], data[START_POINT:END_POINT, 4], data[START_POINT:END_POINT, 5]
n = LIMIT_POINTS
cmap = plt.get_cmap(STYLE)

if plot_type == "3D":
    fig = plt.figure(figsize=(10, 8))
    ax = fig.add_subplot(111, projection='3d')

    pts = np.array([x, y, z]).T.reshape(-1, 1, 3)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    t = np.linspace(0, 1, len(segs))

    lc = Line3DCollection(segs, cmap=cmap)
    lc.set_array(t)
    ax.add_collection3d(lc)

    # ax.set_xlim(x[:n].min(), x[:n].max())
    # ax.set_ylim(y[:n].min(), y[:n].max())
    # ax.set_zlim(z[:n].min(), z[:n].max())
    # ax.xaxis.pane.fill = False
    # ax.yaxis.pane.fill = False
    # ax.zaxis.pane.fill = False
    # ax.xaxis.pane.set_edgecolor('none')
    # ax.yaxis.pane.set_edgecolor('none')
    # ax.zaxis.pane.set_edgecolor('none')
    ax.grid(False)
    # ax.set_axis_off()

else:
    fig, ax = plt.subplots(figsize=(10, 8))

    pts = np.array([x, z]).T.reshape(-1, 1, 2)
    segs = np.concatenate([pts[:-1], pts[1:]], axis=1)
    t = np.linspace(0, 1, len(segs))

    lc = LineCollection(segs, cmap=cmap, alpha=0.85)
    lc.set_array(t)
    ax.add_collection(lc)
    ax.autoscale()

ax.set_xticks([])
ax.set_yticks([])
if plot_type == "3D":
    ax.set_zticks([])

ax.set_title(f"Rho = {RHO}", fontsize=14)
plt.tight_layout()
plt.show()




fig, ax = plt.subplots(figsize=(10, 8))

orbit_indices = upo_data_multi_rho[RHO].index[:maximum_orbits_to_plot]
orbit_indices = orbit_indices[:maximum_orbits_to_plot]

for i, orbit_idx in enumerate(orbit_indices):
    row = upo_data_multi_rho[RHO].loc[orbit_idx]
    ax.plot(row["Trajectory"][:, 0], row["Trajectory"][:, 2],
            label=f"Orbit {i+1}")

ax.set_title(f"Rho = {RHO}")
plt.show()


### **UPO statistics, including shadowing time and closeness to UPO**

In [ ]:
def compute_shadowing_time_per_rho(df_test, df_upo, epsilon):
    shadowing_times = []

    for idx, row in df_upo.iterrows():
        upo_traj = row["Trajectory"]
        upo_time = row["T"]

        min_len = min(len(df_test), len(upo_traj))
        test_traj = df_test[:min_len, 3:6]

        # Compute distance at each time step
        dists = np.linalg.norm(test_traj - upo_traj[:min_len], axis=1)

        shadowing_time = upo_time  

        for t in range(min_len):
            if dists[t] > epsilon:
                shadowing_time = t * epsilon
                break

        shadowing_times.append(shadowing_time)

    return shadowing_times

reset_matplotlib_defaults()


rho_vector = [28, 50, 100, 150, 175, 210]
rho_vector = [28]
DT_UPOS_SHADOWING = 0.001

assert DT_UPOS_SHADOWING <= DT, "UPO data must have a time step greater than or equal to the test data time step for shadowing time computation."

RHO_TO_LOAD = [28]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

results = []

upo_data_multi_rho = {}
for RHO in RHO_TO_LOAD:

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS_SHADOWING}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho[RHO] = df_orbits_individual

for rho_value in rho_vector:
    
    df_upo = upo_data_multi_rho[rho_value]

    # Create KDTree for test data
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]

    # Having upo data and test data, lets compute shadowing time (longest)
    total_close, longest_streak, close_mask = upo_closeness_stats(test_data[:, 3:6], tree = tree, eps = 0.5, mode="shadow", labels=labels)

    results.append({
        "rho": rho_value,
        "total_close": total_close,
        "longest_streak": longest_streak,
        "close_mask": close_mask,
    })

df_shadowing = pd.DataFrame(results)


In [ ]:
RHO_VALUE_PLOTS = 28
eps_values = [0.05, 0.5, 1.0]
dir_plots = "plots/upo_shadowing/"
os.makedirs(dir_plots, exist_ok=True)

orbits_originals_counts = []
orbits_original_counts_test = []

orbits_originals_longest = []
orbits_original_longest_test = []

df_upo = upo_data_multi_rho[RHO_VALUE_PLOTS]

# Create KDTree for test data
tree, labels, df_upo_extended = build_labeled_tree(df_upo)

test_mask = np.isin(test_o[:, 6], RHO_VALUE_PLOTS)
test_data = test_o[test_mask]
train_data = train_o[np.isin(train_o[:, 6], RHO_VALUE_PLOTS)]

dataxyz_test = test_data[:, 3:6]
dataxyz_train = train_data[:, 3:6]

dataxyz = np.vstack((dataxyz_train, dataxyz_test))

for idx, eps in enumerate(eps_values):

    print(f"Computing shadowing stats for epsilon = {eps}...")
    orbit_counts_original, orbit_longest_original = upo_shadow_traj(dataxyz_train, tree, eps, labels)
    orbit_counts_original_test, orbit_longest_original_test = upo_shadow_traj(dataxyz_test, tree, eps, labels)

    orbits_originals_counts.append(orbit_counts_original)
    orbits_originals_longest.append(orbit_longest_original)

    orbits_original_counts_test.append(orbit_counts_original_test)
    orbits_original_longest_test.append(orbit_longest_original_test)

    print(f"Done with epsilon = {eps}")


In [ ]:
dir_plots = "plots/upo_shadowing/"
os.makedirs(dir_plots, exist_ok=True)

total_points_plot_shadowing = dataxyz.shape[0]

for idx, eps in enumerate(eps_values):

    orbit_long = orbits_originals_longest[idx]
    orbit_long = orbit_long * DT # convert to time
    orbit_long_test = orbits_original_longest_test[idx]
    orbit_long_test = orbit_long_test * DT # convert to time

    plt.figure(figsize=(10, 5))

    x1 = range(len(orbit_long))
    plt.scatter(x1, orbit_long,alpha=0.7, label="Train Data")
    plt.scatter(x1, orbit_long_test, alpha=0.7, label="Test Data")

    plt.title(rf"$\varepsilon = {eps}$")
    plt.xlabel("UPO Index")
    plt.ylabel("Longest Time")
    plt.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(dir_plots, rf"UPO_longest_train_test_{DT_UPOS}_{RHO_VALUE_PLOTS}_{eps}.pdf"), format="pdf")
    plt.show()


In [ ]:
eps_values = [0.01, 0.03, 0.05, 0.1, 0.3, 0.5, 1.0]

dir_plots = "plots/upo_shadowing/"
os.makedirs(dir_plots, exist_ok=True)


rhos_to_load = [28, 50, 100, 150, 175, 210]
rhos_to_load = [50] 

# upo_dt_ranges = [0.001, 0.003, 0.005, 0.01, 0.03,  0.05]
upo_dt_ranges = [0.001,0.003, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

upo_data_multi_rho_shadowing = defaultdict(dict)

for RHO in rhos_to_load:
    for custom_dt in upo_dt_ranges:
        custom_dt_rounded = round(custom_dt, 3)
        final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{custom_dt_rounded}.pkl")
        df_orbits_individual = pd.read_pickle(final_filename)
        df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
        print(f"Loaded DataFrame from {final_filename}")
        upo_data_multi_rho_shadowing[RHO][custom_dt_rounded] = df_orbits_individual

results_shadowing_sensitivity = []

for rho_value in rhos_to_load:

    # Load Rho Data
    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]

    train_data = train_o[np.isin(train_o[:, 6], rho_value)]

    dataxyz_test = test_data[:, 3:6]
    dataxyz_train = train_data[:, 3:6]
    
    for dt_val in upo_dt_ranges:
        # Load UPO data for this rho and dt
        df_upo = upo_data_multi_rho_shadowing[rho_value][round(dt_val, 3)]
    
        # Create KDTree for test data
        tree, labels, df_upo_extended = build_labeled_tree(df_upo)

        for idx, eps in enumerate(eps_values):

            print(f"Computing shadowing stats for epsilon = {eps}...")

            _, orbit_longest_original = upo_shadow_traj(dataxyz_train, tree, eps, labels)
            _, orbit_longest_original_test = upo_shadow_traj(dataxyz_test, tree, eps, labels)

            results_shadowing_sensitivity.append({
                "rho": rho_value,
                "eps": eps,
                "dt": dt_val,
                "orbit_longest_original": orbit_longest_original,
                "orbit_longest_original_test": orbit_longest_original_test,
            })

            print(f"Done with epsilon = {eps}")

df_shadowing_sensitivity = pd.DataFrame(results_shadowing_sensitivity)

# Sort this stuff by rho, then dt, then eps for easier plotting later
df_shadowing_sensitivity = df_shadowing_sensitivity.sort_values(["rho", "eps", "dt"]).reset_index(drop=True)

FOLER_or_DIR_PLOTS = "outer_limits_results/upo_shadowing_sensitivity/"
os.makedirs(FOLER_or_DIR_PLOTS, exist_ok=True)

df_shadowing_sensitivity.to_pickle(os.path.join(FOLER_or_DIR_PLOTS, rf"{rhos_to_load[0]}_shadowing_sensitivity.pkl"))

In [ ]:
FOLER_or_DIR_PLOTS = "outer_limits_results/upo_shadowing_sensitivity/"
rhos_to_load = [28] 

file_name = rf"{rhos_to_load[0]}_shadowing_sensitivity.pkl"

path_load = os.path.join(FOLER_or_DIR_PLOTS, file_name)

df_shadowing_sensitivity = pd.read_pickle(path_load)
df_shadowing_sensitivity = df_shadowing_sensitivity.sort_values(["rho", "eps", "dt"]).reset_index(drop=True)

In [ ]:
reset_matplotlib_defaults()

rho_values = df_shadowing_sensitivity["rho"].unique()
dt_values = df_shadowing_sensitivity["dt"].unique()
eps_values = df_shadowing_sensitivity["eps"].unique()

dt_values_to_plot = [0.001,0.003, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1]

dt_average_shadowing = df_shadowing_sensitivity.copy()
plots_dir = "plots/upo_shadowing_sensitivity/"
os.makedirs(plots_dir, exist_ok=True)

# Average orbit_longest_original and orbit_longest_original_test for each row! These are arrays over all UPOS
# dt_average_shadowing["orbit_longest_original"] = dt_average_shadowing["orbit_longest_original"].apply(lambda x: np.mean(x))
dt_average_shadowing["orbit_longest_original_test"] = dt_average_shadowing["orbit_longest_original_test"].apply(lambda x: np.mean(x))


for rho in rho_values:
    df_rho = dt_average_shadowing[dt_average_shadowing["rho"] == rho]
    fig, ax = plt.subplots(figsize=(12, 6))

    for eps in eps_values:
        df_eps = df_rho[df_rho["eps"] == eps].sort_values("dt")
        # ax.plot(df_eps["dt"], df_eps["orbit_longest_original"], marker="o", label=rf"$\varepsilon={eps}$ (Train)")
        ax.plot(df_eps["dt"], df_eps["orbit_longest_original_test"], label=rf"$\varepsilon={eps}$")

    ax.set_xscale("log")
    ax.set_xticks(dt_values_to_plot)
    # ax.get_xaxis().set_major_formatter(matplotlib.ticker.ScalarFormatter())
    ax.xaxis.set_major_formatter(
     matplotlib.ticker.FuncFormatter(lambda x, _: f"{x:g}")
    )
    plt.xticks(rotation=45)

    ax.set_xlabel("UPO Time Step (dt)")
    ax.set_ylabel("Average Maximum Steps", fontsize=22)
    ax.set_ylim([0, 315])
    ax.legend(ncols = 2)
    plt.title(rf"Shadowing Average Maximum Steps for $\rho = {rho}$", fontsize=24)
    plt.tight_layout()
    save_path = os.path.join(plots_dir, f"shadowing_sensitivity_rho_{rho}.pdf")
    plt.savefig(save_path, bbox_inches="tight", format="pdf")
    plt.show()



### **UPO shadowing time vs nr of UPOs for context**

In [ ]:
from itertools import product

RHO_VALUE_PLOTS = 50
eps_values = [0.05, 0.5, 1.0]

dir_plots = "plots/upo_shadowing/"
os.makedirs(dir_plots, exist_ok=True)

DT_TESTS_FOR_NR_OF_UPOS = 0.001
# NUMBER_OF_UPOS_TO_TEST = range(0, 1350, 20)
NUMBER_OF_UPOS_TO_TEST = range(0, 655, 10)

test_mask = np.isin(test_o[:, 6], RHO_VALUE_PLOTS)
test_data = test_o[test_mask]
train_data = train_o[np.isin(train_o[:, 6], RHO_VALUE_PLOTS)]

dataxyz_test = test_data[:, 3:6]
dataxyz_train = train_data[:, 3:6]
dataxyz = np.vstack((dataxyz_train, dataxyz_test))

def compute_shadowing(eps, nr_of_upos, upo_data, rho, dataxyz):
    if nr_of_upos == 0:
        nr_of_upos = 1

    df_upo = upo_data[rho][0:nr_of_upos]

    tree, labels, _ = build_labeled_tree(df_upo)

    print(f"Computing shadowing stats for upos={nr_of_upos} and epsilon={eps}...")

    _, orbit_longest_original = upo_shadow_traj(dataxyz, tree, eps, labels)

    print(f"Done with upos={nr_of_upos} and epsilon={eps}.")

    del tree, labels, df_upo, dataxyz, upo_data
    gc.collect()
    
    return {
        "eps": eps,
        "nr_of_upos": nr_of_upos,
        "orbit_longest_original": orbit_longest_original.max(),
    }

results = []
combos = list(product(eps_values, NUMBER_OF_UPOS_TO_TEST))

with ThreadPoolExecutor(max_workers=4) as executor:  
    futures = {
        executor.submit(compute_shadowing, eps, nr_of_upos, upo_data_multi_rho.copy(), RHO_VALUE_PLOTS, dataxyz): (eps, nr_of_upos)
        for eps, nr_of_upos in combos
    }
    for future in as_completed(futures):
        results.append(future.result())

df_shadowing_nr_upos = pd.DataFrame(results)
df_shadowing_nr_upos = df_shadowing_nr_upos.sort_values(["eps", "nr_of_upos"]).reset_index(drop=True)

# Save
FOLER_or_DIR_PLOTS = "outer_limits_results/upo_shadowing_sensitivity/"
os.makedirs(FOLER_or_DIR_PLOTS, exist_ok=True)

file_name = rf"{RHO_VALUE_PLOTS}_shadowing_time_vs_nr_upos.pkl"
path_load = os.path.join(FOLER_or_DIR_PLOTS, file_name)
df_shadowing_nr_upos.to_pickle(path_load)

In [ ]:
# Load if saved
FOLER_or_DIR_PLOTS = "outer_limits_results/upo_shadowing_sensitivity/"

df_shadowing_nr_upos_all = {}

for rhvtp in [28, 50]:
    file_name = rf"{rhvtp}_shadowing_time_vs_nr_upos.pkl"
    path_load = os.path.join(FOLER_or_DIR_PLOTS, file_name)

    df_shadowing_nr_upos = pd.read_pickle(path_load)
    df_shadowing_nr_upos = df_shadowing_nr_upos.sort_values(["eps", "nr_of_upos"]).reset_index(drop=True)
    df_shadowing_nr_upos_all[rhvtp] = df_shadowing_nr_upos

In [ ]:
reset_matplotlib_defaults()
dir_plots = "plots/upo_shadowing/"
os.makedirs(dir_plots, exist_ok=True)

for rhvtp in [28, 50]:
    df_shadowing_nr_upos = df_shadowing_nr_upos_all[rhvtp]
    fig, ax = plt.subplots(figsize=(10, 5))

    for eps, df_eps in df_shadowing_nr_upos.groupby("eps"):
        df_eps = df_eps.sort_values("nr_of_upos")
        x = df_eps["nr_of_upos"]
        y = df_eps["orbit_longest_original"] * DT  # convert to time
        ax.plot(x, y, marker="o", label=rf"$\varepsilon={eps}$")

    ax.set_xlabel("Number of Context UPOs")
    ax.set_ylabel("Max Shadowing Time")
    ax.set_title(rf"$\rho = {rhvtp}$")
    # ax.set_ylim(0, 50)
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(dir_plots, f"UPO_longest_vs_nr_upos_{DT_UPOS}_{rhvtp}.pdf"), format="pdf")
    plt.show()

### **UPO FORECASTING OVERALL**

In [ ]:
reset_matplotlib_defaults()

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

# rho_vector = [175]
# horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

# rho_vector = [150]
# horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

K = 30

MAX_PLOTS = 500

START_PLOT_IDX = 0
END_PLOT_IDX = MAX_PLOTS

scaler = StandardScaler()

results = []

for rho_value in rho_vector:
    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]
    df_upo = upo_data_multi_rho[rho_value]

    obs_xyz = test_data[:, 3:6]
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    for tau in horizons_experiments:
        print(f"Processing Rho={rho_value}, Tau={tau}...")

        traj_xyz, upo_all_stacked, pred_xyz, upo_segments = UPO_prediction_window(
            obs_xyz,
            window_size=tau,
            tree=tree,
            labels=labels,
            norm=2,
            k=K,
            upo_df=df_upo_extended,
            time_diff=int(DT / DT_UPOS)
        )

        start = 0
        # for indx, segment in enumerate(upo_segments):
        #     plot_upo_window(segment, traj_xyz[start:start + tau], pred_xyz[start:start + tau], K)
        #     start += tau

        scaler.fit(np.vstack([obs_xyz, pred_xyz]))
        obs_xyz_scaled = scaler.transform(obs_xyz)
        pred_xyz_scaled = scaler.transform(pred_xyz)

        results.append({"rho": rho_value, 
                        "tau": tau, 
                        "observed": obs_xyz, 
                        "predicted": pred_xyz,
                        "observed_scaled": obs_xyz_scaled,
                        "predicted_scaled": pred_xyz_scaled
                       })
        
        print(f"Done Rho={rho_value}, Tau={tau}.")

df_results_upo = pd.DataFrame(results).sort_values(by=["rho", "tau"])
df_UPO_results = df_results_upo.copy()
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

pd.to_pickle(df_results_upo, FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

pd.to_pickle(df_results_upo, FULL_PATH)

In [ ]:
# horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
# rho_vector = [28, 50, 100, 150, 175, 210]
# If you want to load the results later, use this code but set the horizons_experiments and rho_vector to what you used in the experiment

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_UPO_results = pd.read_pickle(FULL_PATH)

In [ ]:
PLOT_MAX_VALUES = 1500
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10
METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

df_UPO_results["model"] = "UPO"

reset_matplotlib_defaults()


for METRIC_PLT in METRICS_TO_PLOT:
    plt.figure(figsize=(10, 5))
    for rho, df_rho in df_UPO_results.groupby("rho"):
        rho_met = []
        horizons = []
        for index, row in df_rho.iterrows():
            model_name = row['model']
            horizon = row['tau']
            predicted = row['predicted_scaled']
            observed = row['observed_scaled']
            
            # DONE
            metrics_temp = []

            first_sim_observed = observed[0:SIMULATION_LEN-512 ]
            first_sim_predicted = predicted[0:SIMULATION_LEN-512 ]

            metrics_temp.append(compute_all_metrics(
                    first_sim_predicted, first_sim_observed,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))
            
            for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
                observed_temp = observed[i:i+SIMULATION_LEN]
                predicted_temp = predicted[i:i+SIMULATION_LEN]

                observed_temp = observed_temp[512 :]
                predicted_temp = predicted_temp[512 :]

                metrics_temp.append(compute_all_metrics(
                    predicted_temp, observed_temp,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))

            metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
            metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
            
            rho_met.append(metrics)
            horizons.append(horizon)


        plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
    plt.title(fr"UPO predictor {METRIC_PLT.upper()} per $\rho$ and $\tau$")
    plt.xlabel("Forecast Horizons")
    plt.ylabel(rf"{METRIC_PLT.upper()}")
    if METRIC_PLT == "dstsp":
        plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
    plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
    plt.legend()

    file_name_plot = (
    rf"{model_name}_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
    + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
    )

    FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
    plt.tight_layout()
    plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# df_results_upo
# Columns are: 'rho', 'tau', 'observed', 'predicted'
# Lets plot some observed vs predicted for different rho and tau    
[28, 50, 100, 150, 175, 210]
[1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

RHO_PLOT = 175
TAU_PLOT = 30

MAX_POINTS_PLOT = 500
plot_3d_max_values = 500

subset = df_UPO_results[(df_UPO_results["rho"] == RHO_PLOT) & (df_UPO_results["tau"] == TAU_PLOT)]
obs_xyz = subset["observed_scaled"].values[0]
pred_xyz = subset["predicted_scaled"].values[0]

fig = plot_3D_attractor(obs_xyz[0:0,:], pred_xyz[0:plot_3d_max_values,:], ground_truth=obs_xyz[0:plot_3d_max_values,:], lim_pse=MAX_POINTS_PLOT, lim_gen=MAX_POINTS_PLOT, horizon = TAU_PLOT, title_3D="UPOs", title_big = f"Rho={RHO_PLOT}, Tau={TAU_PLOT}")
plt.show()

In [ ]:
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10
SCALED = True

all_metrics_short = []
all_metrics_long = []

by_tf_short = defaultdict(list)
by_tf_long  = defaultdict(list)


for index, row in df_UPO_results.iterrows():
    if SCALED:
        observed  = row['observed_scaled']
        predicted = row['predicted_scaled']
    else:
        observed  = row['observed']
        predicted = row['predicted']
        
    horizon    = row['tau']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[models_parameters['seq_len'] :]
        predicted_temp = predicted_temp[models_parameters['seq_len'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    
    if FORECAST_TYPES_RANGES['short-term'][0] <= horizon <= FORECAST_TYPES_RANGES['short-term'][1]:
        all_metrics_short.append(metrics)
    elif FORECAST_TYPES_RANGES['long-term'][0] <= horizon <= FORECAST_TYPES_RANGES['long-term'][1]:
        all_metrics_long.append(metrics)

metric_keys = all_metrics_short[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_short]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_short]) for k in metric_keys}

print(f"\n UPO short-term averaged over {len(all_metrics_short)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")

metric_keys = all_metrics_long[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_long]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_long]) for k in metric_keys}

print(f"\n UPO long-term averaged over {len(all_metrics_long)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")


### **UPO FORECASTING GENERAL WITH JUMPS!**

In [ ]:
reset_matplotlib_defaults()

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

rho_vector = [175]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

rho_vector = [150]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

rho_vector = [28, 50, 100, 150, 175, 210]

K = 1

MAX_PLOTS = 500

START_PLOT_IDX = 0
END_PLOT_IDX = MAX_PLOTS

scaler = StandardScaler()

results = []

WINDOW_SIZE = 10

for rho_value in rho_vector:

    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]
    df_upo = upo_data_multi_rho[rho_value]

    obs_xyz = test_data[:, 3:6]
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    traj_xyz, pred_xyz = UPO_prediction_sliding(
        obs_xyz,
        window_size=WINDOW_SIZE,
        tree=tree,
        labels=labels,
        norm=2,
        k=K,
        upo_df=df_upo_extended,
        time_diff=int(DT / DT_UPOS)
    )

    start = 0
    scaler.fit(np.vstack([obs_xyz, pred_xyz]))
    obs_xyz_scaled = scaler.transform(obs_xyz)
    pred_xyz_scaled = scaler.transform(pred_xyz)

    results.append({"rho": rho_value, 
                    "tau": WINDOW_SIZE, 
                    "observed": obs_xyz, 
                    "predicted": pred_xyz,
                    "observed_scaled": obs_xyz_scaled,
                    "predicted_scaled": pred_xyz_scaled
                    })
    
    print(f"Done Rho={rho_value}, Tau={WINDOW_SIZE}.")

df_results_upo_sliding = pd.DataFrame(results).sort_values(by=["rho", "tau"])

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_vs_forecast_horizon_GENERAL" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

pd.to_pickle(df_results_upo_sliding, FULL_PATH)

In [ ]:
# horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
# rho_vector = [28, 50, 100, 150, 175, 210]
# If you want to load the results later, use this code but set the horizons_experiments and rho_vector to what you used in the experiment

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "UPO_vs_forecast_horizon_GENERAL" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_results_upo_sliding = pd.read_pickle(FULL_PATH)

In [ ]:
PLOT_MAX_VALUES = 1500
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10
METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

df_results_upo_sliding["model"] = "UPO"

reset_matplotlib_defaults()


for METRIC_PLT in METRICS_TO_PLOT:
    plt.figure(figsize=(10, 5))
    for rho, df_rho in df_results_upo_sliding.groupby("rho"):
        rho_met = []
        horizons = []
        for index, row in df_rho.iterrows():
            model_name = row['model']
            horizon = row['tau']
            predicted = row['predicted_scaled']
            observed = row['observed_scaled']
            
            # DONE
            metrics_temp = []

            first_sim_observed = observed[0:SIMULATION_LEN-512 ]
            first_sim_predicted = predicted[0:SIMULATION_LEN-512 ]

            metrics_temp.append(compute_all_metrics(
                    first_sim_predicted, first_sim_observed,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))
            
            for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
                observed_temp = observed[i:i+SIMULATION_LEN]
                predicted_temp = predicted[i:i+SIMULATION_LEN]

                observed_temp = observed_temp[512 :]
                predicted_temp = predicted_temp[512 :]

                metrics_temp.append(compute_all_metrics(
                    predicted_temp, observed_temp,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))

            metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
            metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
            
            rho_met.append(metrics)
            horizons.append(horizon)


        plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
    plt.title(fr"UPO predictor {METRIC_PLT.upper()} per $\rho$ and $\tau$")
    plt.xlabel("Forecast Horizons")
    plt.ylabel(rf"{METRIC_PLT.upper()}")
    if METRIC_PLT == "dstsp":
        plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
    plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
    plt.legend()

    file_name_plot = (
    rf"{model_name}_rho_{rho_vector}_GENERAL_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
    + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
    )

    FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
    plt.tight_layout()
    plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# df_results_upo
# Columns are: 'rho', 'tau', 'observed', 'predicted'
# Lets plot some observed vs predicted for different rho and tau    
[28, 50, 100, 150, 175, 210]
[1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

RHO_PLOT = 175
TAU_PLOT = 10

MAX_POINTS_PLOT = 500
plot_3d_max_values = 500

BINS = 5
SMOOTHING = 50
MASE_STEPS = 50

subset = df_results_upo_sliding[(df_results_upo_sliding["rho"] == RHO_PLOT) & (df_results_upo_sliding["tau"] == TAU_PLOT)]
obs_xyz = subset["observed_scaled"].values[0]
pred_xyz = subset["predicted_scaled"].values[0]

observed  = obs_xyz
predicted = pred_xyz

metrics_temp = []

first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

metrics_temp.append(compute_all_metrics(
        first_sim_predicted, first_sim_observed,
        bins=BINS, smoothing=SMOOTHING,
        mase_steps=MASE_STEPS, scale=False
    ))

for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):

    observed_temp = observed[i:i+SIMULATION_LEN]
    predicted_temp = predicted[i:i+SIMULATION_LEN]

    observed_temp = observed_temp[models_parameters['seq_len'] :]
    predicted_temp = predicted_temp[models_parameters['seq_len'] :]

    metrics_temp.append(compute_all_metrics(
        predicted_temp, observed_temp,
        bins=BINS, smoothing=SMOOTHING,
        mase_steps=MASE_STEPS, scale=False
    ))

metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}     


mse_value = format_metric(metrics['mse'])
sbd_value = format_metric(metrics['sbd'])
dstsp_value = format_metric(metrics['dstsp'])
dh_value = format_metric(metrics['dh'])


title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={mse_value}    SBD={sbd_value}    $\mathbf{{D_{{stsp}}}}$={dstsp_value}    $\mathbf{{D_{{H}}}}$={dh_value}"

fig = plot_3D_attractor(obs_xyz[0:0,:], pred_xyz[0:plot_3d_max_values,:], ground_truth=obs_xyz[0:plot_3d_max_values,:], lim_pse=MAX_POINTS_PLOT, lim_gen=MAX_POINTS_PLOT, horizon = TAU_PLOT, title_3D="UPOs", title_big = title_plot_3D)
plt.show()

In [ ]:
df_results_upo_sliding

In [ ]:
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10
SCALED = True

all_metrics_general = []


for index, row in df_results_upo_sliding.iterrows():
    if SCALED:
        observed  = row['observed_scaled']
        predicted = row['predicted_scaled']
    else:
        observed  = row['observed']
        predicted = row['predicted']
        
    horizon    = row['tau']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[models_parameters['seq_len'] :]
        predicted_temp = predicted_temp[models_parameters['seq_len'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    
    all_metrics_general.append(metrics)


metric_keys = all_metrics_general[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_general]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_general]) for k in metric_keys}

print(f"\n UPO general averaged over {len(all_metrics_general)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.5f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.5f}")


In [ ]:
all_metrics_general

### **UPO Predictor Plots (UPOs, Observed, Predicted)**

In [ ]:
# RUN ALL OTHERS!

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

file_name = "_outer_region_performance_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_LSTM = pd.read_pickle(FULL_PATH)

file_name = "_Transformer_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_transformer = pd.read_pickle(FULL_PATH)

file_name = "_TFT_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_tft = pd.read_pickle(FULL_PATH)

file_name = "_pretrained_vs_forecast_horizon_" +  NAME + "_" + str("ALL_RHO")+ "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_pretrained = pd.read_pickle(FULL_PATH)

In [ ]:
def adapt_plotting_style():
    # Font sizes
    # Then apply your custom settings on top
    font_main = 26
    font_title = 26
    font_legend = 24 # !!1 For some reason, for these plots, legend is not visible, lets use some larger fonts (21 was prior to this)
    linewidth = 4

    plt.rcParams.update({
        # Font sizes
        'font.size': font_main,
        'axes.labelsize': font_main,
        'xtick.labelsize': font_main,
        'ytick.labelsize': font_main,
        'legend.fontsize': font_legend,
        'axes.titlesize': font_title,
        'figure.titlesize': font_main,
        # Axis & grid styling
        'axes.grid': True,
        'axes.grid.which': 'major',
        'axes.linewidth': 0.8, 

        # Line widths
        'lines.linewidth': linewidth
    })

In [ ]:
from matplotlib.legend_handler import HandlerTuple

# reset_matplotlib_defaults()

def _draw_single(ax, seg, traj, pred, ki, mode, predicted_tft=None, predicted_panda=None, plot_upo_predictions=True):
    if plot_upo_predictions:
        alpha_2d = 0.3
    else:
        alpha_2d = 0.5
    alpha_3d = 0.2

    upo_handles = []
    if mode == "2D":
        for i in range(ki):
            label = 'UPOs' if i == 0 else '_nolegend_'
            line, = ax.plot(seg[i, :, 0], seg[i, :, 2], label=label, alpha=alpha_2d, linewidth=3)
            upo_handles.append(line)

        if plot_upo_predictions:
            pred_line, = ax.plot(pred[:, 0], pred[:, 2], "--", label='Predicted', color='red', linewidth=4)
            traj_line, = ax.plot(traj[:, 0], traj[:, 2], "--", label='Observed', color='black', linewidth=4)
        else:
            if predicted_tft is not None:
                tft_line, = ax.plot(predicted_tft[:, 0], predicted_tft[:, 2], "--", label='TFT', color='blue', linewidth=3)
            if predicted_panda is not None:
                panda_line, = ax.plot(predicted_panda[:, 0], predicted_panda[:, 2], "--", label='Panda', color='red', linewidth=3)
            traj_line, = ax.plot(traj[:, 0], traj[:, 2], "--", label='Observed', color='black', linewidth=4)

        
        ax.set_xlabel('X')
        ax.set_ylabel('Z')
    else:
        for i in range(ki):
            label = 'UPOs' if i == 0 else '_nolegend_'
            line, = ax.plot(seg[i, :, 0], seg[i, :, 1], seg[i, :, 2], label=label, alpha=alpha_3d, linewidth=3)
            upo_handles.append(line)
        traj_line, = ax.plot(traj[:, 0], traj[:, 1], traj[:, 2], "--", label='Observed', color='black', linewidth=4)
        if plot_upo_predictions:
            pred_line, = ax.plot(pred[:, 0], pred[:, 1], pred[:, 2], "--", label='Predicted', color='red', linewidth=4)
        else:
            if predicted_tft is not None:
                tft_line, = ax.plot(predicted_tft[:, 0], predicted_tft[:, 1], predicted_tft[:, 2], "--", label='TFT', color='blue', linewidth=3)
            if predicted_panda is not None:
                panda_line, = ax.plot(predicted_panda[:, 0], predicted_panda[:, 1], predicted_panda[:, 2], "--", label='Panda', color='red', linewidth=3)
        ax.set_xlabel('X')
        ax.set_ylabel('Y')
        ax.set_zlabel('Z')
    
    labels_list = ['UPOs', 'Observed']
    handles_list = [tuple(upo_handles), traj_line]
    if plot_upo_predictions:
        labels_list.append('Predicted')
        handles_list.append(pred_line)
        ax.set_xlim(x_limits)
    else:
        if predicted_tft is not None:
            labels_list.append('TFT')
            handles_list.append(tft_line)
        if predicted_panda is not None:
            labels_list.append('Panda')
            handles_list.append(panda_line)
    ax.legend(
        handles=handles_list,
        labels=labels_list,
        handler_map={tuple: HandlerTuple(ndivide=None, pad=0.1)},
        handlelength=4
    )


def plot_upo_window(segment, traj_window, predicted, k, mode="3D", oneplot=False, ncols=3, predicted_tft=None, predicted_panda=None, plot_upo_predictions=True):
    if oneplot:
        n = len(segment)
        nrows = math.ceil(n / ncols)
        proj = '3d' if mode == "3D" else None
        fig, axes = plt.subplots(nrows, ncols,
                                 figsize=(6 * ncols, 6 * nrows),
                                 subplot_kw={'projection': proj} if proj else {})
        axes = np.array(axes).flatten()
        for idx in range(n):
            axes[idx].set_title(f'Window {idx + 1}')
            _draw_single(axes[idx], segment[idx], traj_window[idx], predicted[idx], k[idx], mode, predicted_tft=predicted_tft[idx], predicted_panda=predicted_panda[idx], plot_upo_predictions=plot_upo_predictions)
        for idx in range(n, len(axes)):
            axes[idx].set_visible(False)
        plt.tight_layout()
    else:
        if PLOT_FOR_METHODS:
            size = (5,5)
        else:
            size = (10, 10)
        if mode == "3D":
            fig = plt.figure(figsize=size)
            ax = fig.add_subplot(111, projection='3d')
        else:
            fig, ax = plt.subplots(figsize=size)
        # ax.set_title('UPOs vs Observed vs Predicted Trajectory')
        _draw_single(ax, segment, traj_window, predicted, k, mode, predicted_tft=predicted_tft, predicted_panda=predicted_panda, plot_upo_predictions=plot_upo_predictions)

    
    plt.show()
    return fig
adapt_plotting_style()
# reset_matplotlib_defaults()
# ── config ────────────────────────────────────────────────────────────────────
# window 10
#K1
#1
#1024

# K30
# 1072
# 1033
# 1024
#1235
#1340
#1418
#1451

#window 40
# 502, 505, 519, 500 + 5,6,51,57,177

#window 100
# 110, 112


# window 50
# 101, 122, 131

#window 50
#K 500
# 1

# CHANGE PARAMETERS !!!!! 

DT_UPOS = 0.001

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# DATA_DIR = rf"data_upos_original/DATA1_{RHO}"

upo_data_multi_rho = {}
for RHO in RHO_TO_LOAD:

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho[RHO] = df_orbits_individual
    
# LOAD TFT + Pandas Predictions and plot them with the UPO predictions for different windows, all in the same plot (or one plot per window if you want)
indexes_1 = [1,2,3,5,7,8,10,12,13, 14]
indexes_2 = [0, 4, 6, 9]

rho_vector          = [28]
horizons_experiments = [10]
K                   = 30
max_data_points     = 50000
# window_to_plot      = list(range(800, 1100))
window_to_plot      = [553, 596, 896, 915, 972, 980, 1020, 1073, 1287, 1321, 2100, 2605, 2650, 2875, 2505]
window_second_sweep = [0, 4, 6, 10]
# window_second_sweep = np.arange(0, len(window_to_plot)).tolist()  # if you want to plot all windows in the second sweep
# window_to_plot      = list(range(2500, 2600))
USE_ONEPLOT         = False   # True → one big grid figure | False → one figure per window
NCOLS               = 3
PLOT_MODE           = "2D"

# PLOT UPOS = PLOT FOR METHOD FIGURE UPOS AS PREDICTED 
PLOT_UPOS           = False
PLOT_FOR_METHODS      = False

df_tft = df_tft[df_tft["model"] == "TFT"]
df_tft_predictions = df_tft[(df_tft["train_rhos"] == rho_vector[0]) & (df_tft["forecast_horizons"] == horizons_experiments[0])]["predicted"].values[0]


df_panda = df_pretrained[df_pretrained["model"] == "Panda"]
df_panda_predictions = df_panda[(df_panda["rho"] == rho_vector[0]) & (df_panda["horizon"] == horizons_experiments[0])]["predicted"].values[0]

if PLOT_UPOS:
    x_limits = (-1.27, 0.7)
# ─────────────────────────────────────────────────────────────────────────────
plot_dir = "plots/upo_tft_panda/"
os.makedirs(plot_dir, exist_ok=True)
# ─────────────────────────────────────────────────────────────────────────────

scaler  = StandardScaler()
results = []

for rho_value in rho_vector:
    test_mask      = np.isin(test_o[:, 6], rho_value)
    test_data      = test_o[test_mask]
    df_upo         = upo_data_multi_rho[rho_value]
    obs_xyz        = test_data[:, 3:6]

    scaler.fit(np.vstack(obs_xyz))

    obs_xyz = obs_xyz[models_parameters['seq_len'] :] # remove first part to be fair with the models that use seq_len for prediction

    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    for tau in horizons_experiments:
        traj_xyz, upo_all_stacked, pred_xyz, upo_segments = UPO_prediction_window(
            obs_xyz,
            window_size=tau,
            tree=tree,
            labels=labels,
            norm=2,
            k=K,
            upo_df=df_upo_extended,
            time_diff=int(DT / DT_UPOS)
        )

        start, window = 0, 1
        segments_list, trajs_list, preds_list, ks_list = [], [], [], []
        preds_list_tft, preds_list_panda = [], []


        df_tft_inverse = scaler.inverse_transform(df_tft_predictions)
        df_panda_inverse = scaler.inverse_transform(df_panda_predictions)



        for segment in upo_segments:
            if window in window_to_plot:

                index_of_window = window_to_plot.index(window)
                tau_plot = tau if index_of_window in indexes_1 else tau - 1 

                if index_of_window in window_second_sweep:
                    
                    if USE_ONEPLOT:
                        segments_list.append(segment)
                        trajs_list.append(traj_xyz[start:start + tau])
                        preds_list.append(pred_xyz[start:start + tau])
                        preds_list_tft.append(df_tft_inverse[start:start + tau_plot])
                        preds_list_panda.append(df_panda_inverse[start:start + tau_plot])
                        ks_list.append(K)
                    else:
                        fig = plot_upo_window(segment=segment,
                                        traj_window=traj_xyz[start:start + tau],
                                        predicted=pred_xyz[start:start + tau],
                                        k=K,
                                        mode=PLOT_MODE,
                                        oneplot=False,
                                        predicted_tft=df_tft_inverse[start:start + tau_plot],
                                        predicted_panda=df_panda_inverse[start:start + tau_plot],
                                        plot_upo_predictions=PLOT_UPOS)
                        
                        file_name_plot = (
                                rf"UPO_TFT_Panda_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_K{K}_" + f"window {window}_" + PLOT_MODE + rf"_plot_upos_{PLOT_UPOS}_" + NAME + "_" +  rf"{EXPERIMENT_NAME}" + ".pdf"
                                )
                        
                        FULL_PATH_PLOT = os.path.join(plot_dir, file_name_plot)
                        fig.savefig(FULL_PATH_PLOT, bbox_inches='tight', format="pdf")
                        
            start  += tau
            window += 1

        if USE_ONEPLOT and segments_list:
            fig = plot_upo_window(segments_list, 
                            trajs_list, 
                            preds_list, 
                            ks_list,
                            mode=PLOT_MODE, 
                            oneplot=True, 
                            ncols=NCOLS,
                            predicted_tft=preds_list_tft,
                            predicted_panda=preds_list_panda,
                            plot_upo_predictions=PLOT_UPOS
                            )
            fig.savefig(FULL_PATH_PLOT, bbox_inches='tight')
            file_name_plot = (
                    rf"UPO_TFT_Panda_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_K{K}_" + f"window ALL_" + PLOT_MODE + rf"_plot_upos_{PLOT_UPOS}_" + NAME + "_" +  rf"{EXPERIMENT_NAME}" + ".pdf"
                    )
                    
            FULL_PATH_PLOT = os.path.join(plot_dir, file_name_plot)
            fig.savefig(FULL_PATH_PLOT, bbox_inches='tight', format="pdf")


reset_matplotlib_defaults()

## **Forecasting Comparissons Multi Rho**

In [ ]:
plots_dir_3d = "plots/3D_attractors/"
os.makedirs(plots_dir_3d, exist_ok=True)

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10

### **LSTM**

In [ ]:
# TF GEN
SCALE_PER_RHO = True
MAHALANOBIS = False

ATTENTION = False
TEACHER_FORCING = False

if TEACHER_FORCING:
    models_parameters['rolling'] = False
else:
    models_parameters['rolling'] = True
    
percentile = 90

horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
rho_vector = [28, 50, 100, 150, 175, 210]

models_parameters['seq_len'] = 512
models_parameters['batch_size'] = 32

parameter_names = []
parameter_values = []

jobs = []
results = []

for rho_value in rho_vector:

    training_rho = [rho_value]
    testing_rho = [rho_value]

    train_mask = np.isin(train_o[:, 6], training_rho)
    test_mask = np.isin(test_o[:, 6], testing_rho)

    train_data = train_o[train_mask]
    test_data = test_o[test_mask]

    all_data = np.vstack((train_data, test_data))

    for TFGEN in [False, True]:
        for tau in horizons_experiments:
            if tau == 1 and TFGEN:
                continue

            if tau == 1 or TFGEN:
                models_parameters['rolling'] = False
            else:
                models_parameters['rolling'] = True

            # Not used as model is loaded, but just to keep track of the parameters used in the experiment
            if TFGEN:
                models_parameters['dropout'] = 0.2
                models_parameters['recurrent_dropout'] = 0.2
                models_parameters['l2_reg'] = 1e-4
                models_parameters['clipnorm'] = 1.5
            else:
                models_parameters['dropout'] = 0.0
                models_parameters['recurrent_dropout'] = 0.0
                models_parameters['l2_reg'] = 0.0
                models_parameters['clipnorm'] = None

            models_parameters['forecast_size'] = tau

            lstm_model_name = "LSTM_TF" if TFGEN else "LSTM"
            FODLER_or_DIR = "SAVED_MODELS/LSTMS"            
            file_name = lstm_model_name + "_" + str(tau) + "_" + NAME + "_" + str(rho_value)+f"_scale_rho_{SCALE_PER_RHO}" + "_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pkl"
            FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
            model = load_models_from_file(FULL_PATH)

            jobs.append((datasets_parameters, 
                         rho_value, 
                         rho_value, 
                         train_data.copy(), 
                         test_data.copy(), 
                         all_data.copy(), 
                         [tau], 
                         SCALE_PER_RHO, 
                         models_parameters.copy(), 
                         parameter_names, 
                         parameter_values, 
                         TFGEN, 
                         model)) 


with ThreadPoolExecutor(max_workers=32) as ex:
    for r in ex.map(worker_param_hyper, jobs):
        results.append(r)


df_LSTM = pd.DataFrame(results)
df_LSTM = df_LSTM.sort_values(
    ["TEACHER_FORCING", "train_rhos", "test_rhos", "forecast_horizons", "SL", "BS", "Hidden Size 1", "Hidden Size 2"]
).reset_index(drop=True)


FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "_outer_region_performance_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)


df_LSTM.to_pickle(FULL_PATH)


In [ ]:
# LOAD IT!

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "_outer_region_performance_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_LSTM = pd.read_pickle(FULL_PATH)


In [ ]:
PLOT_MAX_VALUES = 500

METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

reset_matplotlib_defaults()
# 1. Plot each metric per rho and horizon, with separate lines for each rho, and separate plots for teacher forcing or not
for METRIC_PLT in METRICS_TO_PLOT:

    for df_tf_or_not, df_tf in df_LSTM.groupby("TEACHER_FORCING"):
        plt.figure(figsize=(10, 5))
        for rho, df_rho in df_tf.groupby("train_rhos"):
            rho_met = []
            horizons = []

            for index, row in df_rho.iterrows():
                model_name = "LSTM"
                horizon = row['forecast_horizons']
                predicted =  row.dfs['pred']
                observed = row.dfs['obs']

                # DONE
                metrics_temp = []

                first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
                first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

                metrics_temp.append(compute_all_metrics(
                        first_sim_predicted, first_sim_observed,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))
                
                for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
                
                    observed_temp = observed[i:i+SIMULATION_LEN]
                    predicted_temp = predicted[i:i+SIMULATION_LEN]

                    observed_temp = observed_temp[models_parameters['seq_len'] :]
                    predicted_temp = predicted_temp[models_parameters['seq_len'] :]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
           
                rho_met.append(metrics)
                horizons.append(horizon)

            if df_tf_or_not:
                model_name = "LSTM-B"
            else:           
                model_name = "LSTM-A"

            
            plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
        plt.title(rf"{model_name} {METRIC_PLT.upper()} per $\rho$ and $\tau$")
        plt.xlabel("Forecast Horizons")
        plt.ylabel(METRIC_PLT.upper())

        if METRIC_PLT == "dstsp":
            plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
        plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
        # plt.yscale('log')
        plt.legend()

        file_name_plot = (
            rf"{model_name}_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
            + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
        )
        FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
        plt.tight_layout()
        plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
        plt.show()


In [ ]:
# 2. Compute and print textual averages per (teacher_forcing, rho) across horizons

# Accumulate metrics per teacher_forcing only
by_tf_short = defaultdict(list)
by_tf_long  = defaultdict(list)

for index, row in df_LSTM.iterrows():
    teacher_f = row['TEACHER_FORCING']
    observed  = row.dfs['obs']
    predicted = row.dfs['pred']
    horizon    = row['forecast_horizons']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[models_parameters['seq_len'] :]
        predicted_temp = predicted_temp[models_parameters['seq_len'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    
    if FORECAST_TYPES_RANGES['short-term'][0] <= horizon <= FORECAST_TYPES_RANGES['short-term'][1]:
        by_tf_short[teacher_f].append(metrics)
    elif FORECAST_TYPES_RANGES['long-term'][0] <= horizon <= FORECAST_TYPES_RANGES['long-term'][1]:
        by_tf_long[teacher_f].append(metrics)

# Print averages per teacher_forcing (all rho + horizons collapsed)
for teacher_f, metrics_list in sorted(by_tf_short.items()):
    metric_keys = metrics_list[0].keys()
    averages = {k: np.mean([m[k] for m in metrics_list]) for k in metric_keys}
    stds = {k: np.std([m[k] for m in metrics_list]) for k in metric_keys}

    print(f"\nTeacher Forcing: {teacher_f} | short-term horizons averaged over {len(metrics_list)} (rho x horizon) combinations")
    for metric, value in averages.items():
        print(f"  {metric:>15}: {value:.2f}")
    for metric, value in stds.items():
        print(f"  {metric:>15} std: {value:.2f}")

for teacher_f, metrics_list in sorted(by_tf_long.items()):
    metric_keys = metrics_list[0].keys()
    averages = {k: np.mean([m[k] for m in metrics_list]) for k in metric_keys}
    stds = {k: np.std([m[k] for m in metrics_list]) for k in metric_keys}

    print(f"\nTeacher Forcing: {teacher_f} | long-term horizons averaged over {len(metrics_list)} (rho x horizon) combinations")
    for metric, value in averages.items():
        print(f"  {metric:>15}: {value:.2f}")
    for metric, value in stds.items():
        print(f"  {metric:>15} std: {value:.2f}")



In [ ]:
# 3. For a specific (teacher_forcing, rho, horizon) combination, plot the 3D attractor with metrics in the title with Time Series and Frequency plots as well

PLOT_MAX_VALUES = 500
plot_3d_max_values = 500


RHO_TO_PLOT = 28
HORIZON_TO_PLOT = 100

for index, row in df_LSTM.iterrows():
    teacher_f = row['TEACHER_FORCING']
    horizon = row['forecast_horizons']
    observed = row.dfs['obs']
    predicted = row.dfs['pred']
    rho = row['train_rhos']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[models_parameters['seq_len'] :]
        predicted_temp = predicted_temp[models_parameters['seq_len'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}     

    # metrics_results = compute_all_metrics(predicted, observed, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS, scale=False)
    # metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics_results.items()}     

    if horizon == HORIZON_TO_PLOT and rho == RHO_TO_PLOT:
        model_plot = rf"LSTM-A" if not teacher_f else rf"LSTM-B"

        mse_value = format_metric(metrics['mse'])
        sbd_value = format_metric(metrics['sbd'])
        dstsp_value = format_metric(metrics['dstsp'])
        dh_value = format_metric(metrics['dh'])

        title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={mse_value}    SBD={sbd_value}    $\mathbf{{D_{{stsp}}}}$={dstsp_value}    $\mathbf{{D_{{H}}}}$={dh_value}"

        # title_plot_3D = rf"$\tau$={horizon}    MSE={metrics['mse']:.3f}    SBD={metrics['sbd']:.3f}    $D_{{stsp}}$={metrics['dstsp']:.3f}    $D_{{H}}$={metrics['dh']:.3f}"
        fig = plot_3D_attractor(observed[0:0,:], predicted[0:PLOT_MAX_VALUES], ground_truth=observed[0:PLOT_MAX_VALUES,:], lim_pse=plot_3d_max_values, lim_gen=plot_3d_max_values, horizon = horizon, title_3D=model_plot, title_big = title_plot_3D)
        
        plt.savefig(os.path.join(plots_dir_3d, f"3D_and_TS_{rho}_horizon_{horizon}_{model_plot}.pdf"), format="pdf", bbox_inches="tight")

        plt.show()

### **Transformers and TFT**

#### **Transformers**

In [ ]:
transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

In [ ]:
# Multi Experiment Transformer for ____ dataset

def run_single_transformer_experiment(job):
    train_rho, test_rho, train_data, test_data, transformer_parameters, tau, rho_value, model = job
    observed, predicted = run_transformer_pipeline(train_data, 
                                                   test_data, 
                                                   transformer_parameters, 
                                                   forecast_horizons=tau, 
                                                   rho_value=rho_value, 
                                                   model=model)

    return {
            "model": "Transformer",
            "train_rhos": train_rho,
            "test_rhos": test_rho,
            "forecast_horizons": tau,
            "observed": observed,
            "predicted": predicted
        }

transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

jobs = []
results = []

# Not using parallelization here, as the transformer training is already quite heavy, and I want to avoid any potential issues with GPU memory or multiprocessing overhead. If needed, this can be parallelized in the future.
# on CPU it is extra slow while on GPU it is manageable, but still takes some time, so I prefer to run it sequentially for now.
for rho_value in rho_vector:
    
    train_rho = rho_value
    test_rho = rho_value

    train_mask = np.isin(train_o[:, 6], train_rho)
    test_mask  = np.isin(test_o[:, 6], test_rho)

    train_data = train_o[train_mask]
    test_data  = test_o[test_mask]

    for tau in horizons_experiments:

        FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
        file_name = "Transformer" + "_" + str(tau) + "_" + NAME + "_" + str(rho_value)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"

        FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
        model = TransformerModel.load(FULL_PATH)

        jobs.append((train_rho, 
                     test_rho, 
                     train_data.copy(), 
                     test_data.copy(), 
                     transformer_parameters.copy(), 
                     tau, 
                     rho_value, 
                     model))


with ThreadPoolExecutor(max_workers=1) as executor:
    for r in executor.map(run_single_transformer_experiment, jobs):
        results.append(r)    
        
df_transformer = pd.DataFrame(results)

df_transformer = df_transformer.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]
).reset_index(drop=True)


FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "_Transformer_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)


df_transformer.to_pickle(FULL_PATH)


In [ ]:
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)    

file_name = "_Transformer_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_transformer = pd.read_pickle(FULL_PATH)

In [ ]:
PLOT_MAX_VALUES = 1500

METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']


PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)


reset_matplotlib_defaults()

# 1. Plot each metric per rho and horizon, with separate lines for each rho
for METRIC_PLT in METRICS_TO_PLOT:
    plt.figure(figsize=(10, 5))
    for rho, df_rho in df_transformer.groupby("train_rhos"):
        rho_met = []
        horizons = []
        for index, row in df_rho.iterrows():
            model_name = row['model']
            horizon = row['forecast_horizons']
            predicted = row['predicted']
            observed = row['observed']
            
            # DONE
            metrics_temp = []

            first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
            first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

            metrics_temp.append(compute_all_metrics(
                    first_sim_predicted, first_sim_observed,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))
            
            for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
                observed_temp = observed[i:i+SIMULATION_LEN]
                predicted_temp = predicted[i:i+SIMULATION_LEN]

                observed_temp = observed_temp[transformer_parameters['input_length'] :]
                predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

                metrics_temp.append(compute_all_metrics(
                    predicted_temp, observed_temp,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))

            metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
            metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
        
            # metrics_results = compute_all_metrics(predicted, observed, 
                                                #   bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS, scale=False)
            rho_met.append(metrics)
            horizons.append(horizon)



        plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
    plt.title(fr"Transformer {METRIC_PLT.upper()} per $\rho$ and $\tau$")
    plt.xlabel("Forecast Horizons")
    plt.ylabel(rf"{METRIC_PLT.upper()}")

    if METRIC_PLT == "dstsp":
        plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
    plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
    plt.legend()

    file_name_plot = (
    rf"{model_name}_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
    + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
    )

    FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
    plt.tight_layout()
    plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# 2. Compute and print textual averages per rho across horizons print textual only
all_metrics_short = []
all_metrics_long = []

for index, row in df_transformer.iterrows():
    predicted = row['predicted']
    observed = row['observed']
    horizon = row['forecast_horizons']
    
    # TODO: split into trajectories and average results!
    # Currently, we are treating the entire test set as one long trajectory,
    #  which can lead to outliers affecting the metrics.
    # This happens at the end and start of each trajectory

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[transformer_parameters['input_length'] :]
        predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    if any(np.isnan(m) for m in metrics.values()):
        continue

    if FORECAST_TYPES_RANGES['short-term'][0] <= horizon <= FORECAST_TYPES_RANGES['short-term'][1]:
        all_metrics_short.append(metrics)
    elif FORECAST_TYPES_RANGES['long-term'][0] <= horizon <= FORECAST_TYPES_RANGES['long-term'][1]:
        all_metrics_long.append(metrics)

metric_keys = all_metrics_short[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_short]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_short]) for k in metric_keys}

print(f"\nTransformer short-term averaged over {len(all_metrics_short)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")

metric_keys = all_metrics_long[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_long]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_long]) for k in metric_keys}

print(f"\nTransformer long-term averaged over {len(all_metrics_long)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")
    


In [ ]:
# Plot 3D attractor for a specific (rho, horizon) combination, with metrics in the title, along with Time Series and Frequency plots as well
PLOT_MAX_VALUES = 500
plot_3d_max_values = 500

RHO_TO_PLOT = 28
HORIZON_TO_PLOT = 10


for index, row in df_transformer.iterrows():
    model_name = row['model']
    horizon = row['forecast_horizons']
    predicted = row['predicted']
    observed = row['observed']
    rho = row['train_rhos']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[transformer_parameters['input_length'] :]
        predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

    # metrics_results = compute_all_metrics(predicted, observed, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS, scale=False)

    if rho == RHO_TO_PLOT and horizon == HORIZON_TO_PLOT    :
        # title_plot_3D = rf"$\tau$={horizon}    MSE={metrics_results['mse']:.3f}    SBD={metrics_results['sbd']:.3f}    $D_{{stsp}}$={metrics_results['dstsp']:.3f}    $D_{{H}}$={metrics_results['dh']:.3f}"
        
        mse_value = format_metric(metrics['mse'])
        sbd_value = format_metric(metrics['sbd'])
        dstsp_value = format_metric(metrics['dstsp'])
        dh_value = format_metric(metrics['dh'])

        title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={mse_value}    SBD={sbd_value}    $\mathbf{{D_{{stsp}}}}$={dstsp_value}    $\mathbf{{D_{{H}}}}$={dh_value}"

        
        fig = plot_3D_attractor(observed[0:0,:], predicted[0:plot_3d_max_values], ground_truth=observed[0:plot_3d_max_values,:], lim_pse=PLOT_MAX_VALUES, lim_gen=PLOT_MAX_VALUES, horizon = horizon, title_3D=model_name, title_big = title_plot_3D)
        
        plt.savefig(os.path.join(plots_dir_3d, f"3D_and_TS_{rho}_horizon_{horizon}_{model_name}.pdf"), format="pdf", bbox_inches="tight")
        
        plt.show()
        reset_matplotlib_defaults()


#### **TFT**

In [ ]:
tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

In [ ]:
# Multi Experiment TFT for ____ dataset

def run_single_tft_experiment(job):
    train_rho, test_rho, train_data, test_data, tft_parameters, tau, rho_value, model = job
    observed, predicted = run_tft_pipeline(train_data, test_data, tft_parameters, forecast_horizons=tau, rho_value=rho_value, model=model)

    return {
            "model": "TFT",
            "train_rhos": train_rho,
            "test_rhos": test_rho,
            "forecast_horizons": tau,
            "observed": observed,
            "predicted": predicted
        }

tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

jobs = []
results = []

# Not using parallelization here, as the transformer training is already quite heavy, and I want to avoid any potential issues with GPU memory or multiprocessing overhead. If needed, this can be parallelized in the future.
# on CPU it is extra slow while on GPU it is manageable, but still takes some time, so I prefer to run it sequentially for now.
for rho_value in rho_vector:

    print(f"Running TFT experiments for rho = {rho_value}...")
    train_rho = rho_value
    test_rho = rho_value

    train_mask = np.isin(train_o[:, 6], train_rho)
    test_mask  = np.isin(test_o[:, 6], test_rho)

    train_data = train_o[train_mask]
    test_data  = test_o[test_mask]

    for tau in horizons_experiments:

        FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
        file_name = "TFT" + "_" + str(tau) + "_" + NAME + "_" + str(rho_value)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"

        FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
        model = TFTModel.load(FULL_PATH)

        jobs.append((train_rho, test_rho, train_data.copy(), test_data.copy(), tft_parameters.copy(), tau, rho_value, model))


with ThreadPoolExecutor(max_workers=1) as executor:
    for r in executor.map(run_single_tft_experiment, jobs):
        results.append(r)    
        
df_tft = pd.DataFrame(results)

df_tft = df_tft.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]
).reset_index(drop=True)

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "_TFT_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_tft.to_pickle(FULL_PATH)


In [ ]:
FOLDER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FOLDER_or_DIR, exist_ok=True)

file_name = "_TFT_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FOLDER_or_DIR, file_name)

df_tft = pd.read_pickle(FULL_PATH)

In [ ]:
PLOT_MAX_VALUES = 1500

METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

reset_matplotlib_defaults()

# 1. Plot each metric per rho and horizon, with separate lines for each rho
for METRIC_PLT in METRICS_TO_PLOT:
    plt.figure(figsize=(10, 5))
    for rho, df_rho in df_tft.groupby("train_rhos"):
        rho_met = []
        horizons = []
        for index, row in df_rho.iterrows():
            model_name = row['model']
            horizon = row['forecast_horizons']
            predicted = row['predicted']
            observed = row['observed']
            
            # DONE
            metrics_temp = []

            first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
            first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

            metrics_temp.append(compute_all_metrics(
                    first_sim_predicted, first_sim_observed,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))
            
            for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
                observed_temp = observed[i:i+SIMULATION_LEN]
                predicted_temp = predicted[i:i+SIMULATION_LEN]

                observed_temp = observed_temp[tft_parameters['input_length'] :]
                predicted_temp = predicted_temp[tft_parameters['input_length'] :]

                metrics_temp.append(compute_all_metrics(
                    predicted_temp, observed_temp,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))

            metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
            metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
            
            rho_met.append(metrics)
            horizons.append(horizon)


        plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
    plt.title(fr"TFT {METRIC_PLT.upper()} per $\rho$ and $\tau$")
    plt.xlabel("Forecast Horizons")
    plt.ylabel(rf"{METRIC_PLT.upper()}")
    if METRIC_PLT == "dstsp":
        plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
    plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
    plt.legend()

    file_name_plot = (
    rf"{model_name}_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
    + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
    )

    FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
    plt.tight_layout()
    plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
    plt.show()


In [ ]:
# 2. Compute and print textual averages per rho across horizons print textual only

all_metrics_short = []
all_metrics_long = []

for index, row in df_tft.iterrows():
    predicted = row['predicted']
    observed = row['observed']
    horizon = row['forecast_horizons']
    

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[tft_parameters['input_length'] :]
        predicted_temp = predicted_temp[tft_parameters['input_length'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    
    if any(np.isnan(m) for m in metrics.values()):
        continue

    if FORECAST_TYPES_RANGES['short-term'][0] <= horizon <= FORECAST_TYPES_RANGES['short-term'][1]:
        all_metrics_short.append(metrics)
    elif FORECAST_TYPES_RANGES['long-term'][0] <= horizon <= FORECAST_TYPES_RANGES['long-term'][1]:
        all_metrics_long.append(metrics)

metric_keys = all_metrics_short[0].keys()
averages_short = {k: np.mean([m[k] for m in all_metrics_short]) for k in metric_keys}
std_short = {k: np.std([m[k] for m in all_metrics_short]) for k in metric_keys}

metric_keys = all_metrics_long[0].keys()
averages_long = {k: np.mean([m[k] for m in all_metrics_long]) for k in metric_keys}
std_long = {k: np.std([m[k] for m in all_metrics_long]) for k in metric_keys}

print(f"\nTFT short-term averaged over {len(all_metrics_short)} (rho x horizon) combinations")
for metric, value in averages_short.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in std_short.items():
    print(f"  {metric:>15} std: {value:.2f}")


print(f"\nTFT long-term averaged over {len(all_metrics_long)} (rho x horizon) combinations")
for metric, value in averages_long.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in std_long.items():
    print(f"  {metric:>15} std: {value:.2f}")


In [ ]:
# Plot 3D attractor for a specific (rho, horizon) combination, with metrics in the title, along with Time Series and Frequency plots as well

PLOT_MAX_VALUES = 500
plot_3d_max_values = 500

RHO_TO_PLOT = 28
HORIZON_TO_PLOT = 100

for index, row in df_tft.iterrows():
    model_name = row['model']
    horizon = row['forecast_horizons']
    predicted = row['predicted']
    observed = row['observed']
    rho_v = row['train_rhos']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[tft_parameters['input_length'] :]
        predicted_temp = predicted_temp[tft_parameters['input_length'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    

    if rho_v == RHO_TO_PLOT and horizon == HORIZON_TO_PLOT:

        mse_value = format_metric(metrics['mse'])
        sbd_value = format_metric(metrics['sbd'])
        dstsp_value = format_metric(metrics['dstsp'])
        dh_value = format_metric(metrics['dh'])

        title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={mse_value}    SBD={sbd_value}    $\mathbf{{D_{{stsp}}}}$={dstsp_value}    $\mathbf{{D_{{H}}}}$={dh_value}"

        fig = plot_3D_attractor(observed[0:0,:], predicted[0:plot_3d_max_values], ground_truth=observed[0:plot_3d_max_values,:], lim_pse=PLOT_MAX_VALUES, lim_gen=PLOT_MAX_VALUES, horizon = horizon, title_3D=model_name, title_big = title_plot_3D)

        plt.savefig(os.path.join(plots_dir_3d, f"3D_and_TS_{rho_v}_horizon_{horizon}_{model_name}.pdf"), format="pdf", bbox_inches="tight")


        plt.show()
        reset_matplotlib_defaults()


### **UPO-based Predictor**

##### **This is the UPOs loader (READ DESCRIPTION)**

In [ ]:
""" 
This is the same as in:
    UPO Forecasting Evaluation -> Extract UPOs -> LOAD UPOS FROM PKL!
No need to run again if you have already run it.

If upo pkl files are not available, run the exytacy and recreate UPOs cell.
    Moreover, UPO files are needed!
"""

# [0.001,0.003, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1]
DT_UPOS = 0.001

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# DATA_DIR = rf"data_upos_original/DATA1_{RHO}"

upo_data_multi_rho = {}
for RHO in RHO_TO_LOAD:

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho[RHO] = df_orbits_individual

##### **UPO Prediction over Multiple Rhos and Horizons**

In [ ]:
reset_matplotlib_defaults()

rho_vector = [28, 50, 100, 150, 175, 210]
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

K = 1

scaler = StandardScaler()
results = []

UPO_USAGE_RHO_HORIZON = []

for rho_value in rho_vector:
    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]
    df_upo = upo_data_multi_rho[rho_value]

    obs_xyz = test_data[:, 3:6]
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    for tau in horizons_experiments:
        print(f"Processing Rho={rho_value}, Tau={tau}...")

        # UPO_prediction_window
        # UPO_Predict_Counter, last return is upo_usage. 
        traj_xyz, upo_all_stacked, pred_xyz, upo_segments = UPO_prediction_window(
            obs_xyz,
            window_size=tau,
            tree=tree,
            labels=labels,
            norm=2,
            k=K,
            upo_df=df_upo_extended,
            time_diff=int(DT / DT_UPOS)
        )

        start = 0

        scaler.fit(np.vstack([obs_xyz, pred_xyz]))
        obs_xyz_scaled = scaler.transform(obs_xyz)
        pred_xyz_scaled = scaler.transform(pred_xyz)

        results.append({"rho": rho_value, 
                        "tau": tau, 
                        "observed": obs_xyz, 
                        "predicted": pred_xyz,
                        "observed_scaled": obs_xyz_scaled,
                        "predicted_scaled": pred_xyz_scaled
                       })
        print(f"Done Rho={rho_value}, Tau={tau}.")

df_results_upo = pd.DataFrame(results).sort_values(by=["rho", "tau"])

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)


file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

pd.to_pickle(df_results_upo, FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)


file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

pd.to_pickle(df_results_upo, FULL_PATH)

In [ ]:
# horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
# rho_vector = [28, 50, 100, 150, 175, 210]
# If you want to load the results later, use this code but set the horizons_experiments and rho_vector to what you used in the experiment
K = 1
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_UPO_results = pd.read_pickle(FULL_PATH)

In [ ]:
PLOT_MAX_VALUES = 1500

METRICS_TO_PLOT = ['mse', 'sbd', 'dstsp']

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

df_UPO_results["model"] = "UPO"

reset_matplotlib_defaults()


for METRIC_PLT in METRICS_TO_PLOT:
    plt.figure(figsize=(10, 5))
    for rho, df_rho in df_UPO_results.groupby("rho"):
        rho_met = []
        horizons = []
        for index, row in df_rho.iterrows():
            model_name = row['model']
            horizon = row['tau']
            predicted = row['predicted_scaled']
            observed = row['observed_scaled']
            
            # DONE
            metrics_temp = []

            first_sim_observed = observed[0:SIMULATION_LEN-512 ]
            first_sim_predicted = predicted[0:SIMULATION_LEN-512 ]

            metrics_temp.append(compute_all_metrics(
                    first_sim_predicted, first_sim_observed,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))
            
            for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
                observed_temp = observed[i:i+SIMULATION_LEN]
                predicted_temp = predicted[i:i+SIMULATION_LEN]

                observed_temp = observed_temp[512 :]
                predicted_temp = predicted_temp[512 :]

                metrics_temp.append(compute_all_metrics(
                    predicted_temp, observed_temp,
                    bins=BINS, smoothing=SMOOTHING,
                    mase_steps=MASE_STEPS, scale=False
                ))

            metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
            metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
            
            rho_met.append(metrics)
            horizons.append(horizon)


        plt.plot(horizons, [met[METRIC_PLT] for met in rho_met], label=rf"$\rho={rho}$")
    plt.title(fr"UPO predictor {METRIC_PLT.upper()} per $\rho$ and $\tau$")
    plt.xlabel("Forecast Horizons")
    plt.ylabel(rf"{METRIC_PLT.upper()}")
    if METRIC_PLT == "dstsp":
        plt.yscale('log') if PLOT_LOG_SCALE_DEFAULT_PREDICTIONS else None 
    plt.ylim(METRIC_RANGES_PLOT[METRIC_PLT])
    plt.legend()

    file_name_plot = (
    rf"{model_name}_rho_{rho_vector}_horizon_{horizons_experiments[-1]}_{METRIC_PLT}_"
    + NAME + "_" + rf"{EXPERIMENT_NAME}" + ".pdf"
    )

    FULL_PATH_PLOT = os.path.join(PLOTS_TAU_DIR, file_name_plot)
    plt.tight_layout()
    plt.savefig(FULL_PATH_PLOT, format="pdf", bbox_inches="tight")
    plt.show()

In [ ]:
# df_results_upo
# Columns are: 'rho', 'tau', 'observed', 'predicted'
# Lets plot some observed vs predicted for different rho and tau    
[28, 50, 100, 150, 175, 210]
[1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

MAX_POINTS_PLOT = 500
plot_3d_max_values = 500
MODEL_NAME = "UPO Predictor"

RHO_PLOT = 28
TAU_PLOT = 100


horizon = TAU_PLOT

subset = df_UPO_results[(df_UPO_results["rho"] == RHO_PLOT) & (df_UPO_results["tau"] == TAU_PLOT)]
observed = subset["observed_scaled"].values[0]
predicted = subset["predicted_scaled"].values[0]

# DONE
metrics_temp = []

first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

metrics_temp.append(compute_all_metrics(
        first_sim_predicted, first_sim_observed,
        bins=BINS, smoothing=SMOOTHING,
        mase_steps=MASE_STEPS, scale=False
    ))

for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):

    observed_temp = observed[i:i+SIMULATION_LEN]
    predicted_temp = predicted[i:i+SIMULATION_LEN]

    observed_temp = observed_temp[tft_parameters['input_length'] :]
    predicted_temp = predicted_temp[tft_parameters['input_length'] :]

    metrics_temp.append(compute_all_metrics(
        predicted_temp, observed_temp,
        bins=BINS, smoothing=SMOOTHING,
        mase_steps=MASE_STEPS, scale=False
    ))

metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

mse_value = format_metric(metrics['mse'])
sbd_value = format_metric(metrics['sbd'])
dstsp_value = format_metric(metrics['dstsp'])
dh_value = format_metric(metrics['dh'])

title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={mse_value}    SBD={sbd_value}    $\mathbf{{D_{{stsp}}}}$={dstsp_value}    $\mathbf{{D_{{H}}}}$={dh_value}"

fig = plot_3D_attractor(observed[0:0,:], predicted[0:plot_3d_max_values], ground_truth=observed[0:plot_3d_max_values,:], lim_pse=MAX_POINTS_PLOT, lim_gen=MAX_POINTS_PLOT, horizon = TAU_PLOT, title_3D=MODEL_NAME, title_big = title_plot_3D)

plt.savefig(os.path.join(plots_dir_3d, f"3D_and_TS_{RHO_PLOT}_horizon_{horizon}_{MODEL_NAME}.pdf"), format="pdf", bbox_inches="tight")

plt.show()
reset_matplotlib_defaults()

# fig = plot_3D_attractor(obs_xyz[0:0,:], pred_xyz[0:plot_3d_max_values,:], ground_truth=obs_xyz[0:plot_3d_max_values,:], lim_pse=MAX_POINTS_PLOT, lim_gen=MAX_POINTS_PLOT, horizon = TAU_PLOT, title_3D="UPOs", title_big = f"Rho={RHO_PLOT}, Tau={TAU_PLOT}")
# plt.show()

In [ ]:
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10
SCALED = True

all_metrics_short = []
all_metrics_long = []

by_tf_short = defaultdict(list)
by_tf_long  = defaultdict(list)


for index, row in df_UPO_results.iterrows():
    if SCALED:
        observed  = row['observed_scaled']
        predicted = row['predicted_scaled']
    else:
        observed  = row['observed']
        predicted = row['predicted']
        
    horizon    = row['tau']

    # DONE
    metrics_temp = []

    first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
    first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

    metrics_temp.append(compute_all_metrics(
            first_sim_predicted, first_sim_observed,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))
    
    for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
    
        observed_temp = observed[i:i+SIMULATION_LEN]
        predicted_temp = predicted[i:i+SIMULATION_LEN]

        observed_temp = observed_temp[models_parameters['seq_len'] :]
        predicted_temp = predicted_temp[models_parameters['seq_len'] :]

        metrics_temp.append(compute_all_metrics(
            predicted_temp, observed_temp,
            bins=BINS, smoothing=SMOOTHING,
            mase_steps=MASE_STEPS, scale=False
        ))

    metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
    metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
    
    if FORECAST_TYPES_RANGES['short-term'][0] <= horizon <= FORECAST_TYPES_RANGES['short-term'][1]:
        all_metrics_short.append(metrics)
    elif FORECAST_TYPES_RANGES['long-term'][0] <= horizon <= FORECAST_TYPES_RANGES['long-term'][1]:
        all_metrics_long.append(metrics)

metric_keys = all_metrics_short[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_short]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_short]) for k in metric_keys}

print(f"\n UPO short-term averaged over {len(all_metrics_short)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")

metric_keys = all_metrics_long[0].keys()
averages = {k: np.mean([m[k] for m in all_metrics_long]) for k in metric_keys}
stds = {k: np.std([m[k] for m in all_metrics_long]) for k in metric_keys}

print(f"\n UPO long-term averaged over {len(all_metrics_long)} (rho x horizon) combinations")
for metric, value in averages.items():
    print(f"  {metric:>15}: {value:.2f}")
for metric, value in stds.items():
    print(f"  {metric:>15} std: {value:.2f}")


#### **UPO UTILIZATION FREQUENCY**

In [ ]:
rho_vector = [28, 50, 100, 150, 175, 210]

# from 10 to 500
horizons_experiments = range(5, 505, 5)

K = 1

PLOTS_TAU_DIR = "plots/tau_results"
os.makedirs(PLOTS_TAU_DIR, exist_ok=True)

MODEL_NAME = "UPO Predictor"

In [ ]:
reset_matplotlib_defaults()

scaler = StandardScaler()
results = []

UPO_USAGE_RHO_HORIZON = []

for rho_value in rho_vector:
    test_mask = np.isin(test_o[:, 6], rho_value)
    test_data = test_o[test_mask]
    df_upo = upo_data_multi_rho[rho_value]

    obs_xyz = test_data[:, 3:6]
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    for tau in horizons_experiments:
        print(f"Processing Rho={rho_value}, Tau={tau}...")

        # UPO_prediction_window
        # UPO_Predict_Counter, last return is upo_usage. 
        traj_xyz, upo_all_stacked, pred_xyz, upo_segments, upo_usage, upo_frequency, total_forecasts = UPO_Predict_Counter(
            obs_xyz,
            window_size=tau,
            tree=tree,
            labels=labels,
            norm=2,
            k=K,
            upo_df=df_upo_extended,
            time_diff=int(DT / DT_UPOS)
        )

        start = 0

        scaler.fit(np.vstack([obs_xyz, pred_xyz]))
        obs_xyz_scaled = scaler.transform(obs_xyz)
        pred_xyz_scaled = scaler.transform(pred_xyz)

        results.append({"rho": rho_value, 
                        "tau": tau, 
                        "observed": obs_xyz, 
                        "predicted": pred_xyz,
                        "observed_scaled": obs_xyz_scaled,
                        "predicted_scaled": pred_xyz_scaled
                       })
        UPO_USAGE_RHO_HORIZON.append({"rho": rho_value, "tau": tau, "upo_usage": upo_usage, "upo_frequency": upo_frequency, "total_forecasts": total_forecasts})
        print(f"Done Rho={rho_value}, Tau={tau}.")

df_results_upo = pd.DataFrame(results).sort_values(by=["rho", "tau"])

UPO_USAGE_RHO_HORIZON_df = pd.DataFrame(UPO_USAGE_RHO_HORIZON)

FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_usage_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
pd.to_pickle(UPO_USAGE_RHO_HORIZON_df, FULL_PATH)


In [ ]:
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "UPO_usage_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + f"K{K}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

UPO_USAGE_RHO_HORIZON_df  = pd.read_pickle(FULL_PATH)


In [ ]:
UPO_USAGE_RHO_HORIZON_df

In [ ]:
### Lets plot UPO usage per rho and horizon
# Data is in UPO_USAGE_RHO_HORIZON_df with columns: 'rho', 'tau', 'upo_usage', 'upo_frequency'
# Each UPO usage is in number of selections of that UPO in the prediction window across the whole test set
# DATAFRAME: UPO_USAGE_RHO_HORIZON_df with columns: 'rho', 'tau', 'upo_usage', 'upo_frequency'
# upo_usage and frequency are lists with the same length, each element corresponds to a UPO label, and contains the usage or frequency of that UPO

# Line plots do not work... lets have some bar charts in 3D
# FOr a single RHO! 
    # X axis: UPO labels
    # Y axis: tau 
    # Z axis: usage or frequency

from matplotlib.ticker import ScalarFormatter
reset_matplotlib_defaults()

# Then apply your custom settings on top
font_main = 20
font_title = 20
font_legend = 20
linewidth = 4

plt.rcParams.update({
    'font.size': font_main,
    'axes.labelsize': font_main,
    'xtick.labelsize': font_main,
    'ytick.labelsize': font_main,
    'legend.fontsize': font_legend,
    'axes.titlesize': font_title,
    'figure.titlesize': font_main,
    'axes.grid': True,
    'axes.grid.which': 'major',
    'axes.linewidth': 0.8,
    'lines.linewidth': linewidth
})

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.rcParamsDefault['axes.prop_cycle'].by_key()['color'])

# --- Selection variables ---
rho_plot = 150
tau_plot = 5

row = UPO_USAGE_RHO_HORIZON_df[
    (UPO_USAGE_RHO_HORIZON_df['rho'] == rho_plot) &
    (UPO_USAGE_RHO_HORIZON_df['tau'] == tau_plot)
].iloc[0]

total_forecasts = row['total_forecasts']
usages          = np.array(row['upo_usage'])
upo_labels      = np.arange(1, len(usages) + 1)

used_mask   = usages > 0
pct_used    = 100.0 * usages[used_mask] / total_forecasts
labels_used = upo_labels[used_mask]

n_used   = len(labels_used)
n_unused = len(usages) - n_used - 1  # -1 for the zero-usage UPOs

fig, ax = plt.subplots(figsize=(10, 5))
sc = ax.scatter(labels_used, pct_used, s=25, c=pct_used, cmap='turbo', alpha=0.8)
# fig.colorbar(sc, ax=ax, label="Selection [%]")
ax.set_xlabel("UPO")
ax.set_ylabel("Selection [%]")
ax.set_title(
    rf"$\rho={rho_plot}$ - "
    rf"used: {n_used}, unused: {n_unused}"
)
min_pct = 100.0 / total_forecasts
ax.set_yscale('log')
ax.yaxis.set_major_formatter(ScalarFormatter())
ax.yaxis.get_major_formatter().set_scientific(False)
max_value = max(pct_used) if len(pct_used) > 0 else min_pct
ax.set_ylim([1e-2, max_value * 1.5])
ax.grid(True, which="both", ls="--", linewidth=0.5, alpha=1)
plt.tight_layout()
plt.savefig(os.path.join(PLOTS_TAU_DIR, f"UPO_selection_rate_rho{rho_plot}_tau{tau_plot}_{MODEL_NAME}.pdf"), format="pdf", bbox_inches="tight")
plt.show()

reset_matplotlib_defaults()


## **Delta Rho Sensitivity Analysis**

### **LSTM**

In [ ]:
# TF GEN
SCALE_PER_RHO = True
TEACHER_FORCING = False

if TEACHER_FORCING:
    models_parameters['rolling'] = False
else:
    models_parameters['rolling'] = True
    

horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]

DELTA_RHO = 5

delta_combinatons = create_delta_combinations(rho_centers, DELTA_RHO)

models_parameters['seq_len'] = 512
models_parameters['batch_size'] = 32

jobs_delta = []
results_delta = []


# Start from the center and extend outwards until the maximum delta is reached, with steps of 1
for rho_value in rho_centers:

    # Choose Desired Rho
    training_rho = [rho_value]

    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]

    delta_combinatons_test = delta_combinatons[rho_value]

    for testing_rho in delta_combinatons_test:
        
        test_mask = np.isin(test_o[:, 6], testing_rho)
        test_data = test_o[test_mask]

        all_data = np.vstack((train_data, test_data))

        for horizon in horizons_for_delta_experment:
            
            FODLER_or_DIR = "SAVED_MODELS/LSTMS"
            file_name = "LSTM" + "_" + str(horizon) + "_" + NAME + "_" + str(rho_value)+f"_scale_rho_{SCALE_PER_RHO}" + "_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pkl"
            FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
            model = load_models_from_file(FULL_PATH)

            jobs_delta.append((datasets_parameters, training_rho, testing_rho, train_data.copy(), test_data.copy(), all_data.copy(), [horizon], SCALE_PER_RHO, models_parameters.copy(), model))


with ThreadPoolExecutor(max_workers=32) as ex:
    for r in ex.map(worker_delta, jobs_delta):
        results_delta.append(r)


df_delta_LSTM = pd.DataFrame(results_delta)
df_delta_LSTM['train_rhos'] = df_delta_LSTM['train_rhos'].apply(lambda x: x[0] if isinstance(x, list) else x)
df_delta_LSTM['test_rhos'] = df_delta_LSTM['test_rhos'].apply(lambda x: x[0] if isinstance(x, list) else x)


df_delta_LSTM = df_delta_LSTM.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]).reset_index(drop=True)


FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "lstm_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)


df_delta_LSTM.to_pickle(FULL_PATH)


In [ ]:
# LOAD IT!

horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "lstm_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_delta_LSTM = pd.read_pickle(FULL_PATH)


In [ ]:
reset_matplotlib_defaults()
FIGS_DIR = "plots/delta_plots/lstm_plots"
os.makedirs(FIGS_DIR, exist_ok=True)

RHO_TO_PLOT_TRAIN = [50, 100, 150]
HORIZON_TO_PLOT = 10
RHO_TO_PLOT_TEST = 150
PLOT_MAX_VALUES = 1000

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10

for train_rho, df_rho in df_delta_LSTM.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] not in RHO_TO_PLOT_TRAIN:
        continue
    metrics_list = []
    plt.figure(figsize=(10, 5))
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row['dfs']['obs']
        predicted = row['dfs']['pred']


        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[models_parameters['seq_len'] :]
            predicted_temp = predicted_temp[models_parameters['seq_len'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

        metrics['test_rho'] = test_rho  
        metrics['horizon'] = horizon

        metrics_list.append(metrics)

    # Plot the list of metrics for this train_rho and horizon
    metrics_df = pd.DataFrame(metrics_list)
    plt.plot(metrics_df['test_rho'], metrics_df['mse'], label='MSE')
    plt.plot(metrics_df['test_rho'], metrics_df['sbd'], label='SBD')
    plt.plot(metrics_df['test_rho'], metrics_df['dstsp'], label=rf'$\mathrm{{D_{{stsp}}}}$') 
    # plt.plot(metrics_df['test_rho'], metrics_df['dh'], label='D_H')
    plt.title(rf"LSTM Train $\rho$={train_rho[0]} and $\tau$={horizon}")
    
    plt.xlabel(rf"Test $\rho$")
    plt.ylabel("Metric Value")
    plt.legend()
    plt.ylim(METRIC_RANGES_DELTA[f"r{train_rho[0]}"])
    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/lstm_train_rho_{train_rho[0]}_horizon_{horizon}_delta_metrics.pdf", format="pdf", bbox_inches="tight")
    plt.show()


for train_rho, df_rho in df_delta_LSTM.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] in RHO_TO_PLOT_TRAIN:
        for _, row in df_rho.iterrows():
            test_rho = row['test_rhos']
            horizon = row['forecast_horizons']
            observed = row['dfs']['obs']
            predicted = row['dfs']['pred']
            if test_rho ==  RHO_TO_PLOT_TEST and horizon == HORIZON_TO_PLOT:
                model_plot = rf"LSTM-A" 

                # DONE
                metrics_temp = []

                first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
                first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

                metrics_temp.append(compute_all_metrics(
                        first_sim_predicted, first_sim_observed,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))
                
                for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
                
                    observed_temp = observed[i:i+SIMULATION_LEN]
                    predicted_temp = predicted[i:i+SIMULATION_LEN]

                    observed_temp = observed_temp[models_parameters['seq_len'] :]
                    predicted_temp = predicted_temp[models_parameters['seq_len'] :]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

                title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={metrics['mse']:.3f}    SBD={metrics['sbd']:.3f}    $\mathbf{{D_{{stsp}}}}$={metrics['dstsp']:.3f}    $\mathbf{{D_{{H}}}}$={metrics['dh']:.3f}"
                fig = plot_3D_attractor(observed[0:0,:], predicted[0:PLOT_MAX_VALUES], ground_truth=observed[0:PLOT_MAX_VALUES,:], lim_pse=PLOT_MAX_VALUES, lim_gen=PLOT_MAX_VALUES, horizon = horizon, title_3D=model_plot, title_big = title_plot_3D)
                plt.savefig(f"{FIGS_DIR}/lstm_train_rho_{train_rho[0]}_test_rho_{test_rho}_horizon_{horizon}_delta_3D.pdf", format="pdf", bbox_inches="tight")
                plt.show()

                reset_matplotlib_defaults()



all_metrics = []  # for global average

for train_rho, df_rho in df_delta_LSTM.groupby("train_rhos"):
    rho_metrics = []
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row["dfs"]['obs']
        predicted = row['dfs']['pred']

        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[models_parameters['seq_len'] :]
            predicted_temp = predicted_temp[models_parameters['seq_len'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

        metrics['test_rho'] = test_rho
        metrics['horizon'] = horizon
        rho_metrics.append(metrics)
        all_metrics.append(metrics)
        
        # print(f"LSTM - Train Rho: {train_rho}, Test Rho: {test_rho}, Horizon: {horizon}")
        # print(f"  MSE: {metrics['mse']:.3f} | SBD: {metrics['sbd']:.3f} | D_stsp: {metrics['dstsp']:.3f} | D_H: {metrics['dh']:.3f}")
    
    metrics_df = pd.DataFrame(rho_metrics)
    
    # Per horizon average
    for horizon, h_df in metrics_df.groupby("horizon"):
        avg = h_df.drop(columns=['test_rho', 'horizon']).mean()
        print(f"\n>>> Train Rho {train_rho} | Horizon {horizon} | AVERAGE across {len(h_df)} test rhos:")
        print(f"  MSE: {avg['mse']:.3f} | SBD: {avg['sbd']:.3f} | D_stsp: {avg['dstsp']:.3f} | D_H: {avg['dh']:.3f}")
    
    # Per rho average (all horizons)
    avg_rho = metrics_df.drop(columns=['test_rho', 'horizon']).mean()
    print(f"\n>>> Train Rho {train_rho} | OVERALL AVERAGE (all horizons, all test rhos):")
    print(f"  MSE: {avg_rho['mse']:.3f} | SBD: {avg_rho['sbd']:.3f} | D_stsp: {avg_rho['dstsp']:.3f} | D_H: {avg_rho['dh']:.3f}\n")

# Global average
global_avg = pd.DataFrame(all_metrics).drop(columns=['test_rho', 'horizon']).mean()
print(f"\n{'='*60}")
print(f">>> LSTM GLOBAL AVERAGE (all rhos, all horizons, all test rhos):")
print(f"  MSE: {global_avg['mse']:.3f} | SBD: {global_avg['sbd']:.3f} | D_stsp: {global_avg['dstsp']:.3f} | D_H: {global_avg['dh']:.3f}")

### **Transformer**

In [ ]:
# Multi Experiment Transformer for ____ dataset

def run_single_transformer_experiment_delta(job):
    train_rho, test_rho, train_data, test_data, transformer_parameters, tau, model = job
    observed, predicted = run_transformer_pipeline(train_data, test_data, transformer_parameters, forecast_horizons=tau, rho_value=train_rho, model=model)

    return {
            "model": "Transformer",
            "train_rhos": train_rho,
            "test_rhos": test_rho,
            "forecast_horizons": tau,
            "observed": observed,
            "predicted": predicted
        }

transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

delta_combinatons = create_delta_combinations(rho_centers, DELTA_RHO)

jobs_delta = []
results_delta = []

# Start from the center and extend outwards until the maximum delta is reached, with steps of 1
for rho_value in rho_centers:

    training_rho = rho_value
    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]

    delta_combinatons_test = delta_combinatons[rho_value]

    for testing_rho in delta_combinatons_test:
        
        test_mask = np.isin(test_o[:, 6], testing_rho)
        test_data = test_o[test_mask]

        all_data = np.vstack((train_data, test_data))

        for horizon in horizons_for_delta_experment:

            FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
            file_name = "Transformer" + "_" + str(horizon) + "_" + NAME + "_" + str(training_rho)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"

            FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
            model = TransformerModel.load(FULL_PATH)

            jobs_delta.append((training_rho, testing_rho, train_data.copy(), test_data.copy(), transformer_parameters.copy(), horizon, model))


with ThreadPoolExecutor(max_workers=1) as executor:
    for r in executor.map(run_single_transformer_experiment_delta, jobs_delta):
        results_delta.append(r)


df_delta_transformer = pd.DataFrame(results_delta)
df_delta_transformer = df_delta_transformer.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]).reset_index(drop=True)


FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "transformer_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)


df_delta_transformer.to_pickle(FULL_PATH)

In [ ]:
horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

FOLDER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FOLDER_or_DIR, exist_ok=True)

file_name = "transformer_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pt"
FULL_PATH = os.path.join(FOLDER_or_DIR, file_name)

df_delta_transformer = pd.read_pickle(FULL_PATH)


In [ ]:
transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

In [ ]:
reset_matplotlib_defaults()
FIGS_DIR = "plots/delta_plots/transformer"
os.makedirs(FIGS_DIR, exist_ok=True)

RHO_TO_PLOT_TRAIN = [50, 100, 150]
HORIZON_TO_PLOT = 10
RHO_TO_PLOT_TEST = 150
PLOT_MAX_VALUES = 1000

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10



os.makedirs(FIGS_DIR, exist_ok=True)

for train_rho, df_rho in df_delta_transformer.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] not in RHO_TO_PLOT_TRAIN:
        continue
    metrics_list = []
    plt.figure(figsize=(10, 5))
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row['observed']
        predicted = row['predicted']

        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
            
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[transformer_parameters['input_length'] :]
            predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

        
        metrics['test_rho'] = test_rho  
        metrics['horizon'] = horizon

        metrics_list.append(metrics)
    # Plot the list of metrics for this train_rho and horizon
    metrics_df = pd.DataFrame(metrics_list)
    plt.plot(metrics_df['test_rho'], metrics_df['mse'], label='MSE')
    plt.plot(metrics_df['test_rho'], metrics_df['sbd'], label='SBD')
    plt.plot(metrics_df['test_rho'], metrics_df['dstsp'], label=rf'$\mathrm{{D_{{stsp}}}}$') 
    # plt.plot(metrics_df['test_rho'], metrics_df['dh'], label='D_H')
    plt.title(rf"Transformer Train $\rho$={train_rho[0]} and $\tau$={horizon}")
    plt.xlabel(rf"Test $\rho$")
    plt.ylabel("Metric Value")
    plt.legend()
    plt.ylim(METRIC_RANGES_DELTA[f"r{train_rho[0]}"])
    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/transformer_train_rho_{train_rho[0]}_horizon_{horizon}_delta_metrics.pdf", format="pdf", bbox_inches="tight")
    plt.show()


for train_rho, df_rho in df_delta_transformer.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] in RHO_TO_PLOT_TRAIN:
        for _, row in df_rho.iterrows():
            test_rho = row['test_rhos']
            horizon = row['forecast_horizons']
            observed = row['observed']
            predicted = row['predicted']
            if test_rho ==  RHO_TO_PLOT_TEST and horizon == HORIZON_TO_PLOT:
                model_plot = rf"Transformer" 

                # DONE
                metrics_temp = []

                first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
                first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

                metrics_temp.append(compute_all_metrics(
                        first_sim_predicted, first_sim_observed,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))
                
                for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
                
                    observed_temp = observed[i:i+SIMULATION_LEN]
                    predicted_temp = predicted[i:i+SIMULATION_LEN]

                    observed_temp = observed_temp[transformer_parameters['input_length'] :]
                    predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}


                title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={metrics['mse']:.3f}    SBD={metrics['sbd']:.3f}    $\mathbf{{D_{{stsp}}}}$={metrics['dstsp']:.3f}    $\mathbf{{D_{{H}}}}$={metrics['dh']:.3f}"
                fig = plot_3D_attractor(observed[0:0,:], predicted[0:PLOT_MAX_VALUES], ground_truth=observed[0:PLOT_MAX_VALUES,:], lim_pse=PLOT_MAX_VALUES, lim_gen=PLOT_MAX_VALUES, horizon = horizon, title_3D=model_plot, title_big = title_plot_3D)
                plt.savefig(f"{FIGS_DIR}/transformer_train_rho_{train_rho[0]}_test_rho_{test_rho}_horizon_{horizon}_delta_3D.pdf", format="pdf", bbox_inches="tight")
                plt.show()

                reset_matplotlib_defaults()


all_metrics = []  # for global average

for train_rho, df_rho in df_delta_transformer.groupby("train_rhos"):
    rho_metrics = []
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row["observed"]
        predicted = row['predicted']
        
        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-transformer_parameters['input_length'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-transformer_parameters['input_length'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[transformer_parameters['input_length'] :]
            predicted_temp = predicted_temp[transformer_parameters['input_length'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}
        
        metrics['test_rho'] = test_rho
        metrics['horizon'] = horizon
        rho_metrics.append(metrics)
        all_metrics.append(metrics)
        
        # print(f"Transformer - Train Rho: {train_rho}, Test Rho: {test_rho}, Horizon: {horizon}")
        # print(f"  MSE: {metrics['mse']:.3f} | SBD: {metrics['sbd']:.3f} | D_stsp: {metrics['dstsp']:.3f} | D_H: {metrics['dh']:.3f}")
    
    metrics_df = pd.DataFrame(rho_metrics)
    
    # Per horizon average
    for horizon, h_df in metrics_df.groupby("horizon"):
        avg = h_df.drop(columns=['test_rho', 'horizon']).mean()
        stderr = h_df.drop(columns=['test_rho', 'horizon']).std()

        print(f"\n>>> Train Rho {train_rho} | Horizon {horizon} | AVERAGE across {len(h_df)} test rhos:")
        print(f"  MSE: {avg['mse']:.3f} | SBD: {avg['sbd']:.3f} | D_stsp: {avg['dstsp']:.3f} | D_H: {avg['dh']:.3f}")
        print(f"  MSE std: {stderr['mse']:.3f} | SBD std: {stderr['sbd']:.3f} | D_stsp std: {stderr['dstsp']:.3f} | D_H std: {stderr['dh']:.3f}")

    # Per rho average (all horizons)
    avg_rho = metrics_df.drop(columns=['test_rho', 'horizon']).mean()
    std_rho = metrics_df.drop(columns=['test_rho', 'horizon']).std()
    print(f"\n>>> Train Rho {train_rho} | OVERALL AVERAGE (all horizons, all test rhos):")
    print(f"  MSE: {avg_rho['mse']:.3f} | SBD: {avg_rho['sbd']:.3f} | D_stsp: {avg_rho['dstsp']:.3f} | D_H: {avg_rho['dh']:.3f}")
    print(f"  MSE std: {std_rho['mse']:.3f} | SBD std: {std_rho['sbd']:.3f} | D_stsp std: {std_rho['dstsp']:.3f} | D_H std: {std_rho['dh']:.3f}\n")


# Global average
global_avg = pd.DataFrame(all_metrics).drop(columns=['test_rho', 'horizon']).mean()
global_std = pd.DataFrame(all_metrics).drop(columns=['test_rho', 'horizon']).std()
print(f"\n{'='*60}")
print(f">>> Transformer GLOBAL AVERAGE (all rhos, all horizons, all test rhos):")
print(f"  MSE: {global_avg['mse']:.3f} | SBD: {global_avg['sbd']:.3f} | D_stsp: {global_avg['dstsp']:.3f} | D_H: {global_avg['dh']:.3f}")
print(f"  MSE std: {global_std['mse']:.3f} | SBD std: {global_std['sbd']:.3f} | D_stsp std: {global_std['dstsp']:.3f} | D_H std: {global_std['dh']:.3f}")

### **TFT**

In [ ]:
def run_single_tft_experiment_delta(job):
    train_rho, test_rho, train_data, test_data, tft_parameters, tau, model = job
    observed, predicted = run_tft_pipeline(train_data, test_data, tft_parameters, forecast_horizons=tau, rho_value=train_rho, model=model)

    return {
            "model": "TFT",
            "train_rhos": train_rho,
            "test_rhos": test_rho,
            "forecast_horizons": tau,
            "observed": observed,
            "predicted": predicted
        }

tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}


horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

delta_combinatons = create_delta_combinations(rho_centers, DELTA_RHO)

jobs_delta = []
results_delta = []

# Start from the center and extend outwards until the maximum delta is reached, with steps of 1
for rho_value in rho_centers:

    training_rho = rho_value
    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]

    delta_combinatons_test = delta_combinatons[rho_value]

    for testing_rho in delta_combinatons_test:
        
        test_mask = np.isin(test_o[:, 6], testing_rho)
        test_data = test_o[test_mask]

        all_data = np.vstack((train_data, test_data))

        for horizon in horizons_for_delta_experment:
             
            FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
            file_name = "TFT" + "_" + str(horizon) + "_" + NAME + "_" + str(training_rho)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"

            FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
            model = TFTModel.load(FULL_PATH)
            jobs_delta.append((training_rho, testing_rho, train_data.copy(), test_data.copy(), tft_parameters.copy(), horizon, model))


with ThreadPoolExecutor(max_workers=1) as executor:
    for r in executor.map(run_single_tft_experiment_delta, jobs_delta):
        results_delta.append(r)


df_delta_tft = pd.DataFrame(results_delta)
df_delta_tft = df_delta_tft.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]).reset_index(drop=True)    


FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "tft_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pt"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)  


df_delta_tft.to_pickle(FULL_PATH)

In [ ]:
horizons_for_delta_experment = [10, 20]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "tft_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pt"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_delta_tft = pd.read_pickle(FULL_PATH)


In [ ]:
tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}


In [ ]:
reset_matplotlib_defaults()
FIGS_DIR = "plots/delta_plots/tft_plots"
os.makedirs(FIGS_DIR, exist_ok=True)

RHO_TO_PLOT_TRAIN = [50, 100, 150]
HORIZON_TO_PLOT = 10
RHO_TO_PLOT_TEST = 150
PLOT_MAX_VALUES = 1000

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10


for train_rho, df_rho in df_delta_tft.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] not in RHO_TO_PLOT_TRAIN:
        continue
    metrics_list = []
    plt.figure(figsize=(10, 5))
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row['observed']
        predicted = row['predicted']


        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[tft_parameters['input_length'] :]
            predicted_temp = predicted_temp[tft_parameters['input_length'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}


        metrics['test_rho'] = test_rho  
        metrics['horizon'] = horizon
        metrics_list.append(metrics)
    # Plot the list of metrics for this train_rho and horizon
    metrics_df = pd.DataFrame(metrics_list)
    plt.plot(metrics_df['test_rho'], metrics_df['mse'], label='MSE')
    plt.plot(metrics_df['test_rho'], metrics_df['sbd'], label='SBD')
    plt.plot(metrics_df['test_rho'], metrics_df['dstsp'], label=rf'$\mathrm{{D_{{stsp}}}}$') 
    # plt.plot(metrics_df['test_rho'], metrics_df['dh'], label='D_H')
    plt.title(rf"TFT Metrics for Train $\rho$={train_rho[0]} and $\tau$={horizon}")
    plt.xlabel(rf"Test $\rho$")
    plt.ylabel("Metric Value")
    plt.legend()
    plt.ylim(METRIC_RANGES_DELTA[f"r{train_rho[0]}"])
    plt.tight_layout()
    plt.savefig(f"{FIGS_DIR}/tft_train_rho_{train_rho[0]}_horizon_{horizon}_delta_metrics.pdf", format="pdf", bbox_inches="tight")
    plt.show()


for train_rho, df_rho in df_delta_tft.groupby(["train_rhos", "forecast_horizons"]):
    if train_rho[0] in RHO_TO_PLOT_TRAIN:
        for _, row in df_rho.iterrows():
            test_rho = row['test_rhos']
            horizon = row['forecast_horizons']
            observed = row['observed']
            predicted = row['predicted']
            if test_rho ==  RHO_TO_PLOT_TEST and horizon == HORIZON_TO_PLOT:
                model_plot = rf"TFT" 

                # DONE
                metrics_temp = []

                first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
                first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

                metrics_temp.append(compute_all_metrics(
                        first_sim_predicted, first_sim_observed,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))
                
                for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
                
                    observed_temp = observed[i:i+SIMULATION_LEN]
                    predicted_temp = predicted[i:i+SIMULATION_LEN]

                    observed_temp = observed_temp[tft_parameters['input_length'] :]
                    predicted_temp = predicted_temp[tft_parameters['input_length'] :]

                    metrics_temp.append(compute_all_metrics(
                        predicted_temp, observed_temp,
                        bins=BINS, smoothing=SMOOTHING,
                        mase_steps=MASE_STEPS, scale=False
                    ))

                metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
                metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

                title_plot_3D = rf"$\boldsymbol{{\tau}}$={horizon}    MSE={metrics['mse']:.3f}    SBD={metrics['sbd']:.3f}    $\mathbf{{D_{{stsp}}}}$={metrics['dstsp']:.3f}    $\mathbf{{D_{{H}}}}$={metrics['dh']:.3f}"
                fig = plot_3D_attractor(observed[0:0,:], predicted[0:PLOT_MAX_VALUES], ground_truth=observed[0:PLOT_MAX_VALUES,:], lim_pse=PLOT_MAX_VALUES, lim_gen=PLOT_MAX_VALUES, horizon = horizon, title_3D=model_plot, title_big = title_plot_3D)
                plt.savefig(f"{FIGS_DIR}/tft_train_rho_{train_rho[0]}_test_rho_{test_rho}_horizon_{horizon}_delta_3D.pdf", format="pdf", bbox_inches="tight")
                plt.show()

                reset_matplotlib_defaults()

all_metrics = []  # for global average

for train_rho, df_rho in df_delta_tft.groupby("train_rhos"):
    rho_metrics = []
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row["observed"]
        predicted = row['predicted']
        
        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-tft_parameters['input_length'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-tft_parameters['input_length'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[tft_parameters['input_length'] :]
            predicted_temp = predicted_temp[tft_parameters['input_length'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}


        metrics['test_rho'] = test_rho
        metrics['horizon'] = horizon
        rho_metrics.append(metrics)
        all_metrics.append(metrics)
        
        # print(f"TFT - Train Rho: {train_rho}, Test Rho: {test_rho}, Horizon: {horizon}")
        # print(f"  MSE: {metrics['mse']:.3f} | SBD: {metrics['sbd']:.3f} | D_stsp: {metrics['dstsp']:.3f} | D_H: {metrics['dh']:.3f}")
    
    metrics_df = pd.DataFrame(rho_metrics)
    
    # Per horizon average
    for horizon, h_df in metrics_df.groupby("horizon"):
        avg = h_df.drop(columns=['test_rho', 'horizon']).mean()
        stderr = h_df.drop(columns=['test_rho', 'horizon']).std()
        print(f"\n>>> Train Rho {train_rho} | Horizon {horizon} | AVERAGE across {len(h_df)} test rhos:")
        print(f"  MSE: {avg['mse']:.3f} | SBD: {avg['sbd']:.3f} | D_stsp: {avg['dstsp']:.3f} | D_H: {avg['dh']:.3f}")
        print(f"  MSE std: {stderr['mse']:.3f} | SBD std: {stderr['sbd']:.3f} | D_stsp std: {stderr['dstsp']:.3f} | D_H std: {stderr['dh']:.3f}")

    # Per rho average (all horizons)
    avg_rho = metrics_df.drop(columns=['test_rho', 'horizon']).mean()
    std_rho = metrics_df.drop(columns=['test_rho', 'horizon']).std()
    print(f"\n>>> Train Rho {train_rho} | OVERALL AVERAGE (all horizons, all test rhos):")
    print(f"  MSE: {avg_rho['mse']:.3f} | SBD: {avg_rho['sbd']:.3f} | D_stsp: {avg_rho['dstsp']:.3f} | D_H: {avg_rho['dh']:.3f}")
    print(f"  MSE std: {std_rho['mse']:.3f} | SBD std: {std_rho['sbd']:.3f} | D_stsp std: {std_rho['dstsp']:.3f} | D_H std: {std_rho['dh']:.3f}\n")

# Global average
global_avg = pd.DataFrame(all_metrics).drop(columns=['test_rho', 'horizon']).mean()
print(f"\n{'='*60}")
print(f">>> TFT GLOBAL AVERAGE (all rhos, all horizons, all test rhos):")
print(f"  MSE: {global_avg['mse']:.3f} | SBD: {global_avg['sbd']:.3f} | D_stsp: {global_avg['dstsp']:.3f} | D_H: {global_avg['dh']:.3f}")
print(f"  MSE std: {global_std['mse']:.3f} | SBD std: {global_std['sbd']:.3f} | D_stsp std: {global_std['dstsp']:.3f} | D_H std: {global_std['dh']:.3f}")

### **UPOs**

In [ ]:
# UPO BASED DELTA SENSITIVITY ANALYSIS
# CONTEXT UPOS ARE LOADED FOR THE CENTRAL RHO AND WE WILL PREDICT DATA WITH RHO VALUES IN DELTA RHO

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

UPO_DT = 0.001
UPO_K = 1

horizons_for_delta_experment = [10, 20]

rho_centers = [50, 100, 150]
rho_centers = [50, 100, 150]
DELTA_RHO = 5

scaler = StandardScaler()
upo_data_delta_senz = defaultdict(dict)

for RHO in rho_centers:
    custom_dt_rounded = round(UPO_DT, 3)
    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{custom_dt_rounded}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_delta_senz[RHO][custom_dt_rounded] = df_orbits_individual



delta_combinatons = create_delta_combinations(rho_centers, DELTA_RHO)

jobs_delta = []
results_delta = []

# Start from the center and extend outwards until the maximum delta is reached, with steps of 1
for rho_value in rho_centers:

    # training_rho = rho_value
    # train_mask = np.isin(train_o[:, 6], training_rho)
    # train_data = train_o[train_mask]

    delta_combinatons_test = delta_combinatons[rho_value]

    df_upo = upo_data_delta_senz[rho_value][UPO_DT]
    tree, labels, df_upo_extended = build_labeled_tree(df_upo)

    for testing_rho in delta_combinatons_test:
        
        test_mask = np.isin(test_o[:, 6], testing_rho)
        test_data = test_o[test_mask]

        obs_xyz = test_data[:, 3:6]

        for horizon in horizons_for_delta_experment:
             
            observed, _, predicted, _ = UPO_prediction_window(
                    obs_xyz,
                    window_size=horizon,
                    tree=tree,
                    labels=labels,
                    norm=2,
                    k=UPO_K,
                    upo_df=df_upo_extended,
                    time_diff=int(DT / custom_dt_rounded)
                )

            scaler.fit(np.vstack([observed, predicted]))
            observed_scaled = scaler.transform(observed)
            predicted_scaled = scaler.transform(predicted)

            results_delta.append({
                "model": "UPO",
                "train_rhos": rho_value,
                "test_rhos": testing_rho,
                "forecast_horizons": horizon,
                "observed": observed_scaled,
                "predicted": predicted_scaled
                })
        
            # jobs_delta.append((training_rho, testing_rho, train_data.copy(), test_data.copy(), tft_parameters.copy(), horizon, model))

df_delta_UPO = pd.DataFrame(results_delta)
df_delta_UPO = df_delta_UPO.sort_values(
    ["train_rhos", "test_rhos", "forecast_horizons"]).reset_index(drop=True)    


FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "UPO_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pt"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)  


df_delta_UPO.to_pickle(FULL_PATH)

In [ ]:
horizons_for_delta_experment = [10, 20]
rho_centers = [50]
DELTA_RHO = 5

FODLER_or_DIR = "outer_limits_results/delta_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "UPO_vs_delta_results_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_for_delta_experment[-1]) + rf"{EXPERIMENT_NAME}" + ".pt"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)  

df_delta_UPO = pd.read_pickle(FULL_PATH)


In [ ]:
metrics_df

In [ ]:
reset_matplotlib_defaults()
FIGS_DIR = "plots/delta_plots/upo_plots"
os.makedirs(FIGS_DIR, exist_ok=True)

RHO_TO_PLOT_TRAIN = [50, 100, 150]
HORIZON_TO_PLOT = 10
RHO_TO_PLOT_TEST = 150
PLOT_MAX_VALUES = 1000

BINS = 5
SMOOTHING = 10
MASE_STEPS = 10


# for train_rho, df_rho in df_delta_UPO.groupby(["train_rhos", "forecast_horizons"]):
#     if train_rho[0] not in RHO_TO_PLOT_TRAIN:
#         continue

#     metrics_list = []
#     plt.figure(figsize=(10, 5))
    
#     for _, row in df_rho.iterrows():
#         test_rho = row['test_rhos']
#         horizon = row['forecast_horizons']
#         observed = row['observed']
#         predicted = row['predicted']


#         # DONE
#         metrics_temp = []

#         first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len']]
#         first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

#         metrics_temp.append(compute_all_metrics(
#                 first_sim_predicted, first_sim_observed,
#                 bins=BINS, smoothing=SMOOTHING,
#                 mase_steps=MASE_STEPS, scale=False
#             ))
        
#         for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
#             observed_temp = observed[i:i+SIMULATION_LEN]
#             predicted_temp = predicted[i:i+SIMULATION_LEN]

#             observed_temp = observed_temp[models_parameters['seq_len'] :]
#             predicted_temp = predicted_temp[models_parameters['seq_len'] :]

#             metrics_temp.append(compute_all_metrics(
#                 predicted_temp, observed_temp,
#                 bins=BINS, smoothing=SMOOTHING,
#                 mase_steps=MASE_STEPS, scale=False
#             ))

#         metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
#         metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}


#         metrics['test_rho'] = test_rho  
#         metrics['horizon'] = horizon
#         metrics_list.append(metrics)
#     # Plot the list of metrics for this train_rho and horizon
#     metrics_df = pd.DataFrame(metrics_list)
#     plt.plot(metrics_df['test_rho'], metrics_df['mse'], label='MSE')
#     plt.plot(metrics_df['test_rho'], metrics_df['sbd'], label='SBD')
#     plt.plot(metrics_df['test_rho'], metrics_df['dstsp'], label=rf'$\mathrm{{D_{{stsp}}}}$') 
#     # plt.plot(metrics_df['test_rho'], metrics_df['dh'], label='D_H')
#     plt.title(rf"UPO Metrics for Train $\rho$={train_rho[0]} and $\tau$={horizon}")
#     plt.xlabel(rf"Test $\rho$")
#     plt.ylabel("Metric Value")
#     plt.legend()
#     plt.ylim(METRIC_RANGES_DELTA_UPO[f"r{train_rho[0]}"])
#     plt.tight_layout()
#     plt.savefig(f"{FIGS_DIR}/upo_train_rho_{train_rho[0]}_horizon_{horizon}_delta_metrics.pdf", format="pdf", bbox_inches="tight")
#     plt.show()


all_metrics = []  # for global average

for train_rho, df_rho in df_delta_UPO.groupby("train_rhos"):
    rho_metrics = []
    for _, row in df_rho.iterrows():
        test_rho = row['test_rhos']
        horizon = row['forecast_horizons']
        observed = row["observed"]
        predicted = row['predicted']

        # DONE
        metrics_temp = []

        first_sim_observed = observed[0:SIMULATION_LEN-models_parameters['seq_len'] ]
        first_sim_predicted = predicted[0:SIMULATION_LEN-models_parameters['seq_len'] ]

        metrics_temp.append(compute_all_metrics(
                first_sim_predicted, first_sim_observed,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))
        
        for i in range(SIMULATION_LEN, len(predicted)-SIMULATION_LEN, SIMULATION_LEN):
        
            observed_temp = observed[i:i+SIMULATION_LEN]
            predicted_temp = predicted[i:i+SIMULATION_LEN]

            observed_temp = observed_temp[models_parameters['seq_len'] :]
            predicted_temp = predicted_temp[models_parameters['seq_len'] :]

            metrics_temp.append(compute_all_metrics(
                predicted_temp, observed_temp,
                bins=BINS, smoothing=SMOOTHING,
                mase_steps=MASE_STEPS, scale=False
            ))

        metrics = {k: np.mean([met[k] for met in metrics_temp]) for k in metrics_temp[0].keys()}
        metrics = {k: v.item() if hasattr(v, 'item') else v for k, v in metrics.items()}

        metrics['test_rho'] = test_rho
        metrics['horizon'] = horizon
        rho_metrics.append(metrics)
        all_metrics.append(metrics)
        
        # print(f"UPO - Train Rho: {train_rho}, Test Rho: {test_rho}, Horizon: {horizon}")
        # print(f"  MSE: {metrics['mse']:.3f} | SBD: {metrics['sbd']:.3f} | D_stsp: {metrics['dstsp']:.3f} | D_H: {metrics['dh']:.3f}")
    
    metrics_df = pd.DataFrame(rho_metrics)
    
    # Per horizon average
    for horizon, h_df in metrics_df.groupby("horizon"):
        avg = h_df.drop(columns=['test_rho', 'horizon']).mean()
        print(f"\n>>> Train Rho {train_rho} | Horizon {horizon} | AVERAGE across {len(h_df)} test rhos:")
        print(f"  MSE: {avg['mse']:.3f} | SBD: {avg['sbd']:.3f} | D_stsp: {avg['dstsp']:.3f} | D_H: {avg['dh']:.3f}")
    
    # Per rho average (all horizons)
    avg_rho = metrics_df.drop(columns=['test_rho', 'horizon']).mean()
    print(f"\n>>> Train Rho {train_rho} | OVERALL AVERAGE (all horizons, all test rhos):")
    print(f"  MSE: {avg_rho['mse']:.3f} | SBD: {avg_rho['sbd']:.3f} | D_stsp: {avg_rho['dstsp']:.3f} | D_H: {avg_rho['dh']:.3f}\n")

# Global average
global_avg = pd.DataFrame(all_metrics).drop(columns=['test_rho', 'horizon']).mean()
print(f"\n{'='*60}")
print(f">>> UPO GLOBAL AVERAGE (all rhos, all horizons, all test rhos):")
print(f"  MSE: {global_avg['mse']:.3f} | SBD: {global_avg['sbd']:.3f} | D_stsp: {global_avg['dstsp']:.3f} | D_H: {global_avg['dh']:.3f}")

## **Hyperparameter Evaluation**

In [ ]:
# Parameters

SEQUENCE_LENGTH_RANGE = range(100, 650, 50)  # from 100 to 550 with step of 50
FORECAST_HORIZONS_RANGE = [10, 40]

# UPO_DT_RANGE = np.arange(0.001, 0.051, 0.001)  # from 0.0001 to 0.05 with step of 0.001
UPO_DT_RANGE = [0.001,0.003, 0.005, 0.01, 0.03, 0.05, 0.07, 0.1]
# UPO_DT_RANGE =   [ 0.07, 0.1, 0.15]
UPO_K_RANGE = range(1, 51, 1)  # from 1 to 49 with step of 1

HYPER_RHO = 28

train_mask = np.isin(train_o[:, 6], HYPER_RHO)
train_data = train_o[train_mask]

test_mask = np.isin(test_o[:, 6], HYPER_RHO)
test_data = test_o[test_mask]
all_data = np.vstack((train_data, test_data))

scaler_UPOs = StandardScaler()
test_data_scaled = scaler_UPOs.fit_transform(test_data[:, 3:6])


BINS = 5
SMOOTHING = 10
MASE_STEPS = 10

plots_dir = "plots/hyper_results"
os.makedirs(plots_dir, exist_ok=True)


PLOT_SIZES = [10, 5]

PLOT_RANGES_METRICS_HYPER_10 = {
    "mse": (0, 1.25),
    "dstsp": (0, 0.5),
}
PLOT_RANGES_METRICS_HYPER_40 = {
    "mse": (0, 2),
    "dstsp": (0, 2),
}

PLOT_RANGES_METRICS_HYPER_10_UPO = {
    "mse": (0, 1.25),
    "dstsp": (0, 0.006),
}
PLOT_RANGES_METRICS_HYPER_40_UPO = {
    "mse": (0, 2),
    "dstsp": (0, 0.02),
}

### **LSTM**

In [ ]:
def worker_hyper(job):
    datasets_parameters, HYPER_RHO, train_data, test_data, all_data, horizon, SCALE_PER_RHO, models_parameters, TEACHER_FORCING = job

    print(f"Starting job with {models_parameters['seq_len']} for horizon {horizon} and rho {HYPER_RHO} and teacher forcing {TEACHER_FORCING}")

    # dfs = train_test_single_model_thread_delta(                             
    #                          datasets_parameters = datasets_parameters, 
    #                          train_rhos = HYPER_RHO, 
    #                          test_rhos = HYPER_RHO, 
    #                          train_data = train_data,
    #                          test_data = test_data, 
    #                          all_data = all_data,
    #                          forecast_horizons = [horizon], 
    #                          scale_per_rho = SCALE_PER_RHO,
    #                          models_parameters = models_parameters)

    dfs = train_test_single_model_thread_hyperparameters(
                             datasets_parameters = datasets_parameters, 
                             train_rhos = HYPER_RHO, 
                             test_rhos = HYPER_RHO, 
                             train_data = train_data,
                             test_data = test_data, 
                             all_data = all_data,
                             forecast_horizons = [horizon],
                             scale_per_rho = SCALE_PER_RHO, 
                             models_parameters = models_parameters,
                             TEACHER_FORCING = TEACHER_FORCING,
                             )
    
    observed = dfs['obs']
    predicted = dfs['pred']
    
    if TEACHER_FORCING:
        model_type = "LSTM-B"
    else:
        model_type = "LSTM-A"

    print(f"ENDING job with {models_parameters['seq_len']} for horizon {horizon} and rho {HYPER_RHO} and teacher forcing {TEACHER_FORCING}\n")

    return {
            "model": model_type,
            "rho": HYPER_RHO,
            "SL": models_parameters['seq_len'],
            "horizon": horizon,
            "observed": observed,
            "predicted": predicted
        }

# TF GEN
SCALE_PER_RHO = True
TEACHER_FORCING = False
    
models_parameters['batch_size'] = 32
models_parameters['epochs'] = 300

jobs_hyper_eval = []
results_hyper_eval = []

for SL in SEQUENCE_LENGTH_RANGE:
    models_parameters['seq_len'] = SL

    for horizon in FORECAST_HORIZONS_RANGE:
        models_parameters['forecast_horizons'] = horizon
        for TF in [True, False]:

            if TF:
                models_parameters['rolling'] = False
            else:
                models_parameters['rolling'] = True

            if TF:
                models_parameters['dropout'] = 0.2
                models_parameters['recurrent_dropout'] = 0.2
                models_parameters['l2_reg'] = 1e-4
                models_parameters['clipnorm'] = 1.5
            else:
                models_parameters['dropout'] = 0.0
                models_parameters['recurrent_dropout'] = 0.0
                models_parameters['l2_reg'] = 0.0
                models_parameters['clipnorm'] = None
                
            jobs_hyper_eval.append((datasets_parameters,
                                    HYPER_RHO, 
                                    train_data, 
                                    test_data,
                                    all_data, 
                                    horizon, 
                                    SCALE_PER_RHO, 
                                    models_parameters.copy(), 
                                    TF))

with ThreadPoolExecutor(max_workers=1) as ex:
    for r in ex.map(worker_hyper, jobs_hyper_eval):
        results_hyper_eval.append(r)

df_hyper_LSTM = pd.DataFrame(results_hyper_eval)
df_hyper_LSTM = df_hyper_LSTM.sort_values(
    ["model", "rho", "SL", "horizon"]).reset_index(drop=True)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "lstm_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+f"_scale_rho_{SCALE_PER_RHO}" + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_LSTM.to_pickle(FULL_PATH)

In [ ]:
# LOAD IT
FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "lstm_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+f"_scale_rho_{SCALE_PER_RHO}" + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_LSTM = pd.read_pickle(FULL_PATH)

In [ ]:

reset_matplotlib_defaults()
results_hyper_LSTM = hyperparameter_analysis(df_hyper_LSTM, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)


metrics_to_plot = ["mse", "sbd", "dstsp", "pe"]
metrics_to_plot = ["mse",  "dstsp"]
horizons = results_hyper_LSTM["horizon"].unique()
models = results_hyper_LSTM["model"].unique()

reset_matplotlib_defaults()

for metric in metrics_to_plot:
    # fig, axes = plt.subplots(1, len(horizons), figsize=(PLOT_SIZES[0] * len(horizons), PLOT_SIZES[1]), sharey=True)

    for  hz in zip(horizons):

        fig, ax = plt.subplots(figsize=(PLOT_SIZES[0], PLOT_SIZES[1])) 
        df_hz = results_hyper_LSTM[results_hyper_LSTM["horizon"] == hz]
        for model in models:
            df_m = df_hz[df_hz["model"] == model].sort_values("SL")
            ax.plot(df_m["SL"], df_m[metric], marker="o", label=model)

        ax.set_title(f"Forecast Horizon: {hz[0]}")
        ax.set_xlabel("Context Window")
        ax.set_ylabel(metric.upper())
        if hz[0] == 10:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10[metric])
        else:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40[metric])
        ax.legend()
        plt.tight_layout()
        save_path = os.path.join(plots_dir, f"{models}_{metric}_horizon_{hz[0]}.pdf")
        plt.savefig(save_path, bbox_inches="tight", format="pdf")
        plt.show()

### **Transformer**

In [ ]:
transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 128,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": False
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

In [ ]:
def run_single_transformer_experiment_hyper(job):
    rho, train_data, test_data, sequence_length, tau,  transformer_parameters  = job
    local_params = transformer_parameters.copy()
    local_params['input_length'] = sequence_length

    print(f"Starting Transformer job with SL {local_params['input_length']} for horizon {tau} and rho {rho}")
    observed, predicted = run_transformer_pipeline(train = train_data, 
                                                   test = test_data, 
                                                   local_parameters= local_params, 
                                                   forecast_horizons=tau, 
                                                   rho_value=rho)

    results = {
        "model": "Transformer",
        "rho": rho,
        "SL": sequence_length,
        "horizon": tau,
        "observed": observed,
        "predicted": predicted
    }

    return results

jobs_hyper_eval = []
results_hyper_eval = []

for SL in SEQUENCE_LENGTH_RANGE:

    for horizon in FORECAST_HORIZONS_RANGE:

        jobs_hyper_eval.append((HYPER_RHO, 
                                train_data, 
                                test_data,
                                SL,
                                horizon, 
                                transformer_parameters))


with ThreadPoolExecutor(max_workers=1) as ex:
    for r in ex.map(run_single_transformer_experiment_hyper, jobs_hyper_eval):
        results_hyper_eval.append(r)

df_hyper_transformer = pd.DataFrame(results_hyper_eval)
df_hyper_transformer = df_hyper_transformer.sort_values(
    ["model", "rho", "SL", "horizon"]).reset_index(drop=True)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "transformer_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+ rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_transformer.to_pickle(FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "transformer_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+ rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_transformer = pd.read_pickle(FULL_PATH)

In [ ]:

results_hyper_transformer = hyperparameter_analysis(df_hyper_transformer, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)


metrics_to_plot = ["mse", "sbd", "dstsp", "pe"]
metrics_to_plot = ["mse",  "dstsp"]
horizons = results_hyper_transformer["horizon"].unique()
models = results_hyper_transformer["model"].unique()

reset_matplotlib_defaults()

for metric in metrics_to_plot:
    # fig, axes = plt.subplots(1, len(horizons), figsize=(PLOT_SIZES[0] * len(horizons), PLOT_SIZES[1]), sharey=True)

    for  hz in zip(horizons):

        fig, ax = plt.subplots(figsize=(PLOT_SIZES[0], PLOT_SIZES[1])) 
        df_hz = results_hyper_transformer[results_hyper_transformer["horizon"] == hz]
        for model in models:
            df_m = df_hz[df_hz["model"] == model].sort_values("SL")
            ax.plot(df_m["SL"], df_m[metric], marker="o", label=model)

        ax.set_title(f"Forecast Horizon: {hz[0]}")
        ax.set_xlabel("Context Window")
        ax.set_ylabel(metric.upper())
        if hz[0] == 10:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10[metric])
        else:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40[metric])
        ax.legend()
        plt.tight_layout()
        save_path = os.path.join(plots_dir, f"{metric}_horizon_{hz[0]}.pdf")
        plt.savefig(save_path, bbox_inches="tight", format="pdf")
        plt.show()

### **TFT**

In [ ]:
tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 128,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}


In [ ]:
def run_single_tft_experiment_hyper(job):
    rho, train_data, test_data, sequence_length, tau,  tft_parameters  = job
    local_params = tft_parameters.copy()
    local_params['input_length'] = sequence_length

    print(f"Starting TFT job with SL {local_params['input_length']} for horizon {tau} and rho {rho}")
    observed, predicted = run_tft_pipeline(train_data, test_data, local_params, forecast_horizons=tau, rho_value=rho)

    results = {
        "model": "TFT",
        "rho": rho,
        "SL": sequence_length,
        "horizon": tau,
        "observed": observed,
        "predicted": predicted
    }

    return results


jobs_hyper_eval = []
results_hyper_eval = []

for SL in SEQUENCE_LENGTH_RANGE:
    for horizon in FORECAST_HORIZONS_RANGE:
        jobs_hyper_eval.append((HYPER_RHO, 
                                train_data, 
                                test_data,
                                SL,
                                horizon, 
                                tft_parameters))


with ThreadPoolExecutor(max_workers=1) as ex:
    for r in ex.map(run_single_tft_experiment_hyper, jobs_hyper_eval):
        results_hyper_eval.append(r)

df_hyper_tft = pd.DataFrame(results_hyper_eval)
df_hyper_tft = df_hyper_tft.sort_values(
    ["model", "rho", "SL", "horizon"]).reset_index(drop=True)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "tft_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+ rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_tft.to_pickle(FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "tft_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO)+ rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_hyper_tft = pd.read_pickle(FULL_PATH)

### **PLOT ALL HYPERPARAMETER RESULTS_TRAINABLE MODELS**

In [ ]:
reset_matplotlib_defaults()
results_hyper_LSTM = hyperparameter_analysis(df_hyper_LSTM, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
results_hyper_transformer = hyperparameter_analysis(df_hyper_transformer, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
results_hyper_tft = hyperparameter_analysis(df_hyper_tft, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

# Group them ALL together for plotting transformer and tft in the same plots
results_combined = pd.concat([results_hyper_LSTM,   results_hyper_transformer, results_hyper_tft], ignore_index=True)


metrics_to_plot = ["mse", "sbd", "dstsp", "pe"]
metrics_to_plot = ["mse",  "dstsp"]
horizons = results_combined["horizon"].unique()
models = results_combined["model"].unique()

reset_matplotlib_defaults()

for metric in metrics_to_plot:
    # fig, axes = plt.subplots(1, len(horizons), figsize=(PLOT_SIZES[0] * len(horizons), PLOT_SIZES[1]), sharey=True)

    for  hz in zip(horizons):

        fig, ax = plt.subplots(figsize=(PLOT_SIZES[0], PLOT_SIZES[1])) 
        df_hz = results_combined[results_combined["horizon"] == hz]
        for model in models:
            df_m = df_hz[df_hz["model"] == model].sort_values("SL")
            ax.plot(df_m["SL"], df_m[metric], marker="o", label=model)

        ax.set_title(f"Forecast Horizon: {hz[0]}")
        ax.set_xlabel("Context Window")
        ax.set_ylabel(metric.upper())



        if hz[0] == 10:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10[metric])
        else:
            ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40[metric])
        ax.legend()

        if metric == "mse" and hz[0] == 10:

            ax.set_yscale('log')
            ax.autoscale(enable=False)
            ax.set_ylim(bottom=1e-4, top=1.3)

        plt.tight_layout()

        save_path = os.path.join(plots_dir, f"RNNSTRANS_{metric}_horizon_{hz[0]}.pdf")
        plt.savefig(save_path, bbox_inches="tight", format="pdf")
        plt.show()

### **UPOs**

#### **Create UPOs for DTs (Run Once, results are saved)**

In [ ]:
RHO_RANGE = [28, 50, 100, 150, 175, 210]

RHO_RANGE = [50]

DATA_DIR = rf"data_upos_original/"
OUTPUT_DIR = "UPOS"

os.makedirs(OUTPUT_DIR, exist_ok=True)

for custom_dt in UPO_DT_RANGE:
    custom_dt_rounded = round(custom_dt, 3)
    for RHO in RHO_RANGE:
        print(f"Processing rho {RHO}... for dt {custom_dt_rounded}")
        DATA_DIR_RHO = os.path.join(DATA_DIR, f"DATA1_{RHO}")
        os.makedirs(DATA_DIR_RHO, exist_ok=True)

        final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{custom_dt_rounded}.pkl")
        df_orbits_individual = load_all_orbits_parallel(DATA_DIR_RHO, rho=RHO, dt=custom_dt_rounded)
        df_orbits_individual = df_orbits_individual.sort_values("K")
        df_orbits_individual.to_pickle(final_filename)

        print(f"Saved orbits for rho {RHO} to {final_filename} and dt {custom_dt_rounded}\n")

In [ ]:
# explicit key initialization for better readability, not strictly necessary

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]
RHO_TO_LOAD = [50]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

upo_data_multi_rho_hyper = defaultdict(dict)
for RHO in RHO_TO_LOAD:
    for custom_dt in UPO_DT_RANGE:
        custom_dt_rounded = round(custom_dt, 3)
        final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{custom_dt_rounded}.pkl")
        df_orbits_individual = pd.read_pickle(final_filename)
        df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
        print(f"Loaded DataFrame from {final_filename}")
        upo_data_multi_rho_hyper[RHO][custom_dt_rounded] = df_orbits_individual

#### **Run UPOs**

In [ ]:

from collections import defaultdict
# explicit key initialization for better readability, not strictly necessary

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]
RHO_TO_LOAD = [28]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)

upo_data_multi_rho_hyper = defaultdict(dict)
for RHO in RHO_TO_LOAD:
    for custom_dt in UPO_DT_RANGE:
        custom_dt_rounded = round(custom_dt, 3)
        final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{custom_dt_rounded}.pkl")
        df_orbits_individual = pd.read_pickle(final_filename)
        df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
        print(f"Loaded DataFrame from {final_filename}")
        upo_data_multi_rho_hyper[RHO][custom_dt_rounded] = df_orbits_individual


rho_vector = [28]
results = []
scaler = StandardScaler()

for rho in rho_vector:
    
    test_mask = np.isin(test_o[:, 6], rho)
    test_data = test_o[test_mask]
    obs_xyz = test_data[:, 3:6]

    for custom_dt in UPO_DT_RANGE:
        custom_dt_rounded = round(custom_dt, 3)
        # Load UPO data for the current rho and dt
        df_upo = upo_data_multi_rho_hyper[rho][custom_dt_rounded]
        tree, labels, df_upo_extended = build_labeled_tree(df_upo)

        for custom_k in UPO_K_RANGE:
            for horizon in FORECAST_HORIZONS_RANGE:
                print(f"Running experiment for {rho} with dt {custom_dt_rounded} and K {custom_k} for horizon {horizon}...")
                
                observed, _, predicted, _ = UPO_prediction_window(
                    obs_xyz,
                    window_size=horizon,
                    tree=tree,
                    labels=labels,
                    norm=2,
                    k=custom_k,
                    upo_df=df_upo_extended,
                    time_diff=int(DT / custom_dt_rounded)
                )

                scaler.fit(np.vstack([observed, predicted]))
                observed_scaled = scaler.transform(observed)
                predicted_scaled = scaler.transform(predicted)

                results.append({
                    "model": "UPO",
                    "rho": rho, 
                    "dt": custom_dt_rounded, 
                    "k": custom_k,
                    "horizon": horizon, 
                    "observed": observed, 
                    "predicted": predicted,
                    "observed_scaled": observed_scaled,
                    "predicted_scaled": predicted_scaled
                    })

                print(f"Finished experiment for {rho} with dt {custom_dt_rounded} and K {custom_k} for horizon {horizon}\n")    

df_upo_results_hyper = pd.DataFrame(results)
df_upo_results_hyper = df_upo_results_hyper.sort_values(["model", "rho", "dt", "k", "horizon"]).reset_index(drop=True)

# Change column tau to horizon if it exists for better consistency
if "tau" in df_upo_results_hyper.columns:
    df_upo_results_hyper.rename(columns={"tau": "horizon"}, inplace=True)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "upo_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_upo_results_hyper.to_pickle(FULL_PATH)


results_upo_hyper = hyperparameter_analysis_upo(df_upo_results_hyper, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "results_upo_hyper_" + NAME + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

results_upo_hyper.to_pickle(FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)

file_name = "upo_vs_hyper_results_" + NAME + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
df_upo_results_hyper = pd.read_pickle(FULL_PATH)


file_name = "results_upo_hyper_" + NAME + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
results_upo_hyper = pd.read_pickle(FULL_PATH)

In [ ]:
reset_matplotlib_defaults()

metrics_to_plot = ["mse", "dstsp"]
horizons = results_upo_hyper["horizon"].unique()
dt_values = results_upo_hyper["dt"].unique()

dt_values_to_plot = [0.001,0.003, 0.005, 0.01, 0.03, 0.05]


for metric in metrics_to_plot:
    for hz in horizons:
        fig, ax = plt.subplots(figsize=(PLOT_SIZES[0], PLOT_SIZES[1]))
        df_hz = results_upo_hyper[results_upo_hyper["horizon"] == hz]

        for dt in dt_values_to_plot:
            df_dt = df_hz[df_hz["dt"] == dt].sort_values("k")
            ax.plot(df_dt["k"], df_dt[metric], label=f"dt={dt}")

        ax.set_title(f"Forecast Horizon: {hz}")
        ax.set_xlabel("Number of K UPOs")
        ax.set_ylabel(metric.upper())
        if metric != "mse":
            if hz == 10:
                ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10_UPO[metric])
            else:
                ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40_UPO[metric])
        if metric == "mse":
            ax.set_yscale("log")
        else:            
            ax.set_yscale("linear")
        # if hz == 10:
        #     ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10[metric])
        # else:
        #     ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40[metric])
        ax.legend(ncols = 2)
        plt.tight_layout()
        save_path = os.path.join(plots_dir, f"upo_{metric}_horizon_{hz}.pdf")
        plt.savefig(save_path, bbox_inches="tight", format="pdf")
        plt.show()


#### **UPO Predictor Evaluation on Number of UPOs utilized for context**

In [ ]:
#### NOOOOOOOOOOOOOOOOOOOOOOO!

from collections import defaultdict
# explicit key initialization for better readability, not strictly necessary

RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]
RHO_TO_LOAD = [28]

DT_TESTS_FOR_NR_OF_UPOS = 0.001
NUMBER_OF_UPOS_TO_TEST = range(0, 1350, 10)  # from 1 to 1350 with step of 1
K_UPOS = 1
FORECAST_HORIZONS_RANGE = [10]
HYPER_RHO = 28
EXPERIMENT_REPETITIONS = 10

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)



upo_data_multi_rho_hyper = defaultdict(dict)
for RHO in RHO_TO_LOAD:
    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_TESTS_FOR_NR_OF_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho_hyper[RHO][DT_TESTS_FOR_NR_OF_UPOS] = df_orbits_individual


rho_vector = [HYPER_RHO]

scaler = StandardScaler()
results = []

for repetition in range(EXPERIMENT_REPETITIONS):

    for rho in rho_vector:

        test_mask = np.isin(test_o[:, 6], rho)
        test_data = test_o[test_mask]
        obs_xyz = test_data[:, 3:6]

        for nr_upos in NUMBER_OF_UPOS_TO_TEST:
            if nr_upos == 0:
                nr_upos = 1  

            # Load UPO data for the current rho and dt
            df_upo = upo_data_multi_rho_hyper[rho][DT_TESTS_FOR_NR_OF_UPOS]

            # Magic stuff
            # Randomly select number of UPOs to use for the prediction, and build the tree and labels for that subset
            df_upos_samnpled = df_upo.sample(nr_upos)

            # re-index orbit it from 1 to nr_upos and drop index and set ORBIT ID as index
            df_upos_samnpled = df_upos_samnpled.reset_index(drop=True)
            df_upos_samnpled["Orbit_ID"] = df_upos_samnpled.index + 1
            df_upos_samnpled = df_upos_samnpled.set_index("Orbit_ID")
        

            tree, labels, df_upo_extended = build_labeled_tree(df_upos_samnpled)
            
            for horizon in FORECAST_HORIZONS_RANGE:
                print(f"Running experiment for {rho} with dt {DT_TESTS_FOR_NR_OF_UPOS}, K =  {K_UPOS}, and {nr_upos} UPOs for horizon {horizon}...")
                
                observed, _, predicted, _ = UPO_prediction_window(
                    obs_xyz,
                    window_size=horizon,
                    tree=tree,
                    labels=labels,
                    norm=2,
                    k=K_UPOS,
                    upo_df=df_upo_extended,
                    time_diff=int(DT / DT_TESTS_FOR_NR_OF_UPOS)
                )

                scaler.fit(np.vstack([observed, predicted]))
                observed_scaled = scaler.transform(observed)
                predicted_scaled = scaler.transform(predicted)

                results.append({
                    "repetition": repetition,
                    "model": "UPO",
                    "rho": rho, 
                    "dt": DT_TESTS_FOR_NR_OF_UPOS, 
                    "k": K_UPOS,
                    "nr_upos": nr_upos,
                    "horizon": horizon, 
                    "observed": observed, 
                    "predicted": predicted,
                    "observed_scaled": observed_scaled,
                    "predicted_scaled": predicted_scaled
                    })

                print(f"Finished experiment for {rho} with dt {DT_TESTS_FOR_NR_OF_UPOS}, K =  {K_UPOS}, and {nr_upos} UPOs for horizon {horizon}\n")    

df_results_upo_context = pd.DataFrame(results)
df_results_upo_context = df_results_upo_context.sort_values(["model", "rho", "dt", "k", "horizon"]).reset_index(drop=True)

# Change column tau to horizon if it exists for better consistency
if "tau" in df_results_upo_context.columns:
    df_results_upo_context.rename(columns={"tau": "horizon"}, inplace=True)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "upo_vs_hyper_results_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_results_upo_context.to_pickle(FULL_PATH)


results_upo_hyper = hyperparameter_analysis_upo(df_results_upo_context, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "results_upo_hyper_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_results_upo_context.to_pickle(FULL_PATH)

In [ ]:
from joblib import Parallel, delayed

def run_single_repetition(repetition, rho_vector, test_o, upo_data_multi_rho_hyper,
                           NUMBER_OF_UPOS_TO_TEST, FORECAST_HORIZONS_RANGE,
                           K_UPOS, DT_TESTS_FOR_NR_OF_UPOS, DT,
                           SIMULATION_LEN, BINS, SMOOTHING, MASE_STEPS):
    scaler = StandardScaler()  # local to each worker
    rep_results = []

    for rho in rho_vector:
        test_mask = np.isin(test_o[:, 6], rho)
        test_data = test_o[test_mask]
        obs_xyz = test_data[:, 3:6]


        for nr_upos in NUMBER_OF_UPOS_TO_TEST:
            if nr_upos == 0:
                nr_upos = 1
            print(f"Repetition {repetition}: Running experiment for {rho} with dt {DT_TESTS_FOR_NR_OF_UPOS}, K =  {K_UPOS}, and {nr_upos} UPOs for horizon {FORECAST_HORIZONS_RANGE}...")
            for k_upos_test in K_UPOS_TO_TEST:
                K_UPOS = k_upos_test
                df_upo = upo_data_multi_rho_hyper[rho][DT_TESTS_FOR_NR_OF_UPOS]
                df_upos_samnpled = df_upo.sample(nr_upos)
                df_upos_samnpled = df_upos_samnpled.reset_index(drop=True)
                df_upos_samnpled["Orbit_ID"] = df_upos_samnpled.index + 1
                df_upos_samnpled = df_upos_samnpled.set_index("Orbit_ID")

                tree, labels, df_upo_extended = build_labeled_tree(df_upos_samnpled)

                for horizon in FORECAST_HORIZONS_RANGE:
                    observed, _, predicted, _ = UPO_prediction_window(
                        obs_xyz,
                        window_size=horizon,
                        tree=tree,
                        labels=labels,
                        norm=2,
                        k=K_UPOS,
                        upo_df=df_upo_extended,
                        time_diff=int(DT / DT_TESTS_FOR_NR_OF_UPOS)
                    )

                    scaler.fit(np.vstack([observed, predicted]))
                    observed_scaled = scaler.transform(observed)
                    predicted_scaled = scaler.transform(predicted)

                    rep_results.append({
                        "repetition": repetition,
                        "model": "UPO",
                        "rho": rho,
                        "dt": DT_TESTS_FOR_NR_OF_UPOS,
                        "k": K_UPOS,
                        "nr_upos": nr_upos,
                        "horizon": horizon,
                        "observed": observed,
                        "predicted": predicted,
                        "observed_scaled": observed_scaled,
                        "predicted_scaled": predicted_scaled
                    })
                    print(f"Repetition {repetition}: Finished experiment for {rho} with dt {DT_TESTS_FOR_NR_OF_UPOS}, K =  {K_UPOS}, and {nr_upos} UPOs for horizon {horizon}\n")

    return rep_results


RHO_TO_LOAD = [28, 50, 100, 150, 175, 210]
RHO_TO_LOAD = [28]

DT_TESTS_FOR_NR_OF_UPOS = 0.001
NUMBER_OF_UPOS_TO_TEST = range(0, 1350, 20)  # from 1 to 1350 with step of 1
K_UPOS_TO_TEST = [1, 10, 30, 50, 100]  # different K values to test

FORECAST_HORIZONS_RANGE = [10, 40, 100]
HYPER_RHO = 28
EXPERIMENT_REPETITIONS = 10

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)



upo_data_multi_rho_hyper = defaultdict(dict)
for RHO in RHO_TO_LOAD:
    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_TESTS_FOR_NR_OF_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho_hyper[RHO][DT_TESTS_FOR_NR_OF_UPOS] = df_orbits_individual


rho_vector = [HYPER_RHO]

scaler = StandardScaler()
results = []


all_results = Parallel(n_jobs=-1, verbose=10)(
    delayed(run_single_repetition)(
        repetition, rho_vector, test_o, upo_data_multi_rho_hyper,
        NUMBER_OF_UPOS_TO_TEST, FORECAST_HORIZONS_RANGE,
        K_UPOS_TO_TEST, DT_TESTS_FOR_NR_OF_UPOS, DT,
        SIMULATION_LEN, BINS, SMOOTHING, MASE_STEPS
    )
    for repetition in range(EXPERIMENT_REPETITIONS)
)

# Flatten list of lists
results = [item for rep in all_results for item in rep]

df_results_upo_context = pd.DataFrame(results)
df_results_upo_context = df_results_upo_context.sort_values(["model", "rho", "dt", "k", "horizon"]).reset_index(drop=True)

# Change column tau to horizon if it exists for better consistency
if "tau" in df_results_upo_context.columns:
    df_results_upo_context.rename(columns={"tau": "horizon"}, inplace=True)

# FODLER_or_DIR = "outer_limits_results/hyper_results"
# os.makedirs(FODLER_or_DIR, exist_ok=True)
# file_name = "upo_vs_hyper_results_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
# FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

# df_results_upo_context.to_pickle(FULL_PATH)


results_upo_hyper = hyperparameter_analysis_upo(df_results_upo_context, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

# drop observed, predicted, observed_scaled and predicted_scaled columns from results_upo_hyper to save space, as they are not needed for plotting
results_upo_hyper = results_upo_hyper.drop(columns=["observed", "predicted", "observed_scaled", "predicted_scaled"], errors='ignore')

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "results_upo_hyper_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

results_upo_hyper.to_pickle(FULL_PATH)

In [ ]:
# Flatten list of lists
results = [item for rep in all_results for item in rep]

df_results_upo_context = pd.DataFrame(results)
df_results_upo_context = df_results_upo_context.sort_values(["model", "rho", "dt", "k", "horizon"]).reset_index(drop=True)

# Change column tau to horizon if it exists for better consistency
if "tau" in df_results_upo_context.columns:
    df_results_upo_context.rename(columns={"tau": "horizon"}, inplace=True)

# FODLER_or_DIR = "outer_limits_results/hyper_results"
# os.makedirs(FODLER_or_DIR, exist_ok=True)
# file_name = "upo_vs_hyper_results_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
# FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

# df_results_upo_context.to_pickle(FULL_PATH)


results_upo_hyper = hyperparameter_analysis_upo(df_results_upo_context, simulation_len=SIMULATION_LEN, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

# drop observed, predicted, observed_scaled and predicted_scaled columns from results_upo_hyper to save space, as they are not needed for plotting
results_upo_hyper = results_upo_hyper.drop(columns=["observed", "predicted", "observed_scaled", "predicted_scaled"], errors='ignore')

FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
file_name = "results_upo_hyper_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

results_upo_hyper.to_pickle(FULL_PATH)

In [ ]:
FODLER_or_DIR = "outer_limits_results/hyper_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
EXPERIMENT_REPETITIONS = 10


file_name = "results_upo_hyper_number_of_context_upos" + NAME + "_"  + str(EXPERIMENT_REPETITIONS) + "_" + str(HYPER_RHO) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
results_upo_hyper = pd.read_pickle(FULL_PATH)

metrics_for_experiments = ["mse", "sbd", "dstsp", "pe"]
# Average over repetitions for metrics above

results_upo_hyper_avg = results_upo_hyper.groupby(["model", "rho", "dt", "k", "nr_upos", "horizon"])[metrics_for_experiments].mean().reset_index()

In [ ]:
reset_matplotlib_defaults()

# Plot the results for the number of context UPOs experiment

metrics_to_plot = ["mse", "dstsp"]
horizons = results_upo_hyper_avg["horizon"].unique()
context_upos = results_upo_hyper_avg["nr_upos"].unique()
number_of_k_upos_shadowed = results_upo_hyper_avg["k"].unique()

for metric in metrics_to_plot:
    for hz in horizons:

        fig, ax = plt.subplots(figsize=(PLOT_SIZES[0], PLOT_SIZES[1]))

        for k in number_of_k_upos_shadowed:

            df_hz_k = results_upo_hyper_avg[(results_upo_hyper_avg["horizon"] == hz) & (results_upo_hyper_avg["k"] == k)]
            ax.plot(df_hz_k["nr_upos"][5:], df_hz_k[metric][5:], label=f"k={k}")
        
        ax.set_title(f"Forecast Horizon: {hz}")
        ax.set_xlabel("Number of Context UPOs")
        ax.set_ylabel(metric.upper())
        ax.set_xticks(range(0, 1350, 200))  # Show every 5th tick for better readability
        # if metric != "mse":
        #     if hz == 10:
        #         ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10_UPO[metric])
        #     else:
        #         ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40_UPO[metric])
        # ax.set_yscale("log")
        # if hz == 10:
        #     ax.set_ylim(PLOT_RANGES_METRICS_HYPER_10[metric])
        # else:
        #     ax.set_ylim(PLOT_RANGES_METRICS_HYPER_40[metric])
        ax.legend(ncols = 2)
        plt.tight_layout()
        save_path = os.path.join(plots_dir, f"upos_context_nr_upos_{metric}_horizon_{hz}.pdf")
        plt.savefig(save_path, bbox_inches="tight", format="pdf")
        plt.show()

## **Comparissons All Models**

In [ ]:
BINS = 5
SMOOTHING = 10
MASE_STEPS = 10

In [ ]:
FODLER_or_DIR = "outer_limits_results/tau_results"
os.makedirs(FODLER_or_DIR, exist_ok=True)
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

file_name = "_outer_region_performance_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_LSTM = pd.read_pickle(FULL_PATH)

file_name = "_Transformer_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_transformer = pd.read_pickle(FULL_PATH)

file_name = "_TFT_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO")+f"_scale_rho_{SCALE_PER_RHO}"  + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_tft = pd.read_pickle(FULL_PATH)

file_name = "_pretrained_vs_forecast_horizon_" +  NAME + "_" + str("ALL_RHO")+ "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_pretrained = pd.read_pickle(FULL_PATH)

file_name = "UPO_vs_forecast_horizon_" + NAME + "_" + str("ALL_RHO") + "_" + str(horizons_experiments[-1]) + rf"{EXPERIMENT_NAME}" + ".pkl"
FULL_PATH = os.path.join(FODLER_or_DIR, file_name)

df_UPO_results = pd.read_pickle(FULL_PATH)

In [ ]:
# Get all data for each model, for rho 28, for all horizons, and concatenate them together for plotting
# LSTM DF has two models and pretrained has 4...
# Pretrained = {"DynaMix", "Chronos2", "TimeSFM", "Panda"}
# LSTM = {"LSTM-A", "LSTM-B"}

# For all models keep columns: "model", "rho", "horizon", "observed", "predicted"
# For LSTM and transformers: rename "forecast_horizons" to "horizon" for consistency with pretrained and UPO
# For UPO rename "tau" to "horizon" for consistency with the others
# FOR LSTM rename training_rhos to rho for consistency with the others

# LSTM
df_LSTM = df_LSTM.rename(columns={
    "train_rhos": "rho",
    "forecast_horizons": "horizon"
})

# Transformer + TFT
df_transformer = df_transformer.rename(columns={
    "forecast_horizons": "horizon",
    "train_rhos": "rho"
})

df_tft = df_tft.rename(columns={
    "forecast_horizons": "horizon",
    "train_rhos": "rho"
})

# UPO
df_UPO_results = df_UPO_results.rename(columns={
    "tau": "horizon"
})

RHO_TO_LOAD = 28
horizons_experiments = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]

df_LSTM_RHO = df_LSTM[df_LSTM["rho"] == RHO_TO_LOAD]
df_transformer_RHO = df_transformer[df_transformer["rho"] == RHO_TO_LOAD]
df_tft_RHO = df_tft[df_tft["rho"] == RHO_TO_LOAD]

df_pretrained_RHO = df_pretrained[df_pretrained["rho"] == RHO_TO_LOAD]
df_UPO_RHO = df_UPO_results[df_UPO_results["rho"] == RHO_TO_LOAD]

# LSTM variants
df_LSTM_A_RHO = df_LSTM_RHO[df_LSTM_RHO["TEACHER_FORCING"] == False]
df_LSTM_B_RHO = df_LSTM_RHO[df_LSTM_RHO["TEACHER_FORCING"] == True]
# Add model column to each and move it to the front for better consistency with the other DFs

df_LSTM_A_RHO["model"] = "LSTM-A"
df_LSTM_B_RHO["model"] = "LSTM-B"

# Add observed and predicted columns to each
df_LSTM_A_RHO["observed"] = df_LSTM_A_RHO["dfs"].apply(lambda x: x['obs'])
df_LSTM_A_RHO["predicted"] = df_LSTM_A_RHO["dfs"].apply(lambda x: x['pred'])
df_LSTM_B_RHO["observed"] = df_LSTM_B_RHO["dfs"].apply(lambda x: x['obs'])
df_LSTM_B_RHO["predicted"] = df_LSTM_B_RHO["dfs"].apply(lambda x: x['pred'])

# Pretrained splits
df_dynamix_RHO = df_pretrained_RHO[df_pretrained_RHO["model"] == "DynaMix"]
df_chronos_RHO = df_pretrained_RHO[df_pretrained_RHO["model"] == "Chronos2"]
df_timesfm_RHO = df_pretrained_RHO[df_pretrained_RHO["model"] == "TimeSFM"]
df_panda_RHO = df_pretrained_RHO[df_pretrained_RHO["model"] == "Panda"]

df_LSTM_A_RHO_all_horizons = df_LSTM_A_RHO[df_LSTM_A_RHO["horizon"].isin(horizons_experiments)]
df_LSTM_B_RHO_all_horizons = df_LSTM_B_RHO[df_LSTM_B_RHO["horizon"].isin(horizons_experiments)]

df_transformer_RHO_all_horizons = df_transformer_RHO[df_transformer_RHO["horizon"].isin(horizons_experiments)]
df_tft_RHO_all_horizons = df_tft_RHO[df_tft_RHO["horizon"].isin(horizons_experiments)]

df_dynamix_RHO_all_horizons = df_dynamix_RHO[df_dynamix_RHO["horizon"].isin(horizons_experiments)]
df_chronos_RHO_all_horizons = df_chronos_RHO[df_chronos_RHO["horizon"].isin(horizons_experiments)]
df_timesfm_RHO_all_horizons = df_timesfm_RHO[df_timesfm_RHO["horizon"].isin(horizons_experiments)]
df_panda_RHO_all_horizons = df_panda_RHO[df_panda_RHO["horizon"].isin(horizons_experiments)]

df_UPO_RHO_all_horizons = df_UPO_RHO[df_UPO_RHO["horizon"].isin(horizons_experiments)]
# Add model name to UPO
df_UPO_RHO_all_horizons["model"] = "UPO"

df_LSTM_A_RHO_all_horizons = extend_with_metrics(df_LSTM_A_RHO_all_horizons, "LSTM-A", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
df_LSTM_B_RHO_all_horizons = extend_with_metrics(df_LSTM_B_RHO_all_horizons, "LSTM-B", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

df_transformer_RHO_all_horizons = extend_with_metrics(df_transformer_RHO_all_horizons, "Transformer", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS) 
df_tft_RHO_all_horizons = extend_with_metrics(df_tft_RHO_all_horizons, "TFT", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

df_dynamix_RHO_all_horizons = extend_with_metrics(df_dynamix_RHO_all_horizons, "DynaMix", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
df_chronos_RHO_all_horizons = extend_with_metrics(df_chronos_RHO_all_horizons, "Chronos2", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
df_timesfm_RHO_all_horizons = extend_with_metrics(df_timesfm_RHO_all_horizons, "TimeSFM", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
df_panda_RHO_all_horizons = extend_with_metrics(df_panda_RHO_all_horizons, "Panda", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)

df_UPO_RHO_all_horizons = extend_with_metrics(df_UPO_RHO_all_horizons, "UPO", sequence_length=512, bins=BINS, smoothing=SMOOTHING, mase_steps=MASE_STEPS)
#
cols_to_keep = ["model", "rho", "horizon", "observed", "predicted", "mse", "sbd", "dstsp"]

dfs = [
    df_LSTM_A_RHO_all_horizons,
    df_LSTM_B_RHO_all_horizons,
    df_transformer_RHO_all_horizons,
    df_tft_RHO_all_horizons,
    df_dynamix_RHO_all_horizons,
    df_chronos_RHO_all_horizons,
    df_timesfm_RHO_all_horizons,
    df_panda_RHO_all_horizons,
    df_UPO_RHO_all_horizons
]

dfs = [df[cols_to_keep] for df in dfs]

df_combined = pd.concat(dfs, ignore_index=True)

In [ ]:
# PLOT ALL MODELS TOGETHER FOR RHO 28, FOR ALL HORIZONS, WITH AN INSET ZOOMING IN ON THE FIRST 50 POINTS
from mpl_toolkits.axes_grid1.inset_locator import inset_axes, mark_inset

df_combined = df_combined.sort_values(["model", "horizon"]).reset_index(drop=True)
metrics_to_plot = ["mse", "sbd", "dstsp"]
horizons = df_combined["horizon"].unique()
models = df_combined["model"].unique()
reset_matplotlib_defaults()

PLOT_SIZES_2 = (10, 7)
POINTS_TO_PLOT_MINI_AXES = 50
LINE_WIDTH =2.6
LINE_STYLE = "-"


plt.rcParams.update(plt.rcParamsDefault)

# Then apply your custom settings on top
font_main = 21
font_title = 21
font_legend = 21
linewidth = 4

plt.rcParams.update({
    'font.size': font_main,
    'axes.labelsize': font_main,
    'xtick.labelsize': font_main,
    'ytick.labelsize': font_main,
    'legend.fontsize': font_legend,
    'axes.titlesize': font_title,
    'figure.titlesize': font_main,
    'axes.grid': True,
    'axes.grid.which': 'major',
    'axes.linewidth': 0.8,
    'lines.linewidth': linewidth
})

plt.rcParams['axes.prop_cycle'] = plt.cycler(color=plt.rcParamsDefault['axes.prop_cycle'].by_key()['color'])

COLOR_LIST = [
    "#029985",   #
    "#954A11",   # 
    "#E3AC20",   # 
    "#0909BB",   # 
    "#944FF4",   # 
    "#EA510A",   # 
    "#46E60C",   # 
    "#4A8AF0",   # 
    "#DD17BF",   # 
]
HIGHLIGHT_COLOR = (0.27, 0.51, 0.71, 0.12)
# ──────────────────────────────────────────────────────────────────────────────

MODEL_COLORS = {model: COLOR_LIST[i] for i, model in enumerate(models)}

for metric in metrics_to_plot:
    fig, ax = plt.subplots(figsize=(PLOT_SIZES_2[0], PLOT_SIZES_2[1]))
    for i, model in enumerate(models):
        df_m = df_combined[df_combined["model"] == model].sort_values("horizon")
        model_name = model
        if model_name == "UPO":
            model_name = "UPO Predictor"
        if model_name == "TimeSFM":
            model_name = "TimesFM"
        ax.plot(df_m["horizon"], df_m[metric], label=model_name, linewidth=LINE_WIDTH,
                linestyle=LINE_STYLE, color=MODEL_COLORS[model])

    ax.set_xlabel(rf"Forecast Horizon for $\rho={RHO_TO_LOAD}$")
    ax.set_ylabel(metric.upper())
    ax.legend(ncol=3, loc='lower center', bbox_to_anchor=(0.5, 0.95), frameon=True, fontsize=19)

    if metric not in ["mse", "sbd"]:
        location = 'upper right'
    else:
        location = 'lower right'
    
    axins = inset_axes(ax, width="40%", height="40%", loc=location, borderpad=1.5)
    zoom_ymin, zoom_ymax = float("inf"), float("-inf")
    for model in models:
        df_m = df_combined[df_combined["model"] == model].sort_values("horizon")
        df_zoom = df_m[df_m["horizon"] <= POINTS_TO_PLOT_MINI_AXES]
        axins.plot(df_zoom["horizon"], df_zoom[metric], linewidth=LINE_WIDTH,
                   linestyle=LINE_STYLE, color=MODEL_COLORS[model])

    axins.tick_params(labelsize=14)
    axins.grid(True, alpha=0.3)
    axins.set_xlim(0, POINTS_TO_PLOT_MINI_AXES)
    axins.set_yscale("log")
    axins.set_ylim(4e-6, 2)
    axins.set_yticks([1e-6, 1e-4, 1e-2, 1])
    axins.patch.set_facecolor('white')
    axins.patch.set_zorder(3)

    pp, p1, p2 = mark_inset(ax, axins, loc1=2, loc2=3, fc=HIGHLIGHT_COLOR,
                             ec="#457B9D", ls="--", lw=0.8)
    pp.set_clip_on(True)
    for artist in (pp, p1, p2):
        artist.set_zorder(7)

    plt.tight_layout(rect=[0, 0, 1, 0.92])
    save_path = os.path.join("plots/", f"tau_results/combined_models_{metric}_rho_{RHO_TO_LOAD}.pdf")
    plt.savefig(save_path, bbox_inches="tight", format="pdf")
    plt.show()

## **Train All Models at Once**

### **Function Workers**

In [ ]:
def worker_train_LSTM(job):
    datasets_parameters, training_rho, train_data, all_data, horizon, scale_per_rho, models_parameters, teacher_forcing = job
    
    if datasets_parameters is None or training_rho is None \
        or horizon is None  or train_data is None or models_parameters is None:

        raise ValueError("Missing arguments")

    local_params = models_parameters.copy()

    if teacher_forcing:
        local_params['forecast_size'] = 1
        # local_params['seq_len'] = local_params['seq_len'] + horizon
    else:
        local_params['forecast_size'] = horizon
    
    model = SequenceModel(local_params, logging=False)    
    model.create_model(dataset_parameters=datasets_parameters[NAME])    
    model.create_scaler(train_data, train_data, scale_per_rho = scale_per_rho)

    print(f"Running Training: Train rho={training_rho}, forecast={horizon}")

    model.train_model(
        data=train_data,
        dataset_parameters=datasets_parameters[NAME],
        all_data=train_data,
        fit_scaler=False,
        scale_per_rho = scale_per_rho,
        verbose = 2
    )

    del train_data, all_data, local_params
    gc.collect()

    model_name = "LSTM_TF" if teacher_forcing else "LSTM"

    print(f"DONE Running Training: Train rho={training_rho}, forecast={horizon}")
    return {
        "model_name": model_name,
        "model": model,
        "training_rho": training_rho,
        "forecast_horizons": horizon,
    }

def worker_train_transformer(job):

    training_rho, train_data, test_data, transformer_parameters, horizon = job
    
    print(f"Running Transformer for horizons {horizon} and rho {training_rho}:")

    # Scale Data
    scaler = StandardScaler()
    scaler.fit(train_data[:, 3:6])
    
    train_data[:, 3:6] = scaler.transform(train_data[:, 3:6])

    # Create time series
    train_series = pd.DataFrame(train_data, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
    train_series = TimeSeries.from_dataframe(train_series[transformer_parameters["features"]].copy()).astype(np.float32)

    input_length = transformer_parameters["input_length"]

    model = TransformerModel(
        input_chunk_length=input_length,     # number of past time steps used as input
        output_chunk_length=horizon,   # how many future steps to predict
        d_model=transformer_parameters["model_dim"],                   # internal embedding dimension
        nhead=transformer_parameters["attention_heads"],               # number of attention heads
        num_encoder_layers=transformer_parameters["encoder_layers"],   # how many encoder layers
        num_decoder_layers=transformer_parameters["decoder_layers"],   # how many decoder layers
        batch_size=transformer_parameters["batch_size"],
        n_epochs=transformer_parameters["num_epochs"],
        likelihood=None,
        pl_trainer_kwargs=transformer_parameters["pl_trainer_kwargs"],
        optimizer_kwargs={'lr': transformer_parameters["learning_rate"]},
    )

    model.fit(train_series.copy() , verbose=True, stride = horizon)
    

    del train_data, test_data, train_series
    gc.collect()

    return {
        "model_name": "Transformer",
        "model": model,
        "training_rho": training_rho,
        "forecast_horizons": horizon,
    }

def worker_train_tft(job):
    training_rho, train_data, test_data, tft_parameters, horizon = job

    print(f"Running TFT for horizons {horizon} and rho {training_rho}:")
    # Scale Data
    scaler = StandardScaler()
    scaler.fit(train_data[:, 3:6])
    train_data[:, 3:6] = scaler.transform(train_data[:, 3:6])

    # Create time series
    train_series = pd.DataFrame(train_data, columns=['sim','sim', 'step','x', 'y', 'z', 'rho'])
    train_series = TimeSeries.from_dataframe(train_series[tft_parameters["features"]].copy()).astype(np.float32)

    input_length = tft_parameters["input_length"]

    model = TFTModel(
        input_chunk_length=input_length,
        output_chunk_length=horizon,
        hidden_size=tft_parameters["model_dim"],          # replaces d_model
        lstm_layers=2,                  # number of LSTM layers inside TFT
        num_attention_heads=tft_parameters["attention_heads"],  # replaces nhead
        batch_size=tft_parameters["batch_size"],
        n_epochs=tft_parameters["num_epochs"],
        likelihood=None,
        pl_trainer_kwargs=tft_parameters["pl_trainer_kwargs"],
        add_relative_index=True, 
        optimizer_kwargs={'lr': tft_parameters["learning_rate"]},

    )

    # print(model.summary())
    
    model.fit(train_series.copy() , verbose=True, stride = horizon)
    
    del train_series, train_data, test_data
    gc.collect()

    return {
        "model_name": "TFT",
        "model": model,
        "training_rho": training_rho,
        "forecast_horizons": horizon,
    }



### **LSTMs**

In [ ]:
# RUN FOR BOTH LSTM A and B, CHANGE PARAM AND RERUN TRAINING!

SCALE_PER_RHO = True
ATTENTION = False
TEACHER_FORCING = False

# |0: Simulation | 1: Sub-Sim (not used) | 2: Time | 3: X | 4: Y | 5: Z | 6: RHO |

MAX_SIMULATIONS_TRAIN = 3
MAX_SIMULATIONS_TEST = 3

DT = 0.05
EXPERIMENT_NAME = "dt005_all_rho_sim100"
EXPERIMENT_NAME = "dt005_sim10_all_rhos_500"

if DT == 0.01:
    forecast_horizons = [100]
    models_parameters['seq_len'] = 1280 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 256 #128 # 64
    TRANSIENT_LEN = 500
elif DT == 0.03:
    forecast_horizons = [40]
    models_parameters['seq_len'] = 512 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 128 #128 # 64
    TRANSIENT_LEN = 300
else:
    forecast_horizons = [20]
    models_parameters['seq_len'] = 512 #512 #512 #forecast_horizons[0] 256
    models_parameters['batch_size'] = 32 #128 # 64
    TRANSIENT_LEN = 200

models_parameters['model_type'] = 'LSTM'  # 'LSTM' 'GRU' 'RNN' 'MLP'
models_parameters['epochs'] = 300
models_parameters['hidden_size'] = [32, 16] #[64, 32] # [32], [64, 32], [64, 16]
models_parameters['stateful'] = True
models_parameters['LR'] = 0.01
models_parameters['rolling'] = True
models_parameters['loss'] = 'mse'
models_parameters['decay_rate'] = 0.95 #0.95
models_parameters['decay_step'] = 100
models_parameters['optimizer'] = 'Adam'
models_parameters['seed'] = 42
models_parameters['metrics'] = ['mae']
models_parameters['TF'] = False
models_parameters['MISO'] = False

if TEACHER_FORCING:
    models_parameters['dropout'] = 0.2
    models_parameters['recurrent_dropout'] = 0.2
    models_parameters['l2_reg'] = 1e-4
    models_parameters['clipnorm'] = 1.5
    models_parameters['rolling'] = False
else:
    models_parameters['dropout'] = 0.0
    models_parameters['recurrent_dropout'] = 0.0
    models_parameters['l2_reg'] = 0.0
    models_parameters['clipnorm'] = None
    models_parameters['rolling'] = True
    
# DATA COLUMNS:
# |0: Simulation | 1: Sub-Sim (not used) | 2: Time | 3: X | 4: Y | 5: Z | 6: RHO |
datasets_parameters['LORENZ']['outputs'] = [3, 4, 5]
datasets_parameters['LORENZ']['inputs'] = [3, 4, 5]

# BASED ON WHAT YOU SELECT, MAKE SURE THE CORRECT EXPERIMENT IS LOADED!
DT_STR = format(DT, 'g').replace('.', '')

res = DataHandler.load_exp_data_(EXPERIMENT_NAME, MAX_SIMULATIONS_TRAIN, MAX_SIMULATIONS_TEST, TRANSIENT_LEN, datasets_parameters, NAME, DT = DT)
SIMULATION_LEN, sim_step, SIMULATIONS, train_o, test_o, all_data = res

# For Delta Experiment
# training_rho_values = [ 50, 100, 150]
# forecast_horizons = [10, 20]

#For original experiments
forecast_horizons = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
training_rho_values = [28, 50, 100, 150, 175, 210]

jobs_train = []
results_train = []

for rho_value in training_rho_values:

    # Choose Desired Rho
    training_rho = rho_value
    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]
       
    for horizon in forecast_horizons:
        jobs_train.append((datasets_parameters, training_rho, train_data.copy(), train_data.copy(), horizon, SCALE_PER_RHO, models_parameters.copy(), TEACHER_FORCING))

with ThreadPoolExecutor(max_workers=32) as ex:
    for r in ex.map(worker_train_LSTM, jobs_train):
        results_train.append(r)

FODLER_or_DIR = "SAVED_MODELS/LSTMS"
os.makedirs(FODLER_or_DIR, exist_ok=True)

for trained_model in results_train:
    forecast_length = trained_model['forecast_horizons']
    model_to_save = trained_model['model']
    training_rho = trained_model['training_rho']

    file_name = trained_model["model_name"] + "_" + str(forecast_length) + "_" + NAME + "_" + str(training_rho)+f"_scale_rho_{SCALE_PER_RHO}" + "_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pkl"
    FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
    
    save_models_to_file(model_to_save, FULL_PATH)


In [ ]:
FODLER_or_DIR = "SAVED_MODELS/LSTMS"
os.makedirs(FODLER_or_DIR, exist_ok=True)

for trained_model in results_train:
    forecast_length = trained_model['forecast_horizons']
    model_to_save = trained_model['model']
    training_rho = trained_model['training_rho']

    file_name = trained_model["model_name"] + "_" + str(forecast_length) + "_" + NAME + "_" + str(training_rho)+f"_scale_rho_{SCALE_PER_RHO}" + "_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pkl"
    FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
    
    save_models_to_file(model_to_save, FULL_PATH)


### **Transformers and TFT**

In [ ]:


MAX_SIMULATIONS_TRAIN = 3
MAX_SIMULATIONS_TEST = 3

DT = 0.05
EXPERIMENT_NAME = "dt005_sim10_all_rhos_500"

if DT == 0.01:
    TRANSIENT_LEN = 500
elif DT == 0.03:
    TRANSIENT_LEN = 300
else:
    TRANSIENT_LEN = 200

# BASED ON WHAT YOU SELECT, MAKE SURE THE CORRECT EXPERIMENT IS LOADED!
DT_STR = format(DT, 'g').replace('.', '')

res = DataHandler.load_exp_data_(EXPERIMENT_NAME, MAX_SIMULATIONS_TRAIN, MAX_SIMULATIONS_TEST, TRANSIENT_LEN, datasets_parameters, NAME, DT = DT)
SIMULATION_LEN, sim_step, SIMULATIONS, train_o, test_o, all_data = res

# For Delta Experiment
training_rho_values = [ 50, 100, 150]
forecast_horizons = [10, 20]

# # For original experiments
forecast_horizons = [1, 5, 10, 20, 30, 40, 50, 100, 200, 500]
training_rho_values = [28, 50, 100, 150, 175, 210]


tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": "gpu",
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}


jobs_train = []
results_train = []

for rho_value in training_rho_values:

    # Choose Desired Rho
    training_rho = rho_value

    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]

    for horizon in forecast_horizons:
        jobs_train.append((training_rho, train_data.copy(), train_data.copy(), transformer_parameters.copy(), horizon))

with ThreadPoolExecutor(max_workers=1) as ex:
    for r in ex.map(worker_train_transformer, jobs_train):
        results_train.append(r)

FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
os.makedirs(FODLER_or_DIR, exist_ok=True)

for trained_model in results_train:
    forecast_length = trained_model['forecast_horizons']
    model_to_save = trained_model['model']
    training_rho = trained_model['training_rho']

    file_name = trained_model["model_name"] + "_" + str(forecast_length) + "_" + NAME + "_" + str(training_rho)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"
    FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
    
    model_to_save.save(FULL_PATH)

jobs_train = []
results_train = []

for rho_value in training_rho_values:

    # Choose Desired Rho
    training_rho = rho_value
    train_mask = np.isin(train_o[:, 6], training_rho)
    train_data = train_o[train_mask]


    for horizon in forecast_horizons:
        jobs_train.append((training_rho, train_data.copy(), train_data.copy(), transformer_parameters.copy(), horizon))

with ThreadPoolExecutor(max_workers=1) as ex:
    for r in ex.map(worker_train_tft, jobs_train):
        results_train.append(r)

FODLER_or_DIR = "SAVED_MODELS/TRANSFORMERS"
os.makedirs(FODLER_or_DIR, exist_ok=True)

for trained_model in results_train:
    forecast_length = trained_model['forecast_horizons']
    model_to_save = trained_model['model']
    training_rho = trained_model['training_rho']

    file_name = trained_model["model_name"] + "_" + str(forecast_length) + "_" + NAME + "_" + str(training_rho)+"_simulations_" + str(MAX_SIMULATIONS_TRAIN) + "_" + rf"{EXPERIMENT_NAME}" + ".pt"
    FULL_PATH = os.path.join(FODLER_or_DIR, file_name)
    
    model_to_save.save(FULL_PATH)


## **Training/Testing Times CPU-GPU**


### **LSTM A/B**

In [ ]:
# TF GEN
SCALE_PER_RHO = True
MAHALANOBIS = False

ATTENTION = False
TEACHER_FORCING = False

if TEACHER_FORCING:
    models_parameters['rolling'] = False
else:
    models_parameters['rolling'] = True

models_parameters['seq_len'] = 512
models_parameters['batch_size'] = 32


horizons_experiments = [1, 100]  
rho_value = 28

train_mask = np.isin(train_o[:, 6], rho_value)
test_mask = np.isin(test_o[:, 6], rho_value)

train_data = train_o[train_mask]
test_data = test_o[test_mask]

maximum_test_size = 30_000

train_data = train_data[:maximum_test_size]
test_data = test_data[:maximum_test_size]

all_data = np.vstack((train_data, test_data))

for TFGEN in [False]:
    
    for tau in horizons_experiments:

        if tau == 1 or TFGEN:
            models_parameters['rolling'] = False
        else:
            models_parameters['rolling'] = True

        models_parameters['forecast_size'] = tau

        dfs = train_test_single_model_thread_hyperparameters(
            datasets_parameters = datasets_parameters,
            train_rhos = [rho_value],
            test_rhos = [rho_value],
            train_data = train_data,
            test_data = test_data,
            all_data = all_data,
            forecast_horizons = [tau],
            scale_per_rho = SCALE_PER_RHO,
            models_parameters = models_parameters,
            TEACHER_FORCING = TFGEN,
            model = None,
            measure_time = True
        )



### **Transformer**

In [ ]:
# Multi Experiment Transformer for ____ dataset



transformer_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": device,
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

rho_value = 28
horizons_experiments = [1, 100]

train_mask = np.isin(train_o[:, 6], rho_value)
test_mask  = np.isin(test_o[:, 6], rho_value)

train_data = train_o[train_mask]
test_data  = test_o[test_mask]

maximum_test_size = 30_000

train_data = train_data[:maximum_test_size]
test_data = test_data[:maximum_test_size]

for tau in horizons_experiments:

    run_transformer_pipeline(train_data, 
                            test_data, 
                            transformer_parameters, 
                            forecast_horizons=tau, 
                            rho_value=rho_value, 
                            model=None,
                            measure_time=True)



### **TFT**

In [ ]:
# Multi Experiment TFT for ____ dataset

tft_parameters = {
    "input_length": 512,            # Look-back window size, pretty similar to RNN seq_len, as I see it.
    "batch_size": 32,                # Number of samples per training batch
    "num_epochs": 300,             # Self-explanatory
    "model_dim": 48,                # "d_model" — size of each internal feature vector
    "attention_heads": 4,           # "nhead" — number of attention heads
    "encoder_layers": 2,           # number of stacked encoder layers
    "decoder_layers": 2,           # number of stacked decoder layers
    "pl_trainer_kwargs": {
        "accelerator": device,
        "devices": 1,
        "precision": '32-true',
        "enable_progress_bar": True
    },
    "learning_rate": 1e-2,   
    "features": ['x', 'y', 'z']  # features to use for training and prediction
}

rho_value = 28
horizons_experiments = [1, 100]

train_mask = np.isin(train_o[:, 6], rho_value)
test_mask  = np.isin(test_o[:, 6], rho_value)

train_data = train_o[train_mask]
test_data  = test_o[test_mask]

maximum_test_size = 30_000

train_data = train_data[:maximum_test_size]
test_data = test_data[:maximum_test_size]

for tau in horizons_experiments:

    run_tft_pipeline(train_data, 
                    test_data, 
                    tft_parameters, 
                    forecast_horizons=tau, 
                    rho_value=rho_value, 
                    model=None,
                    measure_time = True)



### **UPO**

In [ ]:
horizons_experiments = [1, 100]

DT = 0.05
DT_UPOS = 0.001

rho_value = 28
test_mask = np.isin(test_o[:, 6], rho_value)
test_data = test_o[test_mask]

obs_xyz = test_data[:, 3:6]

maximum_test_size = 30_000
train_data = train_data[:maximum_test_size]
test_data = test_data[:maximum_test_size]


RHO_TO_LOAD = [28]

OUTPUT_DIR = "UPOS"
os.makedirs(OUTPUT_DIR, exist_ok=True)
# DATA_DIR = rf"data_upos_original/DATA1_{RHO}"

upo_data_multi_rho = {}
for RHO in RHO_TO_LOAD:

    final_filename = os.path.join(OUTPUT_DIR, f"lorenz_orbits_{RHO}_dt{DT_UPOS}.pkl")
    df_orbits_individual = pd.read_pickle(final_filename)
    df_orbits_individual = df_orbits_individual.sort_values("Orbit_ID")
    print(f"Loaded DataFrame from {final_filename}")
    upo_data_multi_rho[RHO] = df_orbits_individual

df_upo = upo_data_multi_rho[rho_value]

# Measure Training Time
time_start = time.time()
tree, labels, df_upo_extended = build_labeled_tree(df_upo)
time_end = time.time()
print(f"UPO Training Time: {time_end - time_start:.4f} seconds")
# End

time_results = []

for tau in horizons_experiments:
    # Measure Prediction Time
    start = time.time()
    traj_xyz, upo_all_stacked, pred_xyz, upo_segments = UPO_prediction_window(
        obs_xyz,
        window_size=tau,
        tree=tree,
        labels=labels,
        norm=2,
        k=1,
        upo_df=df_upo_extended,
        time_diff=int(DT / DT_UPOS)
    )
    end = time.time()
    print(f"UPO Prediction Time for tau={tau}: {end - start:.4f} seconds")
    # End

In [ ]:
import ast
import json
from collections import defaultdict

NOTEBOOK_PATH = "/home/roland/Nonstat_process/0_Conf_BB_Forecast_Limitss_Chaotic_Systems_Study _V1.ipynb"


def clean_code(code):
    """
    Remove IPython magics and shell commands (%..., %%..., !...)
    to avoid AST parsing errors.
    """
    cleaned_lines = []
    for line in code.splitlines():
        stripped = line.strip()
        if stripped.startswith("%") or stripped.startswith("!"):
            continue
        if stripped.startswith("%%"):
            continue
        cleaned_lines.append(line)
    return "\n".join(cleaned_lines)


def get_notebook_code(path):
    with open(path, "r", encoding="utf-8") as f:
        nb = json.load(f)

    code_cells = [
        "".join(cell["source"])
        for cell in nb["cells"]
        if cell["cell_type"] == "code"
    ]

    full_code = "\n".join(code_cells)
    return clean_code(full_code)


class FunctionAnalyzer(ast.NodeVisitor):
    def __init__(self):
        self.defined = set()
        self.called = set()
        self.call_map = defaultdict(list)

    def visit_FunctionDef(self, node):
        self.defined.add(node.name)
        self.generic_visit(node)

    def visit_Call(self, node):
        if isinstance(node.func, ast.Name):
            self.called.add(node.func.id)
            self.call_map[node.func.id].append(node.lineno)

        elif isinstance(node.func, ast.Attribute):
            self.called.add(node.func.attr)
            self.call_map[node.func.attr].append(node.lineno)

        self.generic_visit(node)

    def visit_Name(self, node):
        if node.id in self.defined:
            self.called.add(node.id)
        self.generic_visit(node)


def analyze_notebook(path):
    code = get_notebook_code(path)

    try:
        tree = ast.parse(code)
    except SyntaxError as e:
        print("Syntax error still present after cleaning:")
        print(e)
        return

    analyzer = FunctionAnalyzer()
    analyzer.visit(tree)

    defined = analyzer.defined
    called = analyzer.called

    used = sorted(defined & called)
    unused = sorted(defined - called)

    print("=" * 50)
    print("FUNCTION USAGE REPORT")
    print("=" * 50)

    print(f"\nTotal functions defined: {len(defined)}")
    print(f"Functions called at least once: {len(used)}")
    print(f"Unused functions: {len(unused)}")

    print("\n--- USED FUNCTIONS ---")
    for fn in used:
        print(f"{fn} (lines: {analyzer.call_map.get(fn, [])})")

    print("\n--- UNUSED FUNCTIONS ---")
    for fn in unused:
        print(fn)


# Run
analyze_notebook(NOTEBOOK_PATH)